# CERTEO — Certified trust maps for cross-modal Earth-observation generators

This notebook runs the full study on **real data only**:

| Dataset (Hugging Face) | What it gives us | Role |
|---|---|---|
| `Hermanni/sen12mscr` (SEN12MS-CR, 175 global scenes × 4 seasons, Sentinel-1 VV/VH + Sentinel-2 cloudy + cloud-free) | scene ids, seasons, two tasks (SAR→optical and SAR-aided cloud removal) | core: generators, floor, calibration, clustering, season shift |
| `125oii/dfc2020` (DFC2020, Sentinel-1/2 + land-cover labels) | new year/regions + labels | external shift test (H5) and usefulness test (H7) |
| `umkc-mcc/SAR2Opt` (TerraSAR-X 1 m + Google Earth optical, 10 cities) | different sensor, resolution, city ids | replication of the clustering result on another sensor (H2b) |

**What it does, end to end**
1. Trains / builds 8 SAR→optical generators (climatology, floor-centre, U-Net-L1, Pix2Pix, Pix2Pix-MC-dropout, Pix2Pix + SAR-TopoGuard, conditional rectified flow sample, flow posterior mean) and 2 cloud-removal models.
2. Trains a per-pixel **floor network** (mixture model of p(optical | SAR)) → identifiability floor 1 − Q_τ(x).
3. Builds 4 trust scores per generator: floor, TopoGuard edge support, sample spread, learned verifier.
4. Computes per-patch trust/error histograms on held-out scenes, winter scenes, DFC2020 and SAR2Opt.
5. Runs conformal risk control (CRC) with **patch** vs **scene** units, PAC (WSR) tiers, shift and recalibration tests.
6. Evaluates the **pre-registered hypotheses H1–H7 (+H2b)** and writes a detailed PASS / FAIL / INSUFFICIENT report.

**Run order:** run every cell top to bottom. Cell 12 is the **smoke run** (small slices of the same real data, ~30–60 min on the GPU) and prints the hypothesis table. Cell 13 is the **full run**. Every stage is resumable: all data shards, checkpoints, histograms, logs and reports are written to `./certeo_work`, and also to your own Hugging Face bucket if you set the `CERTEO_BUCKET` (and, for a private bucket, `HF_TOKEN`) secrets. If the session dies, just run all cells again; finished work is found and skipped.

In [ ]:
# ===== Cell 1: dependencies (only installs what is missing) =====
import importlib.util
import subprocess
import sys

pkgs_cell01 = {"tifffile": "tifffile", "pyarrow": "pyarrow", "scipy": "scipy", "PIL": "pillow", "psutil": "psutil"}
missing_cell01 = [pip_name for mod, pip_name in pkgs_cell01.items() if importlib.util.find_spec(mod) is None]
if importlib.util.find_spec("huggingface_hub") is None:
    missing_cell01.append("huggingface_hub>=1.0")
else:
    import huggingface_hub as hfh_probe_cell01
    if not hasattr(hfh_probe_cell01.HfApi, "batch_bucket_files"):
        missing_cell01.append("huggingface_hub>=1.0")
if missing_cell01:
    print("installing:", missing_cell01)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + missing_cell01)
print("dependencies ok")

In [ ]:
# ===== Cell 2: core imports, budgets, Hugging Face bucket sync, helpers =====
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import ctypes
import gc
import io
import json
import math
import re
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import psutil
import pyarrow.parquet as pq
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import HfApi, HfFileSystem
from PIL import Image
from scipy import stats as sstats

# ---- storage (nothing secret is written in this notebook) ----
# Optional environment variables / notebook secrets:
#   HF_TOKEN        a Hugging Face token (only needed for a private results bucket; public datasets need none)
#   CERTEO_BUCKET   "<owner>/<bucket>" of your own Hugging Face bucket, to keep results across sessions
#   CERTEO_RUN_ROOT folder inside the bucket (default "certeo_v1")
# Without CERTEO_BUCKET everything is kept on local disk under ./certeo_work.
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_BUCKET = os.environ.get("CERTEO_BUCKET", "").strip()
RUN_ROOT = os.environ.get("CERTEO_RUN_ROOT", "certeo_v1")
LOCAL_ROOT = os.path.abspath("certeo_work")
# CERTEO_OFFLINE=1 forces local-only mode even when a bucket is set.
SYNC_ENABLED = bool(HF_BUCKET) and os.environ.get("CERTEO_OFFLINE", "0") != "1"
HF_API = HfApi(token=HF_TOKEN)
HF_FS = HfFileSystem(token=HF_TOKEN)

# ---- resource budgets ----
RAM_CAP_GB = 30.0
RAM_SOFT_GB = 26.0
VRAM_CAP_GB = 70.0
SESSION_START = time.time()
SESSION_DEADLINE = SESSION_START + 11.0 * 3600

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
if DEVICE.type == "cuda":
    total_gb_cell02 = torch.cuda.get_device_properties(0).total_memory / 2**30
    torch.cuda.set_per_process_memory_fraction(min(1.0, min(VRAM_CAP_GB, 0.95 * total_gb_cell02) / total_gb_cell02))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
os.makedirs(LOCAL_ROOT, exist_ok=True)


def rss_gb():
    return psutil.Process().memory_info().rss / 2**30


def ram_guard(extra_gb, what="allocation"):
    used = rss_gb()
    if used + extra_gb > RAM_SOFT_GB:
        raise MemoryError(f"{what}: need ~{extra_gb:.1f} GB on top of {used:.1f} GB; soft limit {RAM_SOFT_GB} GB")


def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass
    return rss_gb()


def gpu_free():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def vram_peak_gb():
    return torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0


def check_deadline(where):
    if time.time() > SESSION_DEADLINE:
        raise TimeoutError(f"session clock near 11 h at '{where}'. Everything finished is in the bucket; re-run all cells to resume.")


def local_path(rel):
    return os.path.join(LOCAL_ROOT, rel)


def bucket_key(rel):
    return f"{RUN_ROOT}/{rel}"


def hf_push(rels):
    """Upload local files (relative paths) to the bucket. Retries; never crashes the run."""
    if not SYNC_ENABLED:
        return
    adds = [(local_path(r), bucket_key(r)) for r in rels if os.path.exists(local_path(r))]
    if not adds:
        return
    for attempt in range(5):
        try:
            HF_API.batch_bucket_files(HF_BUCKET, add=adds)
            return
        except Exception as exc:  # network hiccup: back off and retry
            print(f"  [hf_push retry {attempt + 1}] {type(exc).__name__}: {str(exc)[:120]}")
            time.sleep(5 * 2**attempt)
    print("  [hf_push] WARNING: upload failed after retries; file kept locally:", [a[1] for a in adds][:3])


def hf_pull(rel):
    """True if the file is available locally (downloads it from the bucket when needed)."""
    lp = local_path(rel)
    if os.path.exists(lp):
        return True
    if not SYNC_ENABLED:
        return False
    try:
        if not list(HF_API.get_bucket_paths_info(HF_BUCKET, [bucket_key(rel)])):
            return False
        os.makedirs(os.path.dirname(lp), exist_ok=True)
        HF_API.download_bucket_files(HF_BUCKET, [(bucket_key(rel), lp)], raise_on_missing_files=True)
        return os.path.exists(lp)
    except Exception as exc:
        print(f"  [hf_pull] {rel}: {type(exc).__name__}: {str(exc)[:100]}")
        return False


def atomic_write(rel, writer):
    lp = local_path(rel)
    os.makedirs(os.path.dirname(lp), exist_ok=True)
    base, ext = os.path.splitext(lp)
    tmp = f"{base}.tmp{ext}"
    writer(tmp)
    os.replace(tmp, lp)


def save_npz(rel, push=True, **arrays):
    atomic_write(rel, lambda p: np.savez(p, **arrays))
    if push:
        hf_push([rel])


def load_npz(rel):
    with np.load(local_path(rel), allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


def save_json(rel, obj, push=True):
    def _w(p):
        with open(p, "w") as fh:
            fh.write(json.dumps(obj, indent=1, default=float))
    atomic_write(rel, _w)
    if push:
        hf_push([rel])


def load_json(rel):
    with open(local_path(rel)) as f:
        return json.load(f)


def save_ckpt(rel, obj, push=True):
    atomic_write(rel, lambda p: torch.save(obj, p))
    if push:
        hf_push([rel])


def load_ckpt(rel):
    return torch.load(local_path(rel), map_location=DEVICE, weights_only=False)


def log_line(tag, msg):
    line = f"[{time.strftime('%H:%M:%S')}] [{tag}] {msg} | RAM {rss_gb():.1f} GB | VRAM peak {vram_peak_gb():.1f} GB"
    print(line)
    lp = local_path(f"{tag}/log.txt")
    os.makedirs(os.path.dirname(lp), exist_ok=True)
    with open(lp, "a") as f:
        f.write(line + "\n")


def seed_all(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


hw_cell02 = {"device": str(DEVICE), "torch": torch.__version__,
             "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
             "sync_enabled": SYNC_ENABLED, "bucket": HF_BUCKET, "root": RUN_ROOT,
             "disk_free_gb": round(__import__("shutil").disk_usage(LOCAL_ROOT).free / 1e9, 1)}
print(hw_cell02)

In [ ]:
# ===== Cell 3: configuration (smoke and full) and the scene split =====
COMMON_CFG = {
    "seed": 0,
    "alphas": [0.05, 0.10],
    "alpha_main": 0.05,
    "taus": [0.05, 0.10, 0.15, 0.20],   # 0.20 = sensitivity extra (pre-registered: 0.05-0.15)
    "tau_main": 0.10,
    "n_bins": 200,
    "n_floor_bins": 20,
    "mix_components": 4,
    "K": 5,
    "opt_scale_dn": 3000.0,           # optical DN -> [0,1]: clip(DN / 3000, 0, 1)
    "sar_vv_range": [-25.0, 0.0],     # dB clip for VV (same as SAR-TopoGuard)
    "sar_vh_range": [-32.5, -5.0],    # dB clip for VH
    "topoguard": {"w": 5.0, "r": 3, "s": 5.0, "t": 0.1},  # frozen values from the SAR-TopoGuard paper
    "s2o_gens": ["climatology", "floor_center", "unet_l1", "pix2pix", "pix2pix_mcd",
                 "pix2pix_topoguard", "flow_sample", "flow_mean"],
    "cr_gens": ["cr_unet", "cr_cloudy_identity"],
    "scores": ["floor", "topo", "spread", "learned"],
    "stochastic_gens": ["pix2pix_mcd", "pix2pix_topoguard", "flow_sample", "flow_mean"],
    "max_steps_per_epoch": None,
}
CFG_SMOKE = dict(COMMON_CFG, tag="smoke", n_train_scenes=16, n_aux_scenes=8, n_cal_scenes=40, n_winter_scenes=12,
                 train_rgs_per_scene=1, train_patches_per_rg=64, eval_patches_per_scene=16, aux_patches_per_scene=32,
                 batch=16, epochs_unet=4, epochs_gan=4, epochs_flow=6, epochs_floor=4, epochs_cr=4, epochs_verifier=3,
                 epochs_seg=4, flow_steps=10, resample_R=200, dfc_train_max=768, dfc_val_max=256,
                 sar2opt_train_max=400, sar2opt_test_max=300, epochs_sar2opt=6, ckpt_every=2)
CFG_FULL = dict(COMMON_CFG, tag="full", n_train_scenes=62, n_aux_scenes=22, n_cal_scenes=64, n_winter_scenes=27,
                train_rgs_per_scene=2, train_patches_per_rg=64, eval_patches_per_scene=64, aux_patches_per_scene=64,
                batch=16, epochs_unet=30, epochs_gan=30, epochs_flow=60, epochs_floor=25, epochs_cr=30, epochs_verifier=12,
                epochs_seg=15, flow_steps=20, resample_R=500, dfc_train_max=None, dfc_val_max=None,
                sar2opt_train_max=None, sar2opt_test_max=None, epochs_sar2opt=40, ckpt_every=5)
if os.environ.get("CERTEO_MICRO", "0") == "1":  # only for CPU code testing; never used on molab
    CFG_SMOKE.update(n_train_scenes=2, n_aux_scenes=1, n_cal_scenes=4, n_winter_scenes=2, train_patches_per_rg=6,
                     eval_patches_per_scene=3, aux_patches_per_scene=4, batch=2, epochs_unet=1, epochs_gan=1,
                     epochs_flow=1, epochs_floor=1, epochs_cr=1, epochs_verifier=1, epochs_seg=1, flow_steps=2, K=2,
                     resample_R=10, dfc_train_max=6, dfc_val_max=4, sar2opt_train_max=6, sar2opt_test_max=6,
                     epochs_sar2opt=1, max_steps_per_epoch=2, ckpt_every=1)


def sen12mscr_scene_list():
    """All 175 (season, scene) files of SEN12MS-CR, read from the dataset itself (cached in the bucket)."""
    rel = "data/sen12mscr_scenes.json"
    if hf_pull(rel):
        return load_json(rel)
    out = []
    for season in ["spring", "summer", "fall", "winter"]:
        for p in HF_FS.ls(f"datasets/Hermanni/sen12mscr/{season}", detail=False):
            m = re.search(r"scene_(\d+)\.parquet$", p)
            if m:
                out.append([season, int(m.group(1))])
    out.sort()
    save_json(rel, out)
    return out


def make_scene_split(cfg):
    """Deterministic, scene-disjoint split. Winter is held out entirely (season-shift test).
    Smoke uses prefixes of the same lists, so its data shards are reused by the full run."""
    scenes = sen12mscr_scene_list()
    winter = [s for s in scenes if s[0] == "winter"]
    other = [s for s in scenes if s[0] != "winter"]
    rng = np.random.default_rng(12345)
    other = [other[i] for i in rng.permutation(len(other))]
    winter = [winter[i] for i in rng.permutation(len(winter))]
    n_tr, n_aux = CFG_FULL["n_train_scenes"], CFG_FULL["n_aux_scenes"]
    pools = {"train": other[:n_tr], "aux": other[n_tr:n_tr + n_aux], "cal": other[n_tr + n_aux:], "winter": winter}
    return {"train": pools["train"][:cfg["n_train_scenes"]], "aux": pools["aux"][:cfg["n_aux_scenes"]],
            "cal": pools["cal"][:cfg["n_cal_scenes"]], "winter": pools["winter"][:cfg["n_winter_scenes"]]}


print({k: (v["n_train_scenes"], v["n_aux_scenes"], v["n_cal_scenes"], v["n_winter_scenes"]) for k, v in
       {"smoke": CFG_SMOKE, "full": CFG_FULL}.items()})

In [ ]:
# ===== Cell 4: real-data readers (SEN12MS-CR, DFC2020, SAR2Opt), cached as shards in the bucket =====
def norm_sar_db(sar_hw2, cfg):
    """(H,W,2) dB -> (2,H,W) float16 in [0,1]. VV clip [-25,0] dB like SAR-TopoGuard; VH clip [-32.5,-5]."""
    vv = np.clip(sar_hw2[..., 0], *cfg["sar_vv_range"])
    vh = np.clip(sar_hw2[..., 1], *cfg["sar_vh_range"])
    vv = (vv - cfg["sar_vv_range"][0]) / (cfg["sar_vv_range"][1] - cfg["sar_vv_range"][0])
    vh = (vh - cfg["sar_vh_range"][0]) / (cfg["sar_vh_range"][1] - cfg["sar_vh_range"][0])
    return np.nan_to_num(np.stack([vv, vh]), nan=0.0).astype(np.float16)


def norm_opt_dn(opt_hwc13, cfg):
    """(H,W,13) Sentinel-2 DN -> RGB (B4,B3,B2) as (3,H,W) float16 in [0,1]."""
    rgb = opt_hwc13[..., [3, 2, 1]].astype(np.float32) / cfg["opt_scale_dn"]
    return np.nan_to_num(np.clip(rgb, 0.0, 1.0), nan=0.0).transpose(2, 0, 1).astype(np.float16)


def sen12mscr_rowgroups(season, scene):
    path = f"datasets/Hermanni/sen12mscr/{season}/scene_{scene}.parquet"
    with HF_FS.open(path, "rb", block_size=2**20) as f:
        return pq.ParquetFile(f).metadata.num_row_groups


def sen12mscr_shard(season, scene, rg, cfg):
    """One parquet row group (64 patches) -> normalised arrays; cached as a shard in the bucket."""
    rel = f"data/sen12mscr/{season}_{scene}_rg{rg}.npz"
    if hf_pull(rel):
        return load_npz(rel)
    path = f"datasets/Hermanni/sen12mscr/{season}/scene_{scene}.parquet"
    for attempt in range(4):
        try:
            with HF_FS.open(path, "rb", block_size=64 * 2**20) as f:
                tab = pq.ParquetFile(f).read_row_group(rg, columns=["sar", "cloudy", "target", "sar_shape", "opt_shape", "patch"])
            break
        except Exception as exc:
            print(f"  read retry {attempt + 1} {season}/{scene} rg{rg}: {type(exc).__name__}")
            time.sleep(5 * 2**attempt)
    else:
        raise RuntimeError(f"could not read {path} row group {rg}")
    n = tab.num_rows
    sar = np.empty((n, 2, 256, 256), np.float16)
    opt = np.empty((n, 3, 256, 256), np.float16)
    cld = np.empty((n, 3, 256, 256), np.float16)
    pid = np.empty(n, np.int32)
    for i in range(n):
        ss, os_ = tab["sar_shape"][i].as_py(), tab["opt_shape"][i].as_py()
        sar[i] = norm_sar_db(np.frombuffer(tab["sar"][i].as_py(), np.float32).reshape(ss), cfg)
        opt[i] = norm_opt_dn(np.frombuffer(tab["target"][i].as_py(), np.int16).reshape(os_), cfg)
        cld[i] = norm_opt_dn(np.frombuffer(tab["cloudy"][i].as_py(), np.int16).reshape(os_), cfg)
        pid[i] = int(re.sub(r"\D", "", tab["patch"][i].as_py()) or -1)
    del tab
    save_npz(rel, sar=sar, opt=opt, cld=cld, pid=pid)
    return {"sar": sar, "opt": opt, "cld": cld, "pid": pid}


def scene_rowgroup_choice(season, scene, n_rgs_wanted):
    """Deterministic, spread-out row groups for a scene (metadata cached in the bucket)."""
    rel = "data/sen12mscr_rowgroups.json"
    meta = load_json(rel) if hf_pull(rel) else {}
    key = f"{season}_{scene}"
    if key not in meta:
        meta[key] = sen12mscr_rowgroups(season, scene)
        save_json(rel, meta)
    nrg = meta[key]
    step = max(1, nrg // (n_rgs_wanted + 1))
    return list(dict.fromkeys((nrg // 2 + k * step) % nrg for k in range(n_rgs_wanted)))


def load_sen12mscr_split(scene_list, rgs_per_scene, patches_per_rg, cfg, label):
    """Preallocated (no concatenation copies) arrays for a list of scenes."""
    n_total = len(scene_list) * rgs_per_scene * patches_per_rg
    ram_guard(n_total * 8 * 65536 * 2 / 2**30 * 1.1, f"load {label}")
    sar = np.empty((n_total, 2, 256, 256), np.float16)
    opt = np.empty((n_total, 3, 256, 256), np.float16)
    cld = np.empty((n_total, 3, 256, 256), np.float16)
    scene_idx = np.empty(n_total, np.int32)
    k = 0
    for si, (season, scene) in enumerate(scene_list):
        check_deadline(f"load {label}")
        for rg in scene_rowgroup_choice(season, scene, rgs_per_scene):
            sh = sen12mscr_shard(season, scene, rg, cfg)
            if SYNC_ENABLED:  # the shard is safe in the bucket; keep local disk small
                lp_shard = local_path(f"data/sen12mscr/{season}_{scene}_rg{rg}.npz")
                if os.path.exists(lp_shard):
                    os.remove(lp_shard)
            sel = np.random.default_rng(1000 * scene + rg).permutation(len(sh["pid"]))[:patches_per_rg]
            m = len(sel)
            sar[k:k + m], opt[k:k + m], cld[k:k + m] = sh["sar"][sel], sh["opt"][sel], sh["cld"][sel]
            scene_idx[k:k + m] = si
            k += m
            del sh
        if (si + 1) % 10 == 0:
            print(f"  {label}: {si + 1}/{len(scene_list)} scenes, {k} patches, RAM {rss_gb():.1f} GB")
    trim_ram()
    return {"sar": sar[:k], "opt": opt[:k], "cld": cld[:k], "scene": scene_idx[:k],
            "scene_names": [f"{a}_{b}" for a, b in scene_list]}


def dfc_member_index(zip_name):
    f = HF_FS.open(f"datasets/125oii/dfc2020/{zip_name}", "rb", block_size=8 * 2**20)
    zf = zipfile.ZipFile(f)
    idx = {}
    for n in zf.namelist():
        if n.endswith(".tif") and "__MACOSX" not in n:
            m = re.search(r"_p(\d+)\.tif$", n)
            if m:
                idx[int(m.group(1))] = n
    return zf, idx


def load_dfc2020(split, max_n, cfg, with_sar=True):
    """split in {'validation','0'}; returns RGB [0,1], SAR [0,1], DFC labels (0..9, 255 = ignore).
    Downloads each zip once (fast, large sequential transfer), reads it locally, then deletes it."""
    rel = f"data/dfc2020/{split}_{max_n}_{int(with_sar)}.npz"
    if hf_pull(rel):
        return load_npz(rel)
    from huggingface_hub import hf_hub_download
    import shutil
    cache_dir = local_path("dl_cache_dfc2020")

    def _open_local(zip_name):
        t0 = time.time()
        p = hf_hub_download("125oii/dfc2020", zip_name, repo_type="dataset", cache_dir=cache_dir, token=HF_TOKEN)
        print(f"  downloaded {zip_name}: {os.path.getsize(p) / 1e9:.2f} GB in {time.time() - t0:.0f} s")
        zf = zipfile.ZipFile(p)
        idx = {}
        for nm in zf.namelist():
            if nm.endswith(".tif") and "__MACOSX" not in nm:
                mm = re.search(r"_p(\d+)\.tif$", nm)
                if mm:
                    idx[int(mm.group(1))] = nm
        return zf, idx

    zs2, i_s2 = _open_local(f"s2_{split}.zip")
    zlb, i_lb = _open_local(f"dfc_{split}.zip")
    keys = sorted(set(i_s2) & set(i_lb))
    if with_sar:
        zs1, i_s1 = _open_local(f"s1_{split}.zip")
        keys = sorted(set(keys) & set(i_s1))
    if max_n:
        keys = [keys[i] for i in np.random.default_rng(7).permutation(len(keys))[:max_n]]
    n = len(keys)
    ram_guard(n * (3 + 2 + 0.5) * 65536 * 2 / 2**30, "dfc2020")
    opt = np.empty((n, 3, 256, 256), np.float16)
    sar = np.empty((n, 2, 256, 256), np.float16) if with_sar else np.zeros((0,), np.float16)
    lab = np.empty((n, 256, 256), np.uint8)
    for j, kk in enumerate(keys):
        opt[j] = norm_opt_dn(tifffile.imread(io.BytesIO(zs2.read(i_s2[kk]))), cfg)
        lb = tifffile.imread(io.BytesIO(zlb.read(i_lb[kk]))).astype(np.int32)
        lab[j] = np.where((lb >= 1) & (lb <= 10), lb - 1, 255).astype(np.uint8)
        if with_sar:
            sar[j] = norm_sar_db(tifffile.imread(io.BytesIO(zs1.read(i_s1[kk]))).astype(np.float32), cfg)
        if (j + 1) % 500 == 0:
            print(f"  dfc2020 {split}: {j + 1}/{n}")
    for zf in [zs2, zlb] + ([zs1] if with_sar else []):
        zf.close()
    shutil.rmtree(cache_dir, ignore_errors=True)          # free local disk; the processed shard goes to the bucket
    save_npz(rel, opt=opt, sar=sar, lab=lab, pid=np.array(keys, np.int32))
    return {"opt": opt, "sar": sar, "lab": lab, "pid": np.array(keys, np.int32)}


def load_sar2opt(split, max_n):
    """TerraSAR-X (1 m) / Google Earth pairs, 600x600. Names 'city_row_col.jpg' give city and block ids."""
    rel = f"data/sar2opt/{split}_{max_n}.npz"
    if hf_pull(rel):
        return load_npz(rel)
    names = sorted(p.split("/")[-1] for p in HF_FS.ls(f"datasets/umkc-mcc/SAR2Opt/A/{split}", detail=False) if p.endswith(".jpg"))
    if max_n:
        names = [names[i] for i in sorted(np.random.default_rng(3).permutation(len(names))[:max_n])]

    def _read_pair(nm):
        a = np.array(Image.open(HF_FS.open(f"datasets/umkc-mcc/SAR2Opt/A/{split}/{nm}", "rb")).convert("L"), np.uint8)
        b = np.array(Image.open(HF_FS.open(f"datasets/umkc-mcc/SAR2Opt/B/{split}/{nm}", "rb")).convert("RGB"), np.uint8)
        return a, b.transpose(2, 0, 1)

    with ThreadPoolExecutor(16) as ex:
        pairs = list(ex.map(_read_pair, names))
    sar = np.stack([p[0][None] for p in pairs])
    opt = np.stack([p[1] for p in pairs])
    parts = np.array([[int(x) for x in nm[:-4].split("_")] for nm in names], np.int32)  # city, row, col
    save_npz(rel, sar=sar, opt=opt, parts=parts)
    return {"sar": sar, "opt": opt, "parts": parts}


print("data readers ready")

In [ ]:
# ===== Cell 5: models (U-Net generator, PatchGAN, rectified-flow U-Net), SAR-TopoGuard, floor maths, metrics =====
class UNetG(nn.Module):
    """Pix2Pix-style U-Net (8 levels for 256 px), GroupNorm, dropout in the 3 innermost decoder blocks."""

    def __init__(self, in_ch, out_ch, base=64, depth=8, p_drop=0.5, out_act="sigmoid"):
        super().__init__()
        chs = [min(base * 2**i, base * 8) for i in range(depth)]
        self.downs = nn.ModuleList()
        prev = in_ch
        for i, c in enumerate(chs):
            layers = [] if i == 0 else [nn.LeakyReLU(0.2)]
            layers.append(nn.Conv2d(prev, c, 4, 2, 1))
            if 0 < i < depth - 1:
                layers.append(nn.GroupNorm(8, c))
            self.downs.append(nn.Sequential(*layers))
            prev = c
        self.ups = nn.ModuleList()
        for i in range(depth):
            lvl = depth - 1 - i
            cin = chs[lvl] if i == 0 else chs[lvl] * 2
            cout = chs[lvl - 1] if lvl > 0 else out_ch
            layers = [nn.ReLU(), nn.ConvTranspose2d(cin, cout, 4, 2, 1)]
            if lvl > 0:
                layers.append(nn.GroupNorm(8, cout))
                if i < 3 and p_drop > 0:
                    layers.append(nn.Dropout(p_drop))
            self.ups.append(nn.Sequential(*layers))
        self.out_act = out_act

    def forward(self, x):
        skips = []
        h = x
        for d in self.downs:
            h = d(h)
            skips.append(h)
        h = skips[-1]
        for i, up in enumerate(self.ups):
            if i > 0:
                h = torch.cat([h, skips[-1 - i]], 1)
            h = up(h)
        return torch.sigmoid(h) if self.out_act == "sigmoid" else h


def set_mc_dropout(model, on):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train(on)


class PatchD(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.GroupNorm(8, 128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.GroupNorm(8, 256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 1, 1), nn.GroupNorm(8, 512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 1))

    def forward(self, x):
        return self.net(x)


class ResBlockT(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, 1, 1)
        self.t = nn.Linear(tdim, cout)
        self.n2, self.c2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, 1, 1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x))) + self.t(temb)[:, :, None, None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)


class FlowUNet(nn.Module):
    """Conditional rectified-flow velocity network v(x_t, t | SAR). 256 -> 16 px, channels 64..256."""

    def __init__(self, cond_ch, out_ch=3, base=64, mults=(1, 1, 2, 2, 4)):
        super().__init__()
        self.tdim = base * 4
        self.temb = nn.Sequential(nn.Linear(base, self.tdim), nn.SiLU(), nn.Linear(self.tdim, self.tdim))
        self.inp = nn.Conv2d(out_ch + cond_ch, base, 3, 1, 1)
        chs = [base * m for m in mults]
        self.down_blocks, self.down_samp = nn.ModuleList(), nn.ModuleList()
        prev = base
        for i, c in enumerate(chs):
            self.down_blocks.append(ResBlockT(prev, c, self.tdim))
            self.down_samp.append(nn.Conv2d(c, c, 3, 2, 1) if i < len(chs) - 1 else nn.Identity())
            prev = c
        self.mid1, self.mid2 = ResBlockT(prev, prev, self.tdim), ResBlockT(prev, prev, self.tdim)
        self.up_blocks = nn.ModuleList()
        for i, c in reversed(list(enumerate(chs))):
            self.up_blocks.append(ResBlockT(prev + c, c, self.tdim))
            prev = c
        self.out = nn.Sequential(nn.GroupNorm(8, prev), nn.SiLU(), nn.Conv2d(prev, out_ch, 3, 1, 1))
        self.base = base

    def time_embed(self, t):
        half = self.base // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        a = t[:, None] * 1000.0 * freqs[None]
        return self.temb(torch.cat([a.sin(), a.cos()], 1))

    def forward(self, xt, t, cond):
        temb = self.time_embed(t)
        h = self.inp(torch.cat([xt, cond], 1))
        skips = []
        for blk, ds in zip(self.down_blocks, self.down_samp):
            h = blk(h, temb)
            skips.append(h)
            h = ds(h)
        h = self.mid2(self.mid1(h, temb), temb)
        for blk in self.up_blocks:
            s = skips.pop()
            if h.shape[-1] != s.shape[-1]:
                h = F.interpolate(h, size=s.shape[-2:], mode="nearest")
            h = blk(torch.cat([h, s], 1), temb)
        return self.out(h)


@torch.no_grad()
def flow_sample(model, cond, steps, gen=None):
    x = torch.randn(cond.shape[0], 3, cond.shape[2], cond.shape[3], device=cond.device, generator=gen)
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((cond.shape[0],), i * dt, device=cond.device)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            v = model(x, t, cond)
        x = x + dt * v.float()
    return ((x + 1) / 2).clamp(0, 1)


SOBEL_KX = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
BINOM5 = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0]) / 16.0


def depthwise(img, k2d):
    c = img.shape[1]
    w = k2d.to(img.device, img.dtype)[None, None].repeat(c, 1, 1, 1)
    p = k2d.shape[-1] // 2
    return F.conv2d(F.pad(img, (p, p, p, p), mode="replicate"), w, groups=c)


def grad_mag(img):
    """SAR-TopoGuard eq. (3): channel-mean Sobel gradient magnitude, (B,C,H,W) -> (B,1,H,W)."""
    gx = depthwise(img, SOBEL_KX)
    gy = depthwise(img, SOBEL_KX.t())
    return torch.sqrt(gx**2 + gy**2 + 1e-12).mean(1, keepdim=True)


def gaussian_kernel(sigma, size):
    x = torch.arange(size, dtype=torch.float32) - size // 2
    g = torch.exp(-x**2 / (2 * sigma**2))
    g = g / g.sum()
    return g[:, None] * g[None, :]


GAUSS_LOWPASS = gaussian_kernel(2.0, 9)


def lowpass(img):
    return depthwise(img, GAUSS_LOWPASS)


def topoguard_edge_mask(sar_vv, tg):
    """SAR-TopoGuard eq. (8): M_edge = sigmoid(s (g(X_sar) - t)) and eq. (4) flatness mask."""
    g_sar = grad_mag(sar_vv)
    return torch.sigmoid(tg["s"] * (g_sar - tg["t"])), torch.exp(-10.0 * g_sar.clamp(0, 0.3)), g_sar


def topoguard_select_gate(cands, sar_vv, tg):
    """SAR-TopoGuard eqs. (5)-(8) on K candidates (K,B,3,H,W): select by GMC - w*HFI, then frequency gate."""
    m_edge, m_flat, g_sar = topoguard_edge_mask(sar_vv, tg)
    gt_sar = F.avg_pool2d(g_sar, tg["r"], 1, tg["r"] // 2)
    scores = []
    for k in range(cands.shape[0]):
        gy = grad_mag(cands[k])
        hfi = ((m_flat * gy).sum((1, 2, 3)) / (gy.sum((1, 2, 3)) + 1e-8)).clamp(0, 1)
        a = F.avg_pool2d(gy, tg["r"], 1, tg["r"] // 2).flatten(1)
        b = gt_sar.flatten(1)
        a, b = a - a.mean(1, keepdim=True), b - b.mean(1, keepdim=True)
        gmc = (a * b).sum(1) / (a.norm(dim=1) * b.norm(dim=1) + 1e-8)
        scores.append(gmc - tg["w"] * hfi)
    best = torch.stack(scores).argmax(0)
    y_best = cands[best, torch.arange(cands.shape[1], device=cands.device)]
    k2 = BINOM5[:, None] * BINOM5[None, :]
    y_low = depthwise(y_best, k2)
    return y_low + m_edge * (y_best - y_low), m_edge


def floor_params(raw, M):
    """raw (B, 3*3*M, H, W) -> mixture weights logits, means, std per band: each (B,3,M,H,W)."""
    B, _, H, W = raw.shape
    r = raw.float().view(B, 3, 3, M, H, W)
    return r[:, :, 0], torch.sigmoid(r[:, :, 1]), 0.003 + 0.3 * F.softplus(r[:, :, 2])


def mixture_nll(y, logit_pi, mu, sig):
    z = (y.float().unsqueeze(2) - mu) / sig
    logp = F.log_softmax(logit_pi, 2) - 0.5 * z**2 - torch.log(sig) - 0.5 * math.log(2 * math.pi)
    return -torch.logsumexp(logp, 2).mean()


@torch.no_grad()
def floor_maps(logit_pi, mu, sig, tau):
    """Identifiability floor per band: floor_b(x) = 1 - sup_c P(|y_b - c| <= tau | x), and the best centre c*.
    The sup is taken over an 81-point grid plus every component mean and mean +- tau/2."""
    pi = torch.softmax(logit_pi, 2)
    best = torch.zeros_like(mu[:, :, 0])
    bestc = torch.zeros_like(best)

    def _mass(c):
        return (pi * (torch.special.ndtr((c + tau - mu) / sig) - torch.special.ndtr((c - tau - mu) / sig))).sum(2)

    cands = [torch.full_like(best, float(c)) for c in np.linspace(0, 1, 81)]
    for m in range(mu.shape[2]):
        for off in (0.0, tau / 2, -tau / 2):
            cands.append((mu[:, :, m] + off).clamp(0, 1))
    for c in cands:
        mass = _mass(c.unsqueeze(2))
        upd = mass > best
        best = torch.where(upd, mass, best)
        bestc = torch.where(upd, c, bestc)
    return (1.0 - best).clamp(0, 1), bestc


def psnr_batch(a, b):
    mse = ((a.float() - b.float()) ** 2).mean((1, 2, 3)).clamp_min(1e-10)
    return 10 * torch.log10(1.0 / mse)


SSIM_WIN = gaussian_kernel(1.5, 11)


def ssim_batch(a, b):
    a, b = a.float(), b.float()
    mu_a, mu_b = depthwise(a, SSIM_WIN), depthwise(b, SSIM_WIN)
    saa = depthwise(a * a, SSIM_WIN) - mu_a**2
    sbb = depthwise(b * b, SSIM_WIN) - mu_b**2
    sab = depthwise(a * b, SSIM_WIN) - mu_a * mu_b
    c1, c2 = 0.01**2, 0.03**2
    s = ((2 * mu_a * mu_b + c1) * (2 * sab + c2)) / ((mu_a**2 + mu_b**2 + c1) * (saa + sbb + c2))
    return s.mean((1, 2, 3))


print("models ready; Pix2Pix U-Net params (M):", round(sum(p.numel() for p in UNetG(2, 3).parameters()) / 1e6, 1),
      "| flow U-Net params (M):", round(sum(p.numel() for p in FlowUNet(2).parameters()) / 1e6, 1))

In [ ]:
# ===== Cell 6: resumable training loops (checkpoint every few epochs to the bucket) =====
def to_dev(arr, idx):
    return torch.from_numpy(np.ascontiguousarray(arr[idx])).to(DEVICE, non_blocking=True).float()


def rand_flip(*ts):
    if np.random.rand() < 0.5:
        ts = tuple(t.flip(-1) for t in ts)
    if np.random.rand() < 0.5:
        ts = tuple(t.flip(-2) for t in ts)
    return ts


def epoch_batches(n, batch, cfg):
    perm = np.random.permutation(n)
    nb = n // batch
    if cfg["max_steps_per_epoch"]:
        nb = min(nb, cfg["max_steps_per_epoch"])
    return [np.sort(perm[i * batch:(i + 1) * batch]) for i in range(max(nb, 1))]


def resume_or_init(rel, models, opts):
    """Returns the next epoch to run (0 if fresh)."""
    if hf_pull(rel):
        ck = load_ckpt(rel)
        for k, m in models.items():
            m.load_state_dict(ck["model_" + k])
        for k, o in opts.items():
            o.load_state_dict(ck["opt_" + k])
        torch.set_rng_state(ck["rng_cpu"].cpu() if hasattr(ck["rng_cpu"], "cpu") else ck["rng_cpu"])
        np.random.set_state(ck["rng_np"])
        return ck["epoch"] + 1
    return 0


def save_train_state(rel, models, opts, epoch, cfg, final=False):
    ck = {"epoch": epoch, "rng_cpu": torch.get_rng_state(), "rng_np": np.random.get_state()}
    ck.update({"model_" + k: m.state_dict() for k, m in models.items()})
    ck.update({"opt_" + k: o.state_dict() for k, o in opts.items()})
    push = final or ((epoch + 1) % cfg["ckpt_every"] == 0)
    save_ckpt(rel, ck, push=push)


def train_generic(name, cfg, models, opts, epochs, step_fn, n, label):
    """Shared resumable loop. step_fn(idx) -> dict of float losses. Final weights saved as '<name>_final.pt'."""
    tag = cfg["tag"]
    final_rel = f"{tag}/models/{name}_final.pt"
    if hf_pull(final_rel):
        ck = load_ckpt(final_rel)
        for k, m in models.items():
            m.load_state_dict(ck["model_" + k])
        log_line(tag, f"{name}: loaded finished model from bucket")
        return
    state_rel = f"{tag}/models/{name}_state.pt"
    start = resume_or_init(state_rel, models, opts)
    if start:
        log_line(tag, f"{name}: resuming at epoch {start}")
    t0 = time.time()
    for ep in range(start, epochs):
        check_deadline(name)
        for m in models.values():
            m.train()
        agg = {}
        batches = epoch_batches(n, cfg["batch"], cfg)
        for idx in batches:
            for k, v in step_fn(idx).items():
                agg[k] = agg.get(k, 0.0) + v / len(batches)
        save_train_state(state_rel, models, opts, ep, cfg, final=(ep == epochs - 1))
        log_line(tag, f"{name} epoch {ep + 1}/{epochs} " + " ".join(f"{k}={v:.4f}" for k, v in agg.items()) +
                 f" ({(time.time() - t0) / 60:.1f} min)")
    save_ckpt(final_rel, {"model_" + k: m.state_dict() for k, m in models.items()})
    gpu_free()


def train_unet_l1(name, data, in_keys, cfg, epochs, use_gan=False):
    """U-Net with L1 (and optionally LSGAN PatchGAN, lambda_L1 = 100, as Pix2Pix). in_keys: input arrays."""
    in_ch = sum(data[k].shape[1] for k in in_keys)
    G = UNetG(in_ch, 3).to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    models, opts = {"G": G}, {"G": oG}
    if use_gan:
        D = PatchD(in_ch + 3).to(DEVICE)
        oD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
        models["D"], opts["D"] = D, oD

    def _step(idx):
        x = torch.cat([to_dev(data[k], idx) for k in in_keys], 1)
        y = to_dev(data["opt"], idx)
        x, y = rand_flip(x, y)
        out = {}
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            yh = G(x)
            if use_gan:
                d_fake = models["D"](torch.cat([x, yh.detach()], 1)).float()
                d_real = models["D"](torch.cat([x, y], 1)).float()
                loss_d = 0.5 * (((d_real - 1) ** 2).mean() + (d_fake**2).mean())
        if use_gan:
            opts["D"].zero_grad(set_to_none=True)
            loss_d.backward()
            opts["D"].step()
            out["D"] = loss_d.item()
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            l1 = (yh.float() - y).abs().mean()
            loss = 100.0 * l1 if use_gan else l1
            if use_gan:
                loss = loss + ((models["D"](torch.cat([x, yh], 1)).float() - 1) ** 2).mean()
        oG.zero_grad(set_to_none=True)
        loss.backward()
        oG.step()
        out["L1"] = l1.item()
        return out

    train_generic(name, cfg, models, opts, epochs, _step, len(data["opt"]), name)
    G.eval()
    return G


def train_flow(name, data, cfg, epochs):
    Fm = FlowUNet(cond_ch=2).to(DEVICE)
    oF = torch.optim.AdamW(Fm.parameters(), lr=2e-4, weight_decay=1e-4)

    def _step(idx):
        c = to_dev(data["sar"], idx)
        y = to_dev(data["opt"], idx) * 2 - 1
        c, y = rand_flip(c, y)
        x0 = torch.randn_like(y)
        t = torch.sigmoid(torch.randn(y.shape[0], device=DEVICE))   # logit-normal time sampling
        xt = (1 - t[:, None, None, None]) * x0 + t[:, None, None, None] * y
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            v = Fm(xt, t, c)
        loss = ((v.float() - (y - x0)) ** 2).mean()
        oF.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(Fm.parameters(), 1.0)
        oF.step()
        return {"fm": loss.item()}

    train_generic(name, cfg, {"F": Fm}, {"F": oF}, epochs, _step, len(data["opt"]), name)
    Fm.eval()
    return Fm


def train_floor(name, data, cfg, epochs):
    """Mixture-density U-Net for p(optical_b | SAR) per pixel and band (M components)."""
    M = cfg["mix_components"]
    Fl = UNetG(2, 3 * 3 * M, base=48, p_drop=0.0, out_act=None).to(DEVICE)
    oF = torch.optim.Adam(Fl.parameters(), lr=2e-4, betas=(0.9, 0.999))

    def _step(idx):
        x = to_dev(data["sar"], idx)
        y = to_dev(data["opt"], idx)
        x, y = rand_flip(x, y)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            raw = Fl(x)
        loss = mixture_nll(y, *floor_params(raw, M))
        oF.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(Fl.parameters(), 1.0)
        oF.step()
        return {"nll": loss.item()}

    train_generic(name, cfg, {"F": Fl}, {"F": oF}, epochs, _step, len(data["opt"]), name)
    Fl.eval()
    return Fl


def train_verifier(name, x_in, wrong, cfg, epochs):
    """Learned verifier: P(pixel error > tau_main | SAR, generated output [, cloudy]). x_in float16 (N,C,H,W)."""
    V = UNetG(x_in.shape[1], 1, base=32, p_drop=0.0, out_act=None).to(DEVICE)
    oV = torch.optim.Adam(V.parameters(), lr=2e-4, betas=(0.9, 0.999))
    pos = float(wrong.mean())
    pw = torch.tensor(min(20.0, max(1.0, (1 - pos) / max(pos, 1e-4))), device=DEVICE)
    holder = {"x": x_in, "w": wrong}

    def _step(idx):
        x = to_dev(holder["x"], idx)
        w = torch.from_numpy(holder["w"][idx]).to(DEVICE).float()[:, None]
        x, w = rand_flip(x, w)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            lg = V(x)
        loss = F.binary_cross_entropy_with_logits(lg.float(), w, pos_weight=pw)
        oV.zero_grad(set_to_none=True)
        loss.backward()
        oV.step()
        return {"bce": loss.item()}

    train_generic(name, cfg, {"V": V}, {"V": oV}, epochs, _step, len(x_in), name)
    V.eval()
    return V


def train_segmenter(name, opt, lab, cfg, epochs):
    S = UNetG(3, 10, base=32, p_drop=0.0, out_act=None).to(DEVICE)
    oS = torch.optim.Adam(S.parameters(), lr=2e-4)
    holder = {"o": opt, "l": lab}

    def _step(idx):
        x = to_dev(holder["o"], idx)
        y = torch.from_numpy(holder["l"][idx].astype(np.int64)).to(DEVICE)
        if np.random.rand() < 0.5:
            x, y = x.flip(-1), y.flip(-1)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            lg = S(x)
        loss = F.cross_entropy(lg.float(), y, ignore_index=255)
        oS.zero_grad(set_to_none=True)
        loss.backward()
        oS.step()
        return {"ce": loss.item()}

    train_generic(name, cfg, {"S": S}, {"S": oS}, epochs, _step, len(opt), name)
    S.eval()
    return S


print("training loops ready")

In [ ]:
# ===== Cell 7: generators on a batch, trust scores, per-patch trust/error histograms =====
def build_or_load_models(cfg, train_data=None):
    """Trains (or loads from the bucket) all generators and the floor network on the TRAIN scenes."""
    tag = cfg["tag"]
    bank = {}
    seed_all(cfg["seed"])
    bank["floor"] = train_floor("floor_net", train_data, cfg, cfg["epochs_floor"])
    seed_all(cfg["seed"] + 1)
    bank["unet_l1"] = train_unet_l1("unet_l1", train_data, ["sar"], cfg, cfg["epochs_unet"], use_gan=False)
    seed_all(cfg["seed"] + 2)
    bank["pix2pix"] = train_unet_l1("pix2pix", train_data, ["sar"], cfg, cfg["epochs_gan"], use_gan=True)
    seed_all(cfg["seed"] + 3)
    bank["flow"] = train_flow("flow", train_data, cfg, cfg["epochs_flow"])
    seed_all(cfg["seed"] + 4)
    bank["cr_unet"] = train_unet_l1("cr_unet", train_data, ["sar", "cld"], cfg, cfg["epochs_cr"], use_gan=False)
    clim_rel = f"{tag}/models/climatology.json"
    if hf_pull(clim_rel):
        bank["clim"] = np.array(load_json(clim_rel), np.float32)
    else:
        bank["clim"] = np.array([float(train_data["opt"][:, b].astype(np.float32).mean()) for b in range(3)], np.float32)
        save_json(clim_rel, bank["clim"].tolist())
    return bank


def models_all_final(cfg):
    tag = cfg["tag"]
    names = ["floor_net", "unet_l1", "pix2pix", "flow", "cr_unet"]
    return all(hf_pull(f"{tag}/models/{n}_final.pt") for n in names) and hf_pull(f"{tag}/models/climatology.json")


@torch.no_grad()
def run_generators(bank, sar, cld, cfg):
    """All generators on one batch. Returns outputs, sample spreads, floor maps and the TopoGuard edge mask."""
    tg, K, M = cfg["topoguard"], cfg["K"], cfg["mix_components"]
    B, _, H, W = sar.shape
    out, spread = {}, {}
    with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
        raw = bank["floor"](sar)
    lpi, mu, sg = floor_params(raw, M)
    floors = {}
    for tau in cfg["taus"]:
        fb, cstar = floor_maps(lpi, mu, sg, tau)
        floors[tau] = fb
        if abs(tau - cfg["tau_main"]) < 1e-9:
            out["floor_center"] = cstar
    out["climatology"] = torch.tensor(bank["clim"], device=DEVICE)[None, :, None, None].expand(B, 3, H, W).clone()
    with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
        out["unet_l1"] = bank["unet_l1"](sar).float()
        set_mc_dropout(bank["pix2pix"], False)
        out["pix2pix"] = bank["pix2pix"](sar).float()
        set_mc_dropout(bank["pix2pix"], True)
        mcd = torch.stack([bank["pix2pix"](sar).float() for _ in range(K)])
        set_mc_dropout(bank["pix2pix"], False)
        out["cr_unet"] = bank["cr_unet"](torch.cat([sar, cld], 1)).float()
    out["pix2pix_mcd"] = mcd[0]
    spread["pix2pix_mcd"] = mcd.std(0).amax(1)
    out["pix2pix_topoguard"], m_edge = topoguard_select_gate(mcd, sar[:, :1], tg)
    spread["pix2pix_topoguard"] = spread["pix2pix_mcd"]
    fls = torch.stack([flow_sample(bank["flow"], sar, cfg["flow_steps"]) for _ in range(K)])
    out["flow_sample"], out["flow_mean"] = fls[0], fls.mean(0)
    spread["flow_sample"] = spread["flow_mean"] = fls.std(0).amax(1)
    out["cr_cloudy_identity"] = cld.float()
    return out, spread, floors, m_edge[:, 0]


def verifier_input(gname, sar, yhat, cld):
    parts = [sar.float(), yhat.float()]
    if gname in ("cr_unet", "cr_cloudy_identity"):
        parts.append(cld.float())
    return torch.cat(parts, 1)


def gen_scores(gname, out, spread, floors, m_edge, verifiers, sar, cld, cfg):
    """Score maps (higher = less trustworthy) available for generator gname, each (B,H,W) float32."""
    s = {"floor": floors[cfg["tau_main"]].amax(1).float(), "topo": (1.0 - m_edge).float()}
    if gname in spread:
        s["spread"] = spread[gname].float()
    if verifiers is not None and gname in verifiers:
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            s["learned"] = torch.sigmoid(verifiers[gname](verifier_input(gname, sar, out[gname], cld)).float())[:, 0]
    return s


def all_gens(cfg):
    return cfg["s2o_gens"] + cfg["cr_gens"]


def scatter_hist(idx, w, nb):
    o = torch.zeros(idx.shape[0], nb, device=idx.device, dtype=torch.float32)
    o.scatter_add_(1, idx, w)
    return o


@torch.no_grad()
def eval_batch(bank, verifiers, edges, sar, opt, cld, cfg, seg_model=None, lab=None):
    """Per-patch histograms for every (generator, score, error type, tau). Returns numpy dict for this batch."""
    nb, taus = cfg["n_bins"], cfg["taus"]
    out, spread, floors, m_edge = run_generators(bank, sar, cld, cfg)
    y = opt.float()
    y_low = lowpass(y)
    res = {}
    if seg_model is not None:
        valid = (lab != 255)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            pr = seg_model(y).argmax(1)
        res["segreal"] = np.array([float(valid.sum()), float(((pr == lab) & valid).sum())])
    for g in all_gens(cfg):
        yh = out[g].float()
        e_full = (yh - y).abs().amax(1)
        e_low = (lowpass(yh) - y_low).abs().amax(1)
        res[f"psnr|{g}"] = psnr_batch(yh, y).cpu().numpy()
        res[f"ssim|{g}"] = ssim_batch(yh, y).cpu().numpy()
        scores = gen_scores(g, out, spread, floors, m_edge, verifiers, sar, cld, cfg)
        ones = torch.ones(e_full.shape[0], e_full[0].numel(), device=DEVICE)
        for sname, smap in scores.items():
            ed = edges.get(g, {}).get(sname)
            if ed is None:
                continue
            idx = torch.bucketize(smap.flatten(1).contiguous(), torch.tensor(ed, device=DEVICE, dtype=torch.float32))
            res[f"cnt|{g}|{sname}"] = scatter_hist(idx, ones, nb).cpu().numpy().astype(np.int32)
            for ename, e in (("full", e_full), ("low", e_low)):
                for ti, tau in enumerate(taus):
                    res[f"bad|{g}|{sname}|{ename}|{ti}"] = scatter_hist(idx, (e > tau).flatten(1).float(), nb).cpu().numpy().astype(np.int32)
            if seg_model is not None and sname == "learned" and g in cfg["s2o_gens"]:
                with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
                    pg = seg_model(yh).argmax(1)
                v = (lab != 255).flatten(1).float()
                c = ((pg == lab) & (lab != 255)).flatten(1).float()
                res[f"seg|{g}"] = np.stack([scatter_hist(idx, v, nb).sum(0).cpu().numpy(),
                                            scatter_hist(idx, c, nb).sum(0).cpu().numpy()], 1)
        if g in cfg["s2o_gens"]:
            h1 = np.zeros((len(taus), cfg["n_floor_bins"], 3))
            for ti, tau in enumerate(taus):
                fb = floors[tau]
                fbin = (fb * cfg["n_floor_bins"]).long().clamp(0, cfg["n_floor_bins"] - 1).flatten()
                wr = ((yh - y).abs() > tau).float().flatten()
                for col, w in enumerate((torch.ones_like(wr), wr, fb.flatten().float())):
                    h1[ti, :, col] = torch.zeros(cfg["n_floor_bins"], device=DEVICE).scatter_add_(0, fbin, w).cpu().numpy()
            res[f"h1|{g}"] = h1
    return res


def merge_batch_results(parts):
    keys = parts[0].keys()
    merged = {}
    for k in keys:
        if k.startswith(("h1|", "seg|", "segreal")):
            merged[k] = np.sum([p[k] for p in parts], axis=0)
        else:
            merged[k] = np.concatenate([p[k] for p in parts], axis=0)
    return merged

In [ ]:
# ===== Cell 8: stages — verifiers + score bin edges (AUX scenes), evaluation passes, DFC2020, SAR2Opt =====
def verifier_stage(cfg, bank, aux):
    """Runs every generator on AUX scenes, trains one learned verifier per generator, and fixes score bin edges
    from AUX score quantiles (no calibration or test labels are used here)."""
    tag = cfg["tag"]
    rel_edges = f"{tag}/models/score_edges.json"
    gens = all_gens(cfg)
    if hf_pull(rel_edges) and all(hf_pull(f"{tag}/models/verifier_{g}_final.pt") for g in gens):
        edges = load_json(rel_edges)
        verifiers = {}
        for g in gens:
            V = UNetG(8 if g.startswith("cr_") else 5, 1, base=32, p_drop=0.0, out_act=None).to(DEVICE)
            V.load_state_dict(load_ckpt(f"{tag}/models/verifier_{g}_final.pt")["model_V"])
            verifiers[g] = V.eval()
        log_line(tag, "verifiers and score edges loaded from bucket")
        return verifiers, edges
    n = len(aux["opt"])
    ram_guard(n * 65536 * (3 * len(gens) + 1) / 2**30, "aux generator outputs (uint8)")
    yhat_u8 = {g: np.empty((n, 3, 256, 256), np.uint8) for g in gens}
    samples = {g: {} for g in gens}
    rng = np.random.default_rng(0)
    bs = cfg["batch"]
    for i0 in range(0, n, bs):
        check_deadline("aux generators")
        sl = slice(i0, min(n, i0 + bs))
        sar, cld = to_dev(aux["sar"], sl), to_dev(aux["cld"], sl)
        out, spread, floors, m_edge = run_generators(bank, sar, cld, cfg)
        pick = torch.from_numpy(rng.choice(65536, 4096, replace=False)).to(DEVICE)
        for g in gens:
            yhat_u8[g][sl] = (out[g].clamp(0, 1) * 255).round().byte().cpu().numpy()
            for sname, smap in gen_scores(g, out, spread, floors, m_edge, None, sar, cld, cfg).items():
                samples[g].setdefault(sname, []).append(smap.flatten(1)[:, pick].cpu().numpy().ravel())
    log_line(tag, f"aux generator pass done on {n} patches")
    edges, verifiers = {}, {}
    qs = np.linspace(0, 1, cfg["n_bins"] + 1)[1:-1]
    for g in gens:
        x_in = np.concatenate([aux["sar"], (yhat_u8[g].astype(np.float16) / 255).astype(np.float16)] +
                              ([aux["cld"]] if g.startswith("cr_") else []), 1)
        wrong = (np.abs(yhat_u8[g].astype(np.float32) / 255 - aux["opt"].astype(np.float32)).max(1) > cfg["tau_main"])
        seed_all(cfg["seed"] + 10)
        verifiers[g] = train_verifier(f"verifier_{g}", x_in, wrong, cfg, cfg["epochs_verifier"])
        edges[g] = {s: np.quantile(np.concatenate(v), qs).astype(float).tolist() for s, v in samples[g].items()}
        lv = []
        for i0 in range(0, n, bs):
            sl = slice(i0, min(n, i0 + bs))
            with torch.no_grad(), torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
                pv = torch.sigmoid(verifiers[g](to_dev(x_in, sl)).float())[:, 0].flatten(1)
            lv.append(pv[:, torch.from_numpy(rng.choice(65536, 4096, replace=False)).to(DEVICE)].cpu().numpy().ravel())
        edges[g]["learned"] = np.quantile(np.concatenate(lv), qs).astype(float).tolist()
        del x_in, wrong
        trim_ram()
        log_line(tag, f"verifier + edges done for {g} (aux error rate at tau_main recorded in training log)")
    save_json(rel_edges, edges)
    del yhat_u8, samples
    trim_ram()
    return verifiers, edges


def eval_items(cfg, bank, verifiers, edges, domain, items, loader, seg_model=None):
    """items: list of names; loader(name) -> dict(sar, opt, cld[, lab]) numpy. One histogram file per item."""
    tag = cfg["tag"]
    done = 0
    for name in items:
        rel = f"{tag}/hist/{domain}/{name}.npz"
        if hf_pull(rel):
            done += 1
            continue
        check_deadline(f"eval {domain}")
        d = loader(name)
        n = len(d["opt"])
        parts = []
        for i0 in range(0, n, cfg["batch"]):
            sl = slice(i0, min(n, i0 + cfg["batch"]))
            lab = torch.from_numpy(d["lab"][sl].astype(np.int64)).to(DEVICE) if "lab" in d else None
            parts.append(eval_batch(bank, verifiers, edges, to_dev(d["sar"], sl), to_dev(d["opt"], sl),
                                    to_dev(d["cld"], sl), cfg, seg_model, lab))
        save_npz(rel, **merge_batch_results(parts))
        done += 1
        del d, parts
        log_line(tag, f"eval {domain}: {done}/{len(items)} ({name})")
    gpu_free()


def sen12mscr_item_loader(cfg, scene_names):
    def _load(name):
        season, scene = name.split("_")
        d = load_sen12mscr_split([[season, int(scene)]], 1, cfg["eval_patches_per_scene"], cfg, name)
        return {"sar": d["sar"], "opt": d["opt"], "cld": d["cld"]}
    return _load


def dfc_stage(cfg, bank, verifiers, edges):
    tag = cfg["tag"]
    seg_rel = f"{tag}/models/dfc_segmenter_final.pt"
    if not hf_pull(seg_rel):
        tr = load_dfc2020("0", cfg["dfc_train_max"], cfg, with_sar=False)
        seed_all(cfg["seed"] + 20)
        train_segmenter("dfc_segmenter", tr["opt"], tr["lab"], cfg, cfg["epochs_seg"])
        del tr
        trim_ram()
    S = UNetG(3, 10, base=32, p_drop=0.0, out_act=None).to(DEVICE)
    S.load_state_dict(load_ckpt(seg_rel)["model_S"])
    S.eval()
    va = load_dfc2020("validation", cfg["dfc_val_max"], cfg, with_sar=True)
    n = len(va["opt"])
    chunk = 64
    names = [f"chunk{i:03d}" for i in range((n + chunk - 1) // chunk)]
    save_json(f"{tag}/hist/dfc_names.json", names)

    def _load(name):
        i = int(name[5:])
        sl = slice(i * chunk, min(n, (i + 1) * chunk))
        # DFC2020 has no cloudy optical; the cloud-removal models receive the clear image (documented limitation).
        return {"sar": va["sar"][sl], "opt": va["opt"][sl], "cld": va["opt"][sl], "lab": va["lab"][sl]}

    eval_items(cfg, bank, verifiers, edges, "dfc", names, _load, seg_model=S)
    del va, S
    trim_ram()
    gpu_free()


def sar2opt_stage(cfg):
    """Second sensor: TerraSAR-X 1 m (8-bit) -> Google Earth RGB. Pix2Pix (1-channel input) trained on the
    authors' train split; MC-dropout sample + spread; TopoGuard edge score. Units: tile / 1200-px block / city."""
    tag = cfg["tag"]
    rel = f"{tag}/hist/sar2opt/hist.npz"
    if hf_pull(rel):
        return load_npz(rel)
    tr = load_sar2opt("train", cfg["sar2opt_train_max"])
    te = load_sar2opt("test", cfg["sar2opt_test_max"])
    G = UNetG(1, 3).to(DEVICE)
    D = PatchD(4).to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    oD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    def _step(idx):
        r0, c0 = np.random.randint(0, 600 - 256, 2)
        x = torch.from_numpy(tr["sar"][idx, :, r0:r0 + 256, c0:c0 + 256]).to(DEVICE).float() / 255
        y = torch.from_numpy(tr["opt"][idx, :, r0:r0 + 256, c0:c0 + 256]).to(DEVICE).float() / 255
        x, y = rand_flip(x, y)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            yh = G(x)
            ld = 0.5 * (((D(torch.cat([x, y], 1)).float() - 1) ** 2).mean() + (D(torch.cat([x, yh.detach()], 1)).float() ** 2).mean())
        oD.zero_grad(set_to_none=True)
        ld.backward()
        oD.step()
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            l1 = (yh.float() - y).abs().mean()
            lg = 100 * l1 + ((D(torch.cat([x, yh], 1)).float() - 1) ** 2).mean()
        oG.zero_grad(set_to_none=True)
        lg.backward()
        oG.step()
        return {"L1": l1.item(), "D": ld.item()}

    seed_all(cfg["seed"] + 30)
    train_generic("sar2opt_pix2pix", cfg, {"G": G, "D": D}, {"G": oG, "D": oD}, cfg["epochs_sar2opt"], _step, len(tr["sar"]), "sar2opt")
    G.eval()
    tg, K, nb = cfg["topoguard"], cfg["K"], cfg["n_bins"]
    qs = np.linspace(0, 1, nb + 1)[1:-1]

    @torch.no_grad()
    def _maps(sar_u8):
        x = torch.from_numpy(sar_u8[:, :, 44:556, 44:556]).to(DEVICE).float() / 255
        set_mc_dropout(G, True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            smp = torch.stack([G(x).float() for _ in range(K)])
        set_mc_dropout(G, False)
        m_edge = topoguard_edge_mask(x, tg)[0][:, 0]
        return smp[0], {"topo": 1 - m_edge, "spread": smp.std(0).amax(1)}

    smp_edges = {"topo": [], "spread": []}
    for i0 in range(0, min(len(tr["sar"]), 256), 8):
        _, sc = _maps(tr["sar"][i0:i0 + 8])
        for k in sc:
            smp_edges[k].append(sc[k].flatten()[::97].cpu().numpy())
    ed = {k: torch.tensor(np.quantile(np.concatenate(v), qs), device=DEVICE, dtype=torch.float32) for k, v in smp_edges.items()}
    res = {}
    for i0 in range(0, len(te["sar"]), 8):
        yh, sc = _maps(te["sar"][i0:i0 + 8])
        y = torch.from_numpy(te["opt"][i0:i0 + 8, :, 44:556, 44:556]).to(DEVICE).float() / 255
        e = (yh - y).abs().amax(1).flatten(1)
        ones = torch.ones_like(e)
        part = {"psnr": psnr_batch(yh, y).cpu().numpy()}
        for k, smap in sc.items():
            idx = torch.bucketize(smap.flatten(1).contiguous(), ed[k])
            part[f"cnt|{k}"] = scatter_hist(idx, ones, nb).cpu().numpy().astype(np.int32)
            for ti, tau in enumerate(cfg["taus"]):
                part[f"bad|{k}|{ti}"] = scatter_hist(idx, (e > tau).float(), nb).cpu().numpy().astype(np.int32)
        for k, v in part.items():
            res.setdefault(k, []).append(v)
    res = {k: np.concatenate(v) for k, v in res.items()}
    res["parts"] = te["parts"]
    save_npz(rel, **res)
    log_line(tag, f"SAR2Opt done: {len(te['sar'])} test tiles, {len(np.unique(te['parts'][:, 0]))} cities")
    del tr, te, G, D
    gpu_free()
    return res


print("stages ready")

In [ ]:
# ===== Cell 9: statistics — CRC (patch vs scene units), WSR PAC bound, resampling, shift, label budget =====
P_PIX = 256 * 256


def load_domain(cfg, domain, names):
    """Stacks per-item histogram files into arrays; 'unit' = index of the item (scene) for every patch."""
    data, unit = {}, []
    for ui, nm in enumerate(names):
        rel = f"{cfg['tag']}/hist/{domain}/{nm}.npz"
        if not hf_pull(rel):
            continue
        d = load_npz(rel)
        npatch = len(d[f"psnr|{cfg['s2o_gens'][0]}"])
        unit.extend([ui] * npatch)
        for k, v in d.items():
            data.setdefault(k, []).append(v)
    out = {}
    for k, v in data.items():
        out[k] = np.sum(v, axis=0) if k.startswith(("h1|", "seg|", "segreal")) else np.concatenate(v, axis=0)
    out["unit"] = np.array(unit, np.int32)
    return out


def curves(dom, g, s, e="full", ti=1, pix=P_PIX):
    """K_i(j) = trusted fraction, L_i(j) = trusted-and-wrong fraction, for bins 0..j (trust if bin <= j)."""
    cnt, bad = dom.get(f"cnt|{g}|{s}"), dom.get(f"bad|{g}|{s}|{e}|{ti}")
    if cnt is None or bad is None:
        return None, None
    return np.cumsum(cnt, 1) / pix, np.cumsum(bad, 1) / pix


def unit_mean(L, unit):
    """Mean of per-patch curves within each unit (scene). Rows ordered by unit id."""
    order = np.argsort(unit, kind="stable")
    us = unit[order]
    starts = np.r_[0, np.nonzero(np.diff(us))[0] + 1]
    sums = np.add.reduceat(L[order], starts, axis=0)
    return sums / np.diff(np.r_[starts, len(us)])[:, None]


def crc_pick(Lu, alpha):
    """Largest bin j with (sum_i L_i(j) + 1)/(n + 1) <= alpha (loss bounded by 1). -1 = refuse (trust nothing)."""
    n = Lu.shape[0]
    if n == 0:
        return -1
    ok = np.nonzero((Lu.sum(0) + 1.0) / (n + 1) <= alpha)[0]
    return int(ok.max()) if ok.size else -1


def wsr_ucb(z, delta, grid_n=2001):
    """Waudby-Smith & Ramdas hedged betting upper confidence bound for the mean of z in [0,1].
    Valid bet size for testing 'mean >= m' against 'mean < m' must be < 1/(1-m); we use 0.75/(1-m)."""
    z = np.asarray(z, float)
    n = len(z)
    if n == 0:
        return 1.0
    mu_hat = (0.5 + np.cumsum(z)) / (np.arange(1, n + 1) + 1)
    sig2 = (0.25 + np.cumsum((z - mu_hat) ** 2)) / (np.arange(1, n + 1) + 1)
    sig2_prev = np.r_[0.25, sig2[:-1]]
    lam = np.sqrt(2 * np.log(1 / delta) / (n * sig2_prev))
    for m in np.linspace(1, 0, grid_n):
        lm = np.minimum(lam, 0.75 / max(1.0 - m, 1e-6))
        cap = np.cumprod(1 + lm * (m - z))
        if cap.max() < 1 / delta:
            return float(m)
    return 0.0


def pac_pick(Lu, alpha, delta=0.1):
    """Largest j (fixed sequence from j=0 upward; absolute risk is monotone) whose WSR upper bound <= alpha."""
    j = -1
    for jj in range(Lu.shape[1]):
        if wsr_ucb(Lu[:, jj], delta) <= alpha:
            j = jj
        else:
            break
    return j


def split_resample(L, Kt, unit, alpha, R, seed, frac=0.5, modes=("patch", "scene")):
    """Repeatedly split SCENES into calibration / test. Patch-level CRC treats patches as exchangeable,
    scene-level CRC averages losses per scene. Returns test risk and trusted area for both."""
    rng = np.random.default_rng(seed)
    units = np.unique(unit)
    nc = int(round(frac * len(units)))
    res = {m: {"risk": [], "trust": [], "refused": []} for m in modes}
    for _ in range(R):
        perm = rng.permutation(units)
        cal = np.isin(unit, perm[:nc])
        te = ~cal
        for mode in modes:
            Lu = L[cal] if mode == "patch" else unit_mean(L[cal], unit[cal])
            j = crc_pick(Lu, alpha)
            res[mode]["refused"].append(j < 0)
            res[mode]["risk"].append(L[te][:, j].mean() if j >= 0 else 0.0)
            res[mode]["trust"].append(Kt[te][:, j].mean() if j >= 0 else 0.0)
    return {m: {k: np.array(v) for k, v in d.items()} for m, d in res.items()}


def icc_oneway(x, unit):
    """Intra-class correlation of a per-patch quantity within scenes (one-way ANOVA estimator)."""
    u, inv = np.unique(unit, return_inverse=True)
    k = len(u)
    if k < 2:
        return float("nan"), float("nan")
    ns = np.bincount(inv)
    means = np.bincount(inv, weights=x) / ns
    grand = x.mean()
    msb = (ns * (means - grand) ** 2).sum() / (k - 1)
    msw = ((x - means[inv]) ** 2).sum() / max(len(x) - k, 1)
    n0 = (len(x) - (ns**2).sum() / len(x)) / (k - 1)
    icc = (msb - msw) / (msb + (n0 - 1) * msw + 1e-12)
    return float(icc), float(n0)


def bootstrap_mean_ci(vals_per_unit, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(vals_per_unit, float)
    if len(v) < 2:
        return float(v.mean()) if len(v) else float("nan"), float("nan"), float("nan")
    bs = v[rng.integers(0, len(v), (B, len(v)))].mean(1)
    return float(v.mean()), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

In [ ]:
# ===== Cell 10: hypothesis tests H1–H7 (+H2b) and the detailed report =====
def verdict(cond, insufficient=False):
    return "INSUFFICIENT" if insufficient else ("PASS" if cond else "FAIL")


def analyse(cfg, split):
    tag, a0, t_main = cfg["tag"], cfg["alpha_main"], cfg["taus"].index(cfg["tau_main"])
    R = cfg["resample_R"]
    cal = load_domain(cfg, "cal", [f"{a}_{b}" for a, b in split["cal"]])
    win = load_domain(cfg, "winter", [f"{a}_{b}" for a, b in split["winter"]])
    dfc_names = load_json(f"{tag}/hist/dfc_names.json") if hf_pull(f"{tag}/hist/dfc_names.json") else []
    dfc = load_domain(cfg, "dfc", dfc_names)
    n_cal_units = len(np.unique(cal["unit"]))
    out = {"n_cal_scenes": n_cal_units, "n_cal_patches": int(len(cal["unit"])),
           "n_winter_scenes": int(len(np.unique(win["unit"]))) if len(win["unit"]) else 0,
           "n_dfc_patches": int(len(dfc["unit"])) if len(dfc["unit"]) else 0}
    s2o, gens = cfg["s2o_gens"], all_gens(cfg)

    # ---- per-generator quality and certified trusted area (scene-level CRC) for every score ----
    table = {}
    for g in gens:
        row = {"psnr": float(cal[f"psnr|{g}"].mean()), "ssim": float(cal[f"ssim|{g}"].mean())}
        for s in cfg["scores"]:
            for e in ("full", "low"):
                for alpha in cfg["alphas"]:
                    Kt, L = curves(cal, g, s, e, t_main)
                    if L is None:
                        continue
                    sr = split_resample(L, Kt, cal["unit"], alpha, R, seed=1, modes=("scene",))
                    row[f"cta|{s}|{e}|{alpha}"] = float(sr["scene"]["trust"].mean())
                    row[f"risk|{s}|{e}|{alpha}"] = float(sr["scene"]["risk"].mean())
                    row[f"refused|{s}|{e}|{alpha}"] = float(sr["scene"]["refused"].mean())
        table[g] = row
    out["table"] = table

    # ---- H1: identifiability floor never beaten (S2O generators; band-pixel level) ----
    h1 = {}
    for g in s2o:
        H = cal.get(f"h1|{g}")
        if H is None:
            continue
        per_tau = {}
        for ti, tau in enumerate(cfg["taus"]):
            n, w, sf = H[ti, :, 0], H[ti, :, 1], H[ti, :, 2]
            ok = n >= 2000
            emp = np.where(n > 0, w / np.maximum(n, 1), np.nan)
            fl = np.where(n > 0, sf / np.maximum(n, 1), np.nan)
            viol = ok & (emp < fl - 0.02)
            per_tau[str(tau)] = {"bins_used": int(ok.sum()), "violating_bins": int(viol.sum()),
                                 "violation_frac": float(viol.sum() / max(ok.sum(), 1)),
                                 "mean_floor": float(sf.sum() / max(n.sum(), 1)),
                                 "error_rate": float(w.sum() / max(n.sum(), 1)),
                                 "excess": float((w.sum() - sf.sum()) / max(n.sum(), 1)),
                                 "worst_gap": float(np.nanmin(np.where(ok, emp - fl, np.nan))) if ok.any() else float("nan")}
        h1[g] = per_tau
    fc = h1.get("floor_center", {}).get(str(cfg["tau_main"]), {})
    worst_viol = max((v[str(cfg["tau_main"])]["violation_frac"] for v in h1.values()), default=float("nan"))
    out["H1"] = {"per_generator": h1, "floor_center_calibration_gap": fc.get("error_rate", np.nan) - fc.get("mean_floor", np.nan),
                 "worst_violation_frac": worst_viol,
                 "verdict": verdict(worst_viol <= 0.10, insufficient=not h1 or all(v[str(cfg["tau_main"])]["bins_used"] == 0 for v in h1.values()))}

    # ---- H2: patch-level vs scene-level calibration (primary: pix2pix, learned score, tau_main, alpha_main) ----
    h2 = {}
    for g in s2o:
        Kt, L = curves(cal, g, "learned", "full", t_main)
        if L is None:
            continue
        sr = split_resample(L, Kt, cal["unit"], a0, R, seed=2)
        jj = crc_pick(L, a0)
        icc, n0 = icc_oneway(L[:, jj] if jj >= 0 else L[:, -1], cal["unit"])
        h2[g] = {"P_over_patch": float((sr["patch"]["risk"] > a0).mean()), "P_over_scene": float((sr["scene"]["risk"] > a0).mean()),
                 "mean_risk_patch": float(sr["patch"]["risk"].mean()), "mean_risk_scene": float(sr["scene"]["risk"].mean()),
                 "trust_patch": float(sr["patch"]["trust"].mean()), "trust_scene": float(sr["scene"]["trust"].mean()),
                 "refused_scene": float(sr["scene"]["refused"].mean()), "icc_loss": icc, "patches_per_scene": n0,
                 "kish_effective_n": float(len(L) / (1 + (n0 - 1) * max(icc, 0))) if np.isfinite(icc) else float("nan")}
    prim = h2.get("pix2pix", {})
    insuff2 = (not prim) or prim.get("refused_scene", 1) > 0.5 or n_cal_units // 2 < math.ceil(1 / a0) - 1
    out["H2"] = {"per_generator": h2, "primary": "pix2pix",
                 "verdict": verdict(prim.get("P_over_patch", 0) >= 2 * prim.get("P_over_scene", 0) and prim.get("P_over_patch", 0) > 0, insuff2)}

    # ---- H3: CTA ranking differs from PSNR ranking (Kendall tau < 0.6) over S2O generators ----
    ps = [table[g]["psnr"] for g in s2o]
    ct = [table[g].get(f"cta|learned|full|{a0}", np.nan) for g in s2o]
    kt = float(sstats.kendalltau(ps, ct)[0]) if np.all(np.isfinite(ct)) and np.std(ct) > 0 else float("nan")
    out["H3"] = {"psnr": dict(zip(s2o, ps)), "cta": dict(zip(s2o, ct)), "kendall_tau": float(kt),
                 "verdict": verdict(np.isfinite(kt) and kt < 0.6, insufficient=not np.isfinite(kt))}

    # ---- H4: TopoGuard edge score beats sample spread for the MC-dropout GAN ----
    Kt1, L1 = curves(cal, "pix2pix_mcd", "topo", "full", t_main)
    Kt2, L2 = curves(cal, "pix2pix_mcd", "spread", "full", t_main)
    if L1 is not None and L2 is not None:
        r1 = split_resample(L1, Kt1, cal["unit"], a0, R, seed=3, modes=("scene",))["scene"]["trust"]
        r2 = split_resample(L2, Kt2, cal["unit"], a0, R, seed=3, modes=("scene",))["scene"]["trust"]
        d = r1 - r2
        out["H4"] = {"cta_topo": float(r1.mean()), "cta_spread": float(r2.mean()), "mean_diff": float(d.mean()),
                     "P_topo_better": float((d > 0).mean()),
                     "verdict": verdict(d.mean() > 0 and (d > 0).mean() >= 0.9, insufficient=(r1.max() == 0 and r2.max() == 0))}
    else:
        out["H4"] = {"verdict": "INSUFFICIENT"}

    # ---- H5: season shift (winter) and new-year/region shift (DFC2020); recalibration label budget ----
    h5 = {}
    for g in ["pix2pix", "flow_sample", "unet_l1"]:
        Kc, Lc = curves(cal, g, "learned", "full", t_main)
        if Lc is None:
            continue
        row = {}
        for alpha in cfg["alphas"]:
            j = crc_pick(unit_mean(Lc, cal["unit"]), alpha)
            ent = {"lambda_bin": j, "risk_cal_in_sample": float(Lc[:, j].mean()) if j >= 0 else 0.0}
            for nm, dom in (("winter", win), ("dfc", dfc)):
                Kd, Ld = curves(dom, g, "learned", "full", t_main) if len(dom["unit"]) else (None, None)
                if Ld is None:
                    continue
                per_unit = unit_mean(Ld, dom["unit"])[:, j] if j >= 0 else np.zeros(len(np.unique(dom["unit"])))
                m, lo, hi = bootstrap_mean_ci(per_unit)
                ent[f"{nm}_plain_risk"], ent[f"{nm}_plain_ci"] = m, [lo, hi]
                ent[f"{nm}_plain_trust"] = float(Kd[:, j].mean()) if j >= 0 else 0.0
            kneed = math.ceil(1 / alpha) - 1
            if len(win["unit"]):
                Kw, Lw = curves(win, g, "learned", "full", t_main)
                wu = np.unique(win["unit"])
                if len(wu) >= kneed + 1:
                    rng = np.random.default_rng(5)
                    rr, tt = [], []
                    for _ in range(R):
                        p = rng.permutation(wu)
                        cmask = np.isin(win["unit"], p[:kneed])
                        jw = crc_pick(unit_mean(Lw[cmask], win["unit"][cmask]), alpha)
                        rr.append(Lw[~cmask][:, jw].mean() if jw >= 0 else 0.0)
                        tt.append(Kw[~cmask][:, jw].mean() if jw >= 0 else 0.0)
                    ent["winter_recal_k"] = kneed
                    ent["winter_recal_mean_risk"], ent["winter_recal_P_over"] = float(np.mean(rr)), float(np.mean(np.array(rr) > alpha))
                    ent["winter_recal_trust"] = float(np.mean(tt))
                else:
                    ent["winter_recal_k"] = f"needs >= {kneed + 1} winter scenes, have {len(wu)}"
            row[str(alpha)] = ent
        h5[g] = row
    e5 = h5.get("pix2pix", {}).get(str(a0), {})
    broke = any(e5.get(f"{d}_plain_risk", 0) > a0 for d in ("winter", "dfc"))
    repaired = isinstance(e5.get("winter_recal_mean_risk"), float) and e5["winter_recal_mean_risk"] <= a0
    out["H5"] = {"per_generator": h5, "plain_breaks": broke, "recal_repairs": repaired,
                 "verdict": verdict(broke and repaired, insufficient=("winter_recal_mean_risk" not in e5) and not broke)}

    # ---- H6: low-pass certified area >= 2x full-band (S2O generators) ----
    ratios = {}
    for g in s2o:
        cf, cl = table[g].get(f"cta|learned|full|{a0}"), table[g].get(f"cta|learned|low|{a0}")
        if cf is not None and cl is not None:
            ratios[g] = float(cl / cf) if cf > 0 else (float("inf") if cl > 0 else float("nan"))
    good = [r for r in ratios.values() if np.isfinite(r) or r == float("inf")]
    out["H6"] = {"ratio_low_over_full": ratios, "verdict": verdict(sum(r >= 2 for r in good) > len(good) / 2, insufficient=len(good) == 0)}

    # ---- H7: land-cover agreement inside vs outside the certified mask (DFC2020) ----
    h7 = {}
    if "segreal" in dfc:
        h7["real_optical_accuracy"] = float(dfc["segreal"][1] / max(dfc["segreal"][0], 1))
        for g in s2o:
            Kc, Lc = curves(cal, g, "learned", "full", t_main)
            sg = dfc.get(f"seg|{g}")
            if Lc is None or sg is None:
                continue
            j = crc_pick(unit_mean(Lc, cal["unit"]), a0)
            vin, cin = sg[:j + 1, 0].sum() if j >= 0 else 0.0, sg[:j + 1, 1].sum() if j >= 0 else 0.0
            vout, cout = sg[j + 1:, 0].sum(), sg[j + 1:, 1].sum()
            h7[g] = {"acc_in": float(cin / vin) if vin > 0 else float("nan"), "acc_out": float(cout / vout) if vout > 0 else float("nan"),
                     "frac_labelled_pixels_trusted": float(vin / max(vin + vout, 1))}
    diffs = [v["acc_in"] - v["acc_out"] for k, v in h7.items() if isinstance(v, dict) and np.isfinite(v["acc_in"]) and np.isfinite(v["acc_out"])]
    out["H7"] = {"per_generator": h7, "verdict": verdict(sum(d >= 0.10 for d in diffs) > len(diffs) / 2, insufficient=len(diffs) == 0)}

    # ---- H2b: replication on SAR2Opt (tile vs block vs city units), alpha = 0.10 ----
    s2 = load_npz(f"{tag}/hist/sar2opt/hist.npz") if hf_pull(f"{tag}/hist/sar2opt/hist.npz") else None
    if s2 is not None:
        parts = s2["parts"]
        city = parts[:, 0]
        block = city * 10000 + (parts[:, 1] // 1200) * 100 + parts[:, 2] // 1200
        pix = 512 * 512
        res2 = {}
        for sname in ("topo", "spread"):
            Kt = np.cumsum(s2[f"cnt|{sname}"], 1) / pix
            L = np.cumsum(s2[f"bad|{sname}|{t_main}"], 1) / pix
            cities = np.unique(city)
            half = len(cities) // 2
            rng = np.random.default_rng(9)
            acc = {"tile": [], "block": [], "city": []}
            for _ in range(min(R, 300)):
                cc = rng.permutation(cities)[:half]
                cm = np.isin(city, cc)
                for mode, Lu in (("tile", L[cm]), ("block", unit_mean(L[cm], block[cm])), ("city", unit_mean(L[cm], city[cm]))):
                    j = crc_pick(Lu, 0.10)
                    acc[mode].append((L[~cm][:, j].mean() if j >= 0 else 0.0, Kt[~cm][:, j].mean() if j >= 0 else 0.0, j < 0))
            res2[sname] = {m: {"P_over": float(np.mean([r > 0.10 for r, _, _ in v])), "mean_risk": float(np.mean([r for r, _, _ in v])),
                               "trust": float(np.mean([t for _, t, _ in v])), "refused": float(np.mean([z for _, _, z in v]))} for m, v in acc.items()}
        res2["n_tiles"], res2["n_cities"], res2["n_blocks"] = int(len(city)), int(len(np.unique(city))), int(len(np.unique(block)))
        res2["psnr"] = float(s2["psnr"].mean())
        p = res2["topo"]
        out["H2b"] = {"results": res2, "verdict": verdict(p["tile"]["P_over"] >= 2 * p["block"]["P_over"] and p["tile"]["P_over"] > 0,
                                                           insufficient=p["block"]["refused"] > 0.5)}
    else:
        out["H2b"] = {"verdict": "INSUFFICIENT"}

    # ---- sensitivity: certified trusted area vs tau and alpha (scene-level CRC, learned score) ----
    sens = {}
    for g in s2o:
        for ti, tau in enumerate(cfg["taus"]):
            Kt, L = curves(cal, g, "learned", "full", ti)
            if L is None:
                continue
            for alpha in cfg["alphas"]:
                sr = split_resample(L, Kt, cal["unit"], alpha, min(R, 200), seed=11, modes=("scene",))["scene"]
                sens[f"{g}|{tau}|{alpha}"] = {"cta": float(sr["trust"].mean()), "risk": float(sr["risk"].mean()),
                                              "error_rate": float(L[:, -1].mean())}
    out["sensitivity"] = sens

    # ---- guarantee tiers: PAC (WSR) on all calibration scenes ----
    tiers = {}
    for g in ["pix2pix", "flow_sample"]:
        Kt, L = curves(cal, g, "learned", "full", t_main)
        if L is None:
            continue
        Lu = unit_mean(L, cal["unit"])
        jc, jp = crc_pick(Lu, a0), pac_pick(Lu, a0, 0.1)
        tiers[g] = {"n_scene_units": int(Lu.shape[0]), "crc_trust_in_sample": float(Kt[:, jc].mean()) if jc >= 0 else 0.0,
                    "pac_wsr_trust_in_sample": float(Kt[:, jp].mean()) if jp >= 0 else 0.0,
                    "min_scenes_for_crc": math.ceil(1 / a0) - 1}
    out["tiers"] = tiers
    return out


HYP_TEXT = {
    "H1": ("Identifiability floor is never beaten (no S2O generator below the learned floor by > 2 points in > 10% of bins, tau_main)",
           "If FAIL: the floor network is not calibrated enough to be a lower bound; report excess only through sample spread and drop the floor claim."),
    "H2": ("Patch-level calibration over-promises: P(user over alpha) for patch-level CRC >= 2x scene-level (pix2pix, learned score)",
           "If FAIL: scene clustering is minor on real EO data; the paper becomes a tool + floor + metric paper."),
    "H2b": ("Replication on a second sensor (SAR2Opt): tile-level P(over) >= 2x block-level",
            "If FAIL: the clustering effect is sensor/dataset specific; state that scope."),
    "H3": ("Certified trusted area ranks generators differently from PSNR (Kendall tau < 0.6)",
           "If FAIL: CTA adds little beyond PSNR; keep it only as a reporting aid."),
    "H4": ("TopoGuard edge-support score certifies more area than MC-dropout spread for the GAN",
           "If FAIL: drop the claim that the SAR-TopoGuard edge prior is a trust score; use spread or the learned verifier."),
    "H5": ("Season / region shift breaks plain calibration and k labelled target scenes repair it",
           "If FAIL because nothing broke: shift is benign here; if FAIL because repair failed: the score's ranking is wrong in the target domain (fine-tune the verifier)."),
    "H6": ("Low-pass content certifies >= 2x the area of full-band content",
           "If FAIL: fine detail is not the main source of untrusted pixels; band-wise certification is not worth a section."),
    "H7": ("Land-cover accuracy (DFC2020) inside the certified mask is >= 10 points higher than outside",
           "If FAIL: the mask is valid but not useful downstream; report as limitation."),
}


def hypothesis_report(cfg, res):
    a0 = cfg["alpha_main"]
    lines = [f"# CERTEO hypothesis report — run '{cfg['tag']}'", "",
             f"Calibration scenes: {res['n_cal_scenes']} ({res['n_cal_patches']} patches) | winter scenes: {res['n_winter_scenes']} | "
             f"DFC2020 patches: {res['n_dfc_patches']} | alpha = {a0}, tau = {cfg['tau_main']} (reflectance/3000 scale)", ""]
    if cfg["tag"] == "smoke":
        lines += ["> **SMOKE RUN**: small slices of the real data and short training. Use it to check that every stage works and to "
                  "see the direction of effects. Do not quote these numbers as results.", ""]
    lines += ["| Hypothesis | Statement | Verdict |", "|---|---|---|"]
    for h in ["H1", "H2", "H2b", "H3", "H4", "H5", "H6", "H7"]:
        lines.append(f"| {h} | {HYP_TEXT[h][0]} | **{res.get(h, {}).get('verdict', 'INSUFFICIENT')}** |")
    lines += ["", "## Generator table (scene-level CRC, learned score, full band)", "",
              "| generator | PSNR | SSIM | CTA@0.05 | risk@0.05 | CTA@0.10 | CTA low-pass@0.05 |", "|---|---|---|---|---|---|---|"]
    for g, r in res["table"].items():
        lines.append(f"| {g} | {r['psnr']:.2f} | {r['ssim']:.3f} | {r.get(f'cta|learned|full|{a0}', float('nan')):.3f} | "
                     f"{r.get(f'risk|learned|full|{a0}', float('nan')):.4f} | {r.get('cta|learned|full|0.1', float('nan')):.3f} | "
                     f"{r.get(f'cta|learned|low|{a0}', float('nan')):.3f} |")
    lines += ["", "## Score comparison (CTA@0.05, scene-level, full band)", "", "| generator | floor | topo | spread | learned |", "|---|---|---|---|---|"]
    for g, r in res["table"].items():
        lines.append(f"| {g} | " + " | ".join(f"{r.get(f'cta|{s}|full|{a0}', float('nan')):.3f}" for s in cfg["scores"]) + " |")
    lines += ["", "## Details per hypothesis", ""]
    for h in ["H1", "H2", "H2b", "H3", "H4", "H5", "H6", "H7"]:
        body = {k: v for k, v in res.get(h, {}).items() if k != "verdict"}
        lines += [f"### {h} — {res.get(h, {}).get('verdict', 'INSUFFICIENT')}", f"*Claim:* {HYP_TEXT[h][0]}", "",
                  f"*Pre-registered consequence:* {HYP_TEXT[h][1]}", "", "```json", json.dumps(body, indent=1, default=float)[:6000], "```", ""]
    lines += ["## Sensitivity: CTA (scene-level CRC, learned score) by tau and alpha; error rate = share of pixels off by more than tau", "",
              "| generator | tau | error rate | CTA@0.05 | CTA@0.10 |", "|---|---|---|---|---|"]
    for g in cfg["s2o_gens"]:
        for tau in cfg["taus"]:
            a = res.get("sensitivity", {}).get(f"{g}|{tau}|0.05", {})
            b = res.get("sensitivity", {}).get(f"{g}|{tau}|0.1", {})
            if a:
                lines.append(f"| {g} | {tau} | {a['error_rate']:.3f} | {a['cta']:.3f} | {b.get('cta', float('nan')):.3f} |")
    lines += ["", "## Guarantee tiers on the calibration scenes", "```json", json.dumps(res.get("tiers", {}), indent=1, default=float), "```"]
    return "\n".join(lines)

In [ ]:
# ===== Cell 11: the orchestrator (every stage resumable; every output synced to the bucket) =====
def run_certeo(cfg):
    tag = cfg["tag"]
    t_start = time.time()
    split = make_scene_split(cfg)
    save_json(f"{tag}/split.json", split)
    log_line(tag, f"split: train {len(split['train'])}, aux {len(split['aux'])}, cal {len(split['cal'])}, winter {len(split['winter'])} scenes")
    timings = {}

    t0 = time.time()
    if models_all_final(cfg):
        stub = {"opt": np.zeros((cfg["batch"], 3, 256, 256), np.float16), "sar": np.zeros((cfg["batch"], 2, 256, 256), np.float16),
                "cld": np.zeros((cfg["batch"], 3, 256, 256), np.float16)}
        bank = build_or_load_models(cfg, train_data=stub)   # every model is loaded from its final checkpoint
    else:
        tr = load_sen12mscr_split(split["train"], cfg["train_rgs_per_scene"], cfg["train_patches_per_rg"], cfg, "train")
        log_line(tag, f"train patches: {len(tr['opt'])}")
        bank = build_or_load_models(cfg, train_data=tr)
        del tr
        trim_ram()
    timings["1_train_generators_floor"] = time.time() - t0
    hf_push([f"{tag}/log.txt"])

    t0 = time.time()
    rel_edges = f"{tag}/models/score_edges.json"
    if hf_pull(rel_edges) and all(hf_pull(f"{tag}/models/verifier_{g}_final.pt") for g in all_gens(cfg)):
        verifiers, edges = verifier_stage(cfg, bank, None)
    else:
        aux = load_sen12mscr_split(split["aux"], 1, cfg["aux_patches_per_scene"], cfg, "aux")
        verifiers, edges = verifier_stage(cfg, bank, aux)
        del aux
        trim_ram()
    timings["2_verifiers_edges"] = time.time() - t0
    hf_push([f"{tag}/log.txt"])

    t0 = time.time()
    eval_items(cfg, bank, verifiers, edges, "cal", [f"{a}_{b}" for a, b in split["cal"]], sen12mscr_item_loader(cfg, None))
    eval_items(cfg, bank, verifiers, edges, "winter", [f"{a}_{b}" for a, b in split["winter"]], sen12mscr_item_loader(cfg, None))
    timings["3_eval_sen12mscr"] = time.time() - t0
    hf_push([f"{tag}/log.txt"])

    t0 = time.time()
    dfc_stage(cfg, bank, verifiers, edges)
    timings["4_dfc2020"] = time.time() - t0
    hf_push([f"{tag}/log.txt"])

    t0 = time.time()
    del bank, verifiers
    gpu_free()
    sar2opt_stage(cfg)
    timings["5_sar2opt"] = time.time() - t0
    hf_push([f"{tag}/log.txt"])

    t0 = time.time()
    res = analyse(cfg, split)
    timings["6_analysis"] = time.time() - t0
    res["timings_min"] = {k: round(v / 60, 2) for k, v in timings.items()}
    res["wall_min_this_session"] = round((time.time() - t_start) / 60, 2)
    save_json(f"{tag}/results.json", res)
    report = hypothesis_report(cfg, res)
    def _write_report(p):
        with open(p, "w") as fh:
            fh.write(report)
    atomic_write(f"{tag}/HYPOTHESIS_REPORT.md", _write_report)
    hf_push([f"{tag}/HYPOTHESIS_REPORT.md", f"{tag}/log.txt"])
    log_line(tag, f"run '{tag}' complete in {res['wall_min_this_session']} min (this session)")
    return res, report


print("orchestrator ready")

In [ ]:
# ===== Cell 12: SMOKE RUN — real data, small slices, short training; prints the hypothesis table =====
SMOKE_RESULTS, SMOKE_REPORT = run_certeo(CFG_SMOKE)
print(SMOKE_REPORT)

In [ ]:
# ===== Cell 13: FULL RUN — all scenes in the plan, full training; resumable across sessions =====
print("smoke verdicts:", {h: SMOKE_RESULTS.get(h, {}).get("verdict") for h in ["H1", "H2", "H2b", "H3", "H4", "H5", "H6", "H7"]})
if os.environ.get("CERTEO_SKIP_FULL", "0") == "1":
    FULL_RESULTS, FULL_REPORT = None, "full run skipped (CERTEO_SKIP_FULL=1)"
else:
    FULL_RESULTS, FULL_REPORT = run_certeo(CFG_FULL)
print(FULL_REPORT)

In [ ]:
# ===== Cell 14: where everything lives =====
final_summary_cell14 = {
    "storage": (f"bucket {HF_BUCKET}" if SYNC_ENABLED else f"local folder {LOCAL_ROOT}"),
    "smoke_report": bucket_key("smoke/HYPOTHESIS_REPORT.md"),
    "full_report": bucket_key("full/HYPOTHESIS_REPORT.md"),
    "results_json": [bucket_key("smoke/results.json"), bucket_key("full/results.json")],
    "models": bucket_key("full/models/"), "histograms": bucket_key("full/hist/"), "data_shards": bucket_key("data/"),
    "full_verdicts": {h: (FULL_RESULTS or {}).get(h, {}).get("verdict") for h in ["H1", "H2", "H2b", "H3", "H4", "H5", "H6", "H7"]},
}
print(json.dumps(final_summary_cell14, indent=1))

## Part II — Theory: what limits certification of cross-modal generators, and tests on the real data

**Thesis.** Certifying a SAR→optical generator is limited by two things only: **(1) how much the radar determines the optical image** (identifiability) and **(2) how many independent scenes you calibrate on**. Theorems T2–T5 state both limits exactly; cells 16–23 test every prediction on the real SEN12MS-CR histograms produced above.

Notation. X = SAR patch, Y = optical patch, Ŷ = G(X, ξ) a generator with its own randomness ξ independent of Y given X. A pixel p is *wrong* if max over RGB bands |Ŷ_p − Y_p| > τ. Floor f_p(X) = max_b [1 − Q_{τ,b}(X)], with Q_{τ,b}(X) = sup_c P(|Y_{p,b} − c| ≤ τ | X). A trust rule keeps a set T of pixels; false-trust = |T ∩ wrong| / P; certified trusted area (CTA) = |T| / P.

**T1 (floor, cell 1 of the paper).** For any generator that only sees X: P(pixel p wrong | X) ≥ f_p(X). *Proof:* given X the guess is independent of Y; any fixed guess c hits within τ in band b with probability ≤ Q_{τ,b}(X); missing in one band already makes the pixel wrong.

**T2 (certification ceiling — new).** For any SAR-only generator and any trust rule T = T(X, ξ) with E[false-trust] ≤ α, the expected trusted area is at most CTA\*(α) = the largest E|S|/P over pixel sets S(X) with E Σ_{p∈S} f_p(X) ≤ αP, i.e. fill pixels in order of increasing floor until the summed floor reaches α. *Proof:* E|T ∩ wrong| = E Σ_{p∈T} P(wrong_p | X, ξ) ≥ E Σ_{p∈T} f_p(X); maximising area under this linear budget is a fractional knapsack, solved by the lowest-floor-first rule (randomised rules are mixtures and cannot do better). *Attainment:* the τ-mode generator (output c\* that maximises the window mass) with the floor as score reaches the ceiling when f is the exact miss probability (single band, or bands independent given X). Posterior samplers cannot: their hit probability is an average of window masses, never the maximum. **Meaning: there is a hard, computable ceiling on how much of any translated image can ever be certified, set by the radar alone; and the generator that maximises trust is neither the mean nor a sampler, but the τ-mode.**

**T3 (unit equivalence — new, exact).** With n patches in each of m calibration scenes, patch-level CRC at level α makes exactly the same decision as scene-level CRC at level α′ = α + (1 − α)(1 − 1/n)/(m + 1). Hence its guarantee for a new scene is only E[risk] ≤ α′, and for step-shaped losses E[risk] = ⌊α′(m+1)⌋/(m+1), which exceeds α whenever ⌊α′(m+1)⌋ > ⌊α(m+1)⌋. *Proof:* Σ_{patches} L = n Σ_{scenes} L̄, substitute in (ΣL + 1)/(N + 1) ≤ α and solve. **Meaning: the excess (up to (1−α)/(m+1)) does not shrink with more patches; only more scenes remove it.**

**T4 (training-conditional law — new).** Losses are monotone in the threshold. Let j⁺ be the first threshold whose population risk R(j⁺) exceeds α. Then exactly {user over budget} = {λ̂ ≥ j⁺} = {Â(j⁺) ≤ t}, where Â is the mean of m independent scene losses and t = α − (1−α)/m (scene-level) or α − (1−α)/(mn) (patch-level). By the CLT, P(over) ≈ Φ((t − R(j⁺))·√m / τ_n), with τ_n² = σ_b² + σ_w²/n (between-scene and within-scene variance). **Consequences:** patch-level P(over) climbs toward 1/2 as patches per scene grow; scene-level P(over) stays near a floor set by m and σ_b. Reliability is bought with scenes, not patches.

**T5 (studentised scene PAC — procedure).** λ̂ = max{j : Â(j) + t_{m−1, 1−δ} · τ̂(j)/√m ≤ α} gives P(over) → δ as m grows (asymptotic; the exact finite-sample alternative is the WSR bound of cell 9).

**T6 (concept-shift impossibility, from Part I).** If p(Y | X) changes but p(X) does not, no label-free method can keep risk ≤ α in both worlds.

**Pre-registered hypotheses for Part II (written before any real-data run of these cells):**

| ID | Prediction | Kill rule / consequence |
|---|---|---|
| H8 | No SAR-only (generator, score) certifies more oracle area than CTA\*(α) + 0.02, α ∈ {0.05, 0.10} | Violation ⇒ the floor network under-estimates the floor (not calibrated); T2 then holds only for the true floor, and we must report the floor as an estimate. |
| H9 | The τ-mode generator (floor_center) with the floor score certifies at least as much oracle area as the posterior samplers (flow_sample, pix2pix_mcd) with the same score, α = 0.05 | Failure ⇒ the floor net's τ-mode is poor in practice; T2's attainment claim is reported as theory only. |
| H10 | Patch-level CRC at α and scene-level CRC at α′ choose the same threshold in every real resample (equal patches per scene) | Any mismatch ⇒ a bug in code or in the derivation of T3. |
| H11 | In the real-data scene superpopulation, mean risk of patch-level CRC ≤ α′ (+2 s.e.) and of scene-level CRC ≤ α (+2 s.e.) in every configuration | Failure ⇒ CRC assumptions are violated on real data (e.g. non-monotone losses). |
| H12 | T4's prediction of P(over) matches the observed value within 0.05 everywhere; patch-level P(over) rises with n; scene-level stays ≤ patch-level (+0.02); the exact event identity holds in every draw | Failure ⇒ T4 is not an accurate description of real EO data; keep only the exact identity. |
| H13 | Studentised scene PAC: P(over) ≤ δ + 0.05 (δ = 0.10) and certified area ≥ WSR-PAC area | Failure ⇒ recommend the exact WSR bound in the library instead. |

In [ ]:
# ===== Cell 16: theory setup — choose the run (full if finished, else smoke) and load the real calibration histograms =====
TH_CFG = CFG_FULL if hf_pull(f"{CFG_FULL['tag']}/results.json") else CFG_SMOKE
TH_TAG = TH_CFG["tag"]
TH_ALPHA = TH_CFG["alpha_main"]
TH_TI = TH_CFG["taus"].index(TH_CFG["tau_main"])
TH_DELTA = 0.10
TH_DRAWS = 400 if TH_TAG == "full" else 150
TH_SPLIT = load_json(f"{TH_TAG}/split.json") if hf_pull(f"{TH_TAG}/split.json") else make_scene_split(TH_CFG)
TH_CAL = load_domain(TH_CFG, "cal", [f"{a}_{b}" for a, b in TH_SPLIT["cal"]])
TH_EDGES = load_json(f"{TH_TAG}/models/score_edges.json")
TH_S2O = TH_CFG["s2o_gens"]


def th_world(g, s, ti=TH_TI):
    """Real per-scene curves for (generator, score): WL, WK of shape (M scenes, n0 patches, bins), the same
    n0 patches per scene (fixed random subset), and the population curves R(j), K(j) (scene-weighted)."""
    Kt, L = curves(TH_CAL, g, s, "full", ti)
    if L is None:
        return None
    units = np.unique(TH_CAL["unit"])
    n0 = min(int((TH_CAL["unit"] == u).sum()) for u in units)
    rng = np.random.default_rng(0)
    WL, WK = [], []
    for u in units:
        rows = np.nonzero(TH_CAL["unit"] == u)[0]
        pick = rows[rng.permutation(len(rows))[:n0]]
        WL.append(L[pick])
        WK.append(Kt[pick])
    WL, WK = np.stack(WL), np.stack(WK)
    return {"WL": WL, "WK": WK, "R": WL.mean(1).mean(0), "K": WK.mean(1).mean(0), "M": WL.shape[0], "n0": n0}


def th_draw(W, m, n, rng):
    """m scenes drawn with replacement from the real scenes, n patches without replacement inside each."""
    si = rng.integers(0, W["M"], m)
    pi = np.argsort(rng.random((m, W["n0"])), 1)[:, :n]
    return W["WL"][si[:, None], pi], W["WK"][si[:, None], pi]


def th_alpha_prime(alpha, m, n):
    return alpha + (1 - alpha) * (1 - 1 / n) / (m + 1)


TH_PRIMARY = th_world("pix2pix", "learned")
print({"theory_run": TH_TAG, "scenes": TH_PRIMARY["M"] if TH_PRIMARY else 0, "patches_per_scene": TH_PRIMARY["n0"] if TH_PRIMARY else 0,
       "pix2pix_error_rate_tau_main": round(float(TH_PRIMARY["R"][-1]), 4) if TH_PRIMARY else None})

In [ ]:
# ===== Cell 17: T3 on real data — exact unit equivalence (H10) and marginal risk bounds (H11) =====
def th_t3(g, s, alpha, m_list, n_list, draws, seed):
    W = th_world(g, s)
    if W is None:
        return []
    rng = np.random.default_rng(seed)
    out = []
    for m in m_list:
        for n in n_list:
            if n > W["n0"]:
                continue
            a2 = th_alpha_prime(alpha, m, n)
            mism, rp, rs = 0, [], []
            for _ in range(draws):
                Ld, _ = th_draw(W, m, n, rng)
                jp = crc_pick(Ld.reshape(m * n, -1), alpha + 1e-12)      # +1e-12: floating-point ties only
                js_equiv = crc_pick(Ld.mean(1), a2 + 1e-12)
                js = crc_pick(Ld.mean(1), alpha)
                mism += int(jp != js_equiv)
                rp.append(W["R"][jp] if jp >= 0 else 0.0)
                rs.append(W["R"][js] if js >= 0 else 0.0)
            rp, rs = np.array(rp), np.array(rs)
            out.append({"gen": g, "score": s, "alpha": alpha, "m": m, "n": n, "alpha_prime": a2, "identity_mismatches": mism,
                        "patch_mean_risk": float(rp.mean()), "patch_se": float(rp.std(ddof=1) / np.sqrt(draws)),
                        "scene_mean_risk": float(rs.mean()), "scene_se": float(rs.std(ddof=1) / np.sqrt(draws)),
                        "patch_exceeds_alpha": bool(rp.mean() > alpha)})
    return out


m_list_cell17 = sorted({max(5, TH_PRIMARY["M"] // 4), TH_PRIMARY["M"] // 2, TH_PRIMARY["M"]}) if TH_PRIMARY else []
n_list_cell17 = sorted({1, 4, 16, TH_PRIMARY["n0"]}) if TH_PRIMARY else []
T3_ROWS = []
for g_cell17 in ["pix2pix", "flow_sample", "unet_l1"]:
    for s_cell17 in ["learned", "floor", "topo"]:
        for a_cell17 in TH_CFG["alphas"]:
            T3_ROWS += th_t3(g_cell17, s_cell17, a_cell17, m_list_cell17, n_list_cell17, max(60, TH_DRAWS // 4), seed=17)
H10_RESULT = {"configs": len(T3_ROWS), "total_mismatches": int(sum(r["identity_mismatches"] for r in T3_ROWS))}
H10_RESULT["verdict"] = verdict(H10_RESULT["total_mismatches"] == 0, insufficient=len(T3_ROWS) == 0)
h11_bad_cell17 = [r for r in T3_ROWS if r["patch_mean_risk"] > r["alpha_prime"] + 2 * r["patch_se"] + 1e-12
                  or r["scene_mean_risk"] > r["alpha"] + 2 * r["scene_se"] + 1e-12]
H11_RESULT = {"configs": len(T3_ROWS), "violations": len(h11_bad_cell17), "examples": h11_bad_cell17[:5],
              "configs_where_patch_level_exceeds_alpha": int(sum(r["patch_exceeds_alpha"] for r in T3_ROWS)),
              "max_patch_excess_over_alpha": float(max((r["patch_mean_risk"] - r["alpha"] for r in T3_ROWS), default=float("nan")))}
H11_RESULT["verdict"] = verdict(len(h11_bad_cell17) == 0, insufficient=len(T3_ROWS) == 0)
save_json(f"{TH_TAG}/theory/t3_unit_equivalence.json", {"rows": T3_ROWS, "H10": H10_RESULT, "H11": H11_RESULT})
print("H10:", H10_RESULT)
print("H11:", {k: v for k, v in H11_RESULT.items() if k != "examples"})

In [ ]:
# ===== Cell 18: T4 on real data — the training-conditional law (H12) =====
def th_t4(g, s, alpha, m_list, n_list, draws, seed):
    W = th_world(g, s)
    if W is None:
        return [], {}
    above = np.nonzero(W["R"] > alpha)[0]
    if above.size == 0:
        return [], {"note": "population risk never exceeds alpha: every threshold is safe"}
    jp = int(above.min())
    v = W["WL"][:, :, jp]                                        # (M, n0) real losses at j+
    mu_s, s2_s = v.mean(1), v.var(1, ddof=1) if W["n0"] > 1 else np.zeros(W["M"])
    var_b, mean_w = float(mu_s.var()), float(s2_s.mean())
    icc, n0_eff = icc_oneway(v.ravel(), np.repeat(np.arange(W["M"]), W["n0"]))
    info = {"j_plus": jp, "R_j_plus": float(W["R"][jp]), "sigma_b2": var_b, "sigma_w2": mean_w, "icc": icc}
    rng = np.random.default_rng(seed)
    rows = []
    for m in m_list:
        for n in n_list:
            if n > W["n0"]:
                continue
            tau2 = var_b + mean_w * (W["n0"] - n) / (W["n0"] * n) if W["n0"] > 1 else var_b
            for mode in ("patch", "scene"):
                t = alpha - (1 - alpha) / (m * n) if mode == "patch" else alpha - (1 - alpha) / m
                pred = float(sstats.norm.cdf((t - W["R"][jp]) * np.sqrt(m) / np.sqrt(tau2))) if tau2 > 0 else float(W["R"][jp] <= t)
                fails, ident_bad = 0, 0
                for _ in range(draws):
                    Ld, _ = th_draw(W, m, n, rng)
                    Lu = Ld.reshape(m * n, -1) if mode == "patch" else Ld.mean(1)
                    j = crc_pick(Lu, alpha)
                    over = j >= 0 and W["R"][j] > alpha
                    fails += int(over)
                    ident_bad += int((j >= jp) != (Ld.mean(1)[:, jp].mean() <= t + 1e-12))
                rows.append({"gen": g, "score": s, "m": m, "n": n, "mode": mode, "observed": fails / draws, "predicted": pred,
                             "event_identity_failures": ident_bad})
    return rows, info


m_list_cell18 = sorted({max(5, TH_PRIMARY["M"] // 4), TH_PRIMARY["M"] // 2, TH_PRIMARY["M"]}) if TH_PRIMARY else []
n_list_cell18 = sorted({n for n in (1, 2, 4, 8, 16, 32, 64) if TH_PRIMARY and n <= TH_PRIMARY["n0"]} | ({TH_PRIMARY["n0"]} if TH_PRIMARY else set()))
T4_ROWS, T4_INFO = [], {}
for g_cell18 in ["pix2pix", "flow_sample"]:
    rows_cell18, info_cell18 = th_t4(g_cell18, "learned", TH_ALPHA, m_list_cell18, n_list_cell18, TH_DRAWS, seed=18)
    T4_ROWS += rows_cell18
    T4_INFO[g_cell18] = info_cell18
if T4_ROWS:
    err_cell18 = max(abs(r["observed"] - r["predicted"]) for r in T4_ROWS)
    ident_cell18 = sum(r["event_identity_failures"] for r in T4_ROWS)
    rises_cell18, below_cell18 = True, True
    for g_cell18 in {r["gen"] for r in T4_ROWS}:
        for m_cell18 in m_list_cell18:
            pr_cell18 = sorted([r for r in T4_ROWS if r["gen"] == g_cell18 and r["m"] == m_cell18 and r["mode"] == "patch"], key=lambda r: r["n"])
            sc_cell18 = {r["n"]: r["observed"] for r in T4_ROWS if r["gen"] == g_cell18 and r["m"] == m_cell18 and r["mode"] == "scene"}
            if len(pr_cell18) >= 2 and pr_cell18[-1]["observed"] + 0.02 < pr_cell18[0]["observed"]:
                rises_cell18 = False
            if any(sc_cell18[r["n"]] > r["observed"] + 0.02 for r in pr_cell18 if r["n"] >= 2):
                below_cell18 = False
    H12_RESULT = {"max_abs_error": err_cell18, "event_identity_failures": ident_cell18, "patch_rises_with_n": rises_cell18,
                  "scene_not_above_patch": below_cell18, "info": T4_INFO}
    informative_cell18 = max(r["observed"] for r in T4_ROWS if r["mode"] == "patch") >= 0.02   # some failures exist to be predicted
    H12_RESULT["informative"] = informative_cell18
    H12_RESULT["verdict"] = verdict(err_cell18 <= 0.05 and ident_cell18 == 0 and rises_cell18 and below_cell18,
                                    insufficient=not informative_cell18)
else:
    H12_RESULT = {"verdict": "INSUFFICIENT", "info": T4_INFO}
save_json(f"{TH_TAG}/theory/t4_law.json", {"rows": T4_ROWS, "H12": H12_RESULT})
print("H12:", {k: v for k, v in H12_RESULT.items() if k != "info"})
for r_cell18 in [r for r in T4_ROWS if r["gen"] == "pix2pix" and r["m"] == (m_list_cell18[1] if len(m_list_cell18) > 1 else -1)]:
    print(f"  pix2pix m={r_cell18['m']:3d} n={r_cell18['n']:3d} {r_cell18['mode']:5s} observed {r_cell18['observed']:.3f}  predicted {r_cell18['predicted']:.3f}")

In [ ]:
# ===== Cell 19: T5 on real data — studentised scene PAC vs CRC vs exact WSR (H13) =====
def th_wsr_ucb_fast(z, delta):
    """Same WSR bound as cell 9, found by bisection (rejection of 'mean >= m' is monotone in m)."""
    z = np.asarray(z, float)
    n = len(z)
    mu_hat = (0.5 + np.cumsum(z)) / (np.arange(1, n + 1) + 1)
    sig2 = (0.25 + np.cumsum((z - mu_hat) ** 2)) / (np.arange(1, n + 1) + 1)
    lam = np.sqrt(2 * np.log(1 / delta) / (n * np.r_[0.25, sig2[:-1]]))

    def _rejected(mv):
        return np.cumprod(1 + np.minimum(lam, 0.75 / max(1 - mv, 1e-6)) * (mv - z)).max() >= 1 / delta

    lo, hi = 0.0, 1.0
    for _ in range(30):
        mid = (lo + hi) / 2
        if _rejected(mid):
            hi = mid
        else:
            lo = mid
    return hi


def th_pick_spac(S, alpha, delta):
    m = S.shape[0]
    ub = S.mean(0) + sstats.t.ppf(1 - delta, m - 1) * S.std(0, ddof=1) / np.sqrt(m)
    ok = np.nonzero(ub <= alpha)[0]
    return int(ok.max()) if ok.size else -1


def th_pick_wsr(S, alpha, delta):
    j = -1
    for jj in range(S.shape[1]):
        if th_wsr_ucb_fast(S[:, jj], delta) <= alpha:
            j = jj
        else:
            break
    return j


def th_t5(g, s, alpha, delta, m_list, draws, seed):
    W = th_world(g, s)
    if W is None:
        return []
    rng = np.random.default_rng(seed)
    rows = []
    for m in m_list:
        acc = {k: {"over": [], "area": []} for k in ("crc_scene", "spac", "wsr_pac")}
        for _ in range(draws):
            Ld, _ = th_draw(W, m, W["n0"], rng)
            S = Ld.mean(1)
            for k, j in (("crc_scene", crc_pick(S, alpha)), ("spac", th_pick_spac(S, alpha, delta)), ("wsr_pac", th_pick_wsr(S, alpha, delta))):
                acc[k]["over"].append(j >= 0 and W["R"][j] > alpha)
                acc[k]["area"].append(W["K"][j] if j >= 0 else 0.0)
        oracle_j = np.nonzero(W["R"] <= alpha)[0]
        rows.append({"gen": g, "score": s, "m": m, "oracle_area": float(W["K"][oracle_j.max()]) if oracle_j.size else 0.0,
                     **{f"{k}_P_over": float(np.mean(v["over"])) for k, v in acc.items()},
                     **{f"{k}_area": float(np.mean(v["area"])) for k, v in acc.items()}})
    return rows


T5_ROWS = []
for g_cell19 in ["pix2pix", "flow_sample", "floor_center"]:
    T5_ROWS += th_t5(g_cell19, "learned", TH_ALPHA, TH_DELTA, [m for m in m_list_cell18 if m >= 10], max(60, TH_DRAWS // 4), seed=19)
if T5_ROWS:
    H13_RESULT = {"max_P_over_spac": max(r["spac_P_over"] for r in T5_ROWS),
                  "spac_area_minus_wsr_area_min": min(r["spac_area"] - r["wsr_pac_area"] for r in T5_ROWS)}
    H13_RESULT["verdict"] = verdict(H13_RESULT["max_P_over_spac"] <= TH_DELTA + 0.05 and H13_RESULT["spac_area_minus_wsr_area_min"] >= -1e-9)
else:
    H13_RESULT = {"verdict": "INSUFFICIENT"}
save_json(f"{TH_TAG}/theory/t5_spac.json", {"rows": T5_ROWS, "H13": H13_RESULT})
print("H13:", H13_RESULT)
for r_cell19 in T5_ROWS:
    print(f"  {r_cell19['gen']:13s} m={r_cell19['m']:3d} | P(over): CRC {r_cell19['crc_scene_P_over']:.3f} SPAC {r_cell19['spac_P_over']:.3f} "
          f"WSR {r_cell19['wsr_pac_P_over']:.3f} | area: CRC {r_cell19['crc_scene_area']:.3f} SPAC {r_cell19['spac_area']:.3f} "
          f"WSR {r_cell19['wsr_pac_area']:.3f} oracle {r_cell19['oracle_area']:.3f}")

In [ ]:
# ===== Cell 20: T2 on real data — the certification ceiling (H8) and who attains it (H9) =====
def th_ceiling(alpha):
    """CTA*(alpha) from the real floor map: pixels ordered by floor; each bin costs (lower bin edge) x (pixels).
    Using the lower edge and allowing a partial last bin makes the ceiling conservative (as large as possible)."""
    cnt = TH_CAL["cnt|floor_center|floor"].sum(0).astype(float)
    frac = cnt / cnt.sum()
    low = np.r_[0.0, np.array(TH_EDGES["floor_center"]["floor"], float)]
    cost = np.cumsum(frac * low)
    area = np.cumsum(frac)
    j = int(np.searchsorted(cost, alpha, side="right")) - 1
    a = area[j] if j >= 0 else 0.0
    if j + 1 < len(frac):
        spare = alpha - (cost[j] if j >= 0 else 0.0)
        a += frac[j + 1] if low[j + 1] <= 0 else min(frac[j + 1], spare / low[j + 1])
    return float(a)


def th_oracle_area(g, s, alpha):
    Kt, L = curves(TH_CAL, g, s, "full", TH_TI)
    if L is None:
        return None
    ok = np.nonzero(L.mean(0) <= alpha)[0]
    return float(Kt.mean(0)[ok.max()]) if ok.size else 0.0


T2_RESULT = {}
for a_cell20 in TH_CFG["alphas"]:
    ceil_cell20 = th_ceiling(a_cell20)
    areas_cell20 = {f"{g}|{s}": th_oracle_area(g, s, a_cell20) for g in TH_S2O for s in TH_CFG["scores"]}
    areas_cell20 = {k: v for k, v in areas_cell20.items() if v is not None}
    T2_RESULT[str(a_cell20)] = {"ceiling": ceil_cell20, "oracle_areas": areas_cell20,
                                "efficiency": {k: (v / ceil_cell20 if ceil_cell20 > 0 else float("nan")) for k, v in areas_cell20.items()},
                                "max_excess_over_ceiling": max(v - ceil_cell20 for v in areas_cell20.values())}
H8_RESULT = {"max_excess": {a: T2_RESULT[a]["max_excess_over_ceiling"] for a in T2_RESULT},
             "ceilings": {a: T2_RESULT[a]["ceiling"] for a in T2_RESULT}}
H8_RESULT["verdict"] = verdict(all(v <= 0.02 for v in H8_RESULT["max_excess"].values()))
oa_cell20 = T2_RESULT[str(TH_ALPHA)]["oracle_areas"]
tm_cell20 = oa_cell20.get("floor_center|floor", float("nan"))
smp_cell20 = {k: oa_cell20.get(k, float("nan")) for k in ("flow_sample|floor", "pix2pix_mcd|floor")}
H9_RESULT = {"tau_mode_area": tm_cell20, "sampler_areas": smp_cell20,
             "median_or_mean_areas": {k: oa_cell20.get(k) for k in ("unet_l1|floor", "flow_mean|floor")}}
H9_RESULT["verdict"] = verdict(all(tm_cell20 >= v - 0.01 for v in smp_cell20.values()), insufficient=not np.isfinite(tm_cell20))
save_json(f"{TH_TAG}/theory/t2_ceiling.json", {"T2": T2_RESULT, "H8": H8_RESULT, "H9": H9_RESULT})
print("H8:", H8_RESULT)
print("H9:", H9_RESULT)
print(f"top-5 efficiency at alpha={TH_ALPHA}:",
      sorted(T2_RESULT[str(TH_ALPHA)]["efficiency"].items(), key=lambda kv: -kv[1])[:5])

In [ ]:
# ===== Cell 21: label-budget law — how many labelled scenes buy how much certified area (from T4/T5) =====
def th_budget(g, s, alpha, delta):
    W = th_world(g, s)
    if W is None:
        return None
    S = W["WL"].mean(1)                                   # scene losses with all n0 patches
    var_tot = S.var(0, ddof=1)
    within = W["WL"].var(1, ddof=1).mean(0) / W["n0"] if W["n0"] > 1 else np.zeros_like(var_tot)
    var_b = np.clip(var_tot - within, 0, None)            # between-scene part: what infinite patches would leave
    z = sstats.norm.ppf(1 - delta)
    gap = alpha - W["R"]
    ok = gap > 0
    need = np.where(ok, (z * np.sqrt(var_tot) / np.where(ok, gap, 1)) ** 2, np.inf)
    need_inf = np.where(ok, (z * np.sqrt(var_b) / np.where(ok, gap, 1)) ** 2, np.inf)
    rows = []
    for target in (0.05, 0.10, 0.20, 0.30, 0.40, 0.50):
        cand = np.nonzero(ok & (W["K"] >= target))[0]
        if cand.size:
            rows.append({"target_area": target, "scenes_needed": float(np.min(need[cand])),
                         "scenes_needed_with_infinite_patches": float(np.min(need_inf[cand]))})
        else:
            rows.append({"target_area": target, "scenes_needed": float("inf"), "scenes_needed_with_infinite_patches": float("inf"),
                         "note": "area not reachable at this alpha (above the oracle area)"})
    return rows


BUDGET = {g: th_budget(g, "learned", TH_ALPHA, TH_DELTA) for g in ["pix2pix", "flow_sample", "floor_center", "unet_l1"]}
save_json(f"{TH_TAG}/theory/label_budget.json", BUDGET)
for g_cell21, rows_cell21 in BUDGET.items():
    if rows_cell21:
        print(g_cell21, [(r["target_area"], round(r["scenes_needed"], 1), round(r["scenes_needed_with_infinite_patches"], 1)) for r in rows_cell21])

In [ ]:
# ===== Cell 22: figure — reliability vs patches (T4), ceiling vs oracle areas (T2), label budget =====
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

C_BLUE_CELL22, C_ORANGE_CELL22, INK_CELL22, MUTED_CELL22 = "#2a78d6", "#eb6834", "#2b2b2a", "#8a8984"
fig_cell22, ax_cell22 = plt.subplots(1, 3, figsize=(15, 4.4))
for axis_cell22 in ax_cell22:
    axis_cell22.spines[["top", "right"]].set_visible(False)
    axis_cell22.grid(alpha=0.25, linewidth=0.6)
m_mid_cell22 = m_list_cell18[1] if len(m_list_cell18) > 1 else (m_list_cell18[0] if m_list_cell18 else None)
for mode_cell22, col_cell22 in (("patch", C_ORANGE_CELL22), ("scene", C_BLUE_CELL22)):
    rr_cell22 = sorted([r for r in T4_ROWS if r["gen"] == "pix2pix" and r["m"] == m_mid_cell22 and r["mode"] == mode_cell22], key=lambda r: r["n"])
    if rr_cell22:
        xs_cell22 = [r["n"] for r in rr_cell22]
        ax_cell22[0].plot(xs_cell22, [r["observed"] for r in rr_cell22], "-o", color=col_cell22, lw=2, ms=8, label=f"{mode_cell22}-level, observed")
        ax_cell22[0].plot(xs_cell22, [r["predicted"] for r in rr_cell22], "--", color=col_cell22, lw=2, label=f"{mode_cell22}-level, T4 prediction")
ax_cell22[0].set_xscale("log", base=2)
ax_cell22[0].set_xlabel("patches per calibration scene (n)", color=INK_CELL22)
ax_cell22[0].set_ylabel("P(user over budget)", color=INK_CELL22)
ax_cell22[0].set_title(f"P(user over budget) vs patches per scene (T4)\n(pix2pix, m={m_mid_cell22} scenes, alpha={TH_ALPHA})", fontsize=10)
ax_cell22[0].set_ylim(-0.02, 0.6)
ax_cell22[0].legend(frameon=False, fontsize=8)
t2a_cell22 = T2_RESULT.get(str(TH_ALPHA), {})
if t2a_cell22:
    best_cell22 = {}
    for key_cell22, val_cell22 in t2a_cell22["oracle_areas"].items():
        gen_cell22 = key_cell22.split("|")[0]
        best_cell22[gen_cell22] = max(best_cell22.get(gen_cell22, 0.0), val_cell22)
    names_cell22 = list(best_cell22)
    ax_cell22[1].bar(range(len(names_cell22)), [best_cell22[n] for n in names_cell22], color=C_BLUE_CELL22, width=0.6)
    ax_cell22[1].axhline(t2a_cell22["ceiling"], color=INK_CELL22, lw=1.5, ls="--")
    ax_cell22[1].text(len(names_cell22) - 0.5, t2a_cell22["ceiling"], f" ceiling CTA* = {t2a_cell22['ceiling']:.3f}", va="bottom", ha="right", fontsize=8, color=INK_CELL22)
    ax_cell22[1].set_xticks(range(len(names_cell22)))
    ax_cell22[1].set_xticklabels(names_cell22, rotation=35, ha="right", fontsize=8)
    ax_cell22[1].set_ylabel("best oracle certified area", color=INK_CELL22)
    ax_cell22[1].set_title(f"Best oracle certified area vs identifiability ceiling (T2)\nalpha={TH_ALPHA}", fontsize=10)
bp_cell22 = BUDGET.get("pix2pix") or []
fin_cell22 = [r for r in bp_cell22 if np.isfinite(r["scenes_needed"])]
if fin_cell22:
    ax_cell22[2].plot([r["target_area"] for r in fin_cell22], [r["scenes_needed"] for r in fin_cell22], "-o", color=C_BLUE_CELL22, lw=2, ms=8, label="with the patches we have")
    ax_cell22[2].plot([r["target_area"] for r in fin_cell22], [r["scenes_needed_with_infinite_patches"] for r in fin_cell22], "--o", color=C_ORANGE_CELL22, lw=2, ms=8, label="with infinitely many patches")
    ax_cell22[2].set_yscale("log")
    ax_cell22[2].set_ylim(bottom=1)
    ax_cell22[2].legend(frameon=False, fontsize=8)
ax_cell22[2].set_xlabel("certified area wanted", color=INK_CELL22)
ax_cell22[2].set_ylabel(f"labelled scenes needed (P(over) <= {TH_DELTA})", color=INK_CELL22)
ax_cell22[2].set_title("Label budget (pix2pix, learned score)", fontsize=10)
fig_cell22.tight_layout()
atomic_write(f"{TH_TAG}/theory/theory_figure.png", lambda p: fig_cell22.savefig(p, dpi=160))
hf_push([f"{TH_TAG}/theory/theory_figure.png"])
plt.close(fig_cell22)
print("figure saved:", bucket_key(f"{TH_TAG}/theory/theory_figure.png"))

In [ ]:
# ===== Cell 23: theory hypothesis report (H8–H13), saved to the bucket =====
TH_HYP_TEXT = {
    "H8": "No SAR-only (generator, score) exceeds the certification ceiling CTA*(alpha) + 0.02",
    "H9": "tau-mode generator + floor score certifies >= the posterior samplers with the same score",
    "H10": "Patch-level CRC at alpha == scene-level CRC at alpha' in every real resample (exact identity)",
    "H11": "Patch-level mean risk <= alpha'; scene-level mean risk <= alpha (real scene superpopulation)",
    "H12": "T4 predicts P(user over budget) within 0.05; patch-level rises with n; scene-level stays below",
    "H13": "Studentised scene PAC keeps P(over) <= delta + 0.05 and certifies >= the WSR-PAC area",
}
TH_RESULTS = {"H8": H8_RESULT, "H9": H9_RESULT, "H10": H10_RESULT, "H11": H11_RESULT, "H12": H12_RESULT, "H13": H13_RESULT}
lines_cell23 = [f"# CERTEO theory report — run '{TH_TAG}'", "",
                f"Calibration scenes: {TH_PRIMARY['M'] if TH_PRIMARY else 0} | patches per scene used: {TH_PRIMARY['n0'] if TH_PRIMARY else 0} | "
                f"alpha = {TH_ALPHA}, delta = {TH_DELTA}, tau = {TH_CFG['tau_main']} | draws per configuration: {TH_DRAWS}", ""]
if TH_TAG == "smoke":
    lines_cell23 += ["> **SMOKE RUN**: few scenes and short training. These verdicts check that the theory cells work; do not quote them.", ""]
lines_cell23 += ["| Hypothesis | Prediction | Verdict |", "|---|---|---|"]
for h_cell23, txt_cell23 in TH_HYP_TEXT.items():
    lines_cell23.append(f"| {h_cell23} | {txt_cell23} | **{TH_RESULTS[h_cell23].get('verdict', 'INSUFFICIENT')}** |")
lines_cell23 += ["", "## Ceiling and efficiency (T2)", "", "| (generator, score) | oracle area @alpha | share of ceiling |", "|---|---|---|"]
for k_cell23, v_cell23 in sorted(T2_RESULT.get(str(TH_ALPHA), {}).get("oracle_areas", {}).items(), key=lambda kv: -kv[1]):
    lines_cell23.append(f"| {k_cell23} | {v_cell23:.3f} | {T2_RESULT[str(TH_ALPHA)]['efficiency'][k_cell23]:.2f} |")
lines_cell23 += ["", f"Ceiling CTA*: {H8_RESULT.get('ceilings')}", "", "## Training-conditional law (T4), pix2pix", "",
                 "| m | n | mode | observed P(over) | predicted |", "|---|---|---|---|---|"]
for r_cell23 in [r for r in T4_ROWS if r["gen"] == "pix2pix"]:
    lines_cell23.append(f"| {r_cell23['m']} | {r_cell23['n']} | {r_cell23['mode']} | {r_cell23['observed']:.3f} | {r_cell23['predicted']:.3f} |")
lines_cell23 += ["", "## Scene PAC (T5)", "", "```json", json.dumps(T5_ROWS, indent=1, default=float)[:5000], "```",
                 "", "## Label budget", "", "```json", json.dumps(BUDGET, indent=1, default=float)[:5000], "```", "", "## Details", ""]
for h_cell23, res_cell23 in TH_RESULTS.items():
    lines_cell23 += [f"### {h_cell23} — {res_cell23.get('verdict')}", "```json",
                     json.dumps({k: v for k, v in res_cell23.items() if k != "verdict"}, indent=1, default=float)[:4000], "```", ""]
THEORY_REPORT = "\n".join(lines_cell23)


def th_write_report(p):
    with open(p, "w") as fh:
        fh.write(THEORY_REPORT)


atomic_write(f"{TH_TAG}/THEORY_REPORT.md", th_write_report)
save_json(f"{TH_TAG}/theory/theory_results.json", TH_RESULTS)
hf_push([f"{TH_TAG}/THEORY_REPORT.md", f"{TH_TAG}/log.txt"])
print(THEORY_REPORT[:3000])

In [ ]:
# ===== Cell 24: one-shot results dump — every number needed for the paper, in one printout and one file =====
# Prints and saves <run>/ALL_RESULTS_DUMP.md: both reports verbatim, every result file, extra diagnostics for the
# failed theory hypotheses (H9, H12, H13), patch-vs-scene over a grid of (alpha, tau), and the SECONDARY operating
# point alpha = 0.10, tau = 0.20. That point was fixed after the smoke run and before any full-run result
# (research notebook #39, lab report §25); it is a labelled secondary analysis, not the pre-registered one.
# CPU only, reads the bucket; about 5-10 minutes on the full run.
def dump_all_cell24(cfg, R=300, alpha2=0.10, tau2=0.20):
    tag = cfg["tag"]
    out = []
    P = out.append
    t_start = time.time()

    def js(obj, lim=None):
        s = json.dumps(obj, indent=1, default=float)
        return s if lim is None or len(s) <= lim else s[:lim] + "\n... (cut)"

    def f3(x):
        try:
            return f"{float(x):.3f}"
        except (TypeError, ValueError):
            return str(x)

    def read_text(rel):
        if not hf_pull(rel):
            return "(missing)"
        with open(local_path(rel)) as fh:
            return fh.read()

    P(f"# CERTEO full results dump — run '{tag}'")
    P(f"alpha_main {cfg['alpha_main']}, tau_main {cfg['tau_main']}, taus {cfg['taus']}, alphas {cfg['alphas']}, "
      f"secondary point alpha {alpha2}, tau {tau2}, resamples {R}")

    # ---- 1. both reports verbatim ----
    for rel in (f"{tag}/HYPOTHESIS_REPORT.md", f"{tag}/THEORY_REPORT.md"):
        P(f"\n\n######## FILE {rel} ########\n")
        P(read_text(rel))

    # ---- 2. results.json, section by section ----
    res = load_json(f"{tag}/results.json") if hf_pull(f"{tag}/results.json") else {}
    P("\n\n######## results.json ########")
    P(f"sizes: cal scenes {res.get('n_cal_scenes')}, cal patches {res.get('n_cal_patches')}, winter scenes "
      f"{res.get('n_winter_scenes')}, DFC patches {res.get('n_dfc_patches')} | timings (min) {res.get('timings_min')} | "
      f"wall {res.get('wall_min_this_session')} min")
    table = res.get("table", {})
    for a in cfg["alphas"]:
        P(f"\n## Quality and scene-level CTA, alpha={a}, tau_main (cta / refused share for learned)")
        P("| gen | PSNR | SSIM | floor | topo | spread | learned | learned low-pass | refused (learned) | risk (learned) |")
        P("|---|---|---|---|---|---|---|---|---|---|")
        for g, r in table.items():
            P(f"| {g} | {f3(r.get('psnr'))} | {f3(r.get('ssim'))} | " + " | ".join(f3(r.get(f"cta|{s}|full|{a}")) for s in cfg["scores"])
              + f" | {f3(r.get(f'cta|learned|low|{a}'))} | {f3(r.get(f'refused|learned|full|{a}'))} | {f3(r.get(f'risk|learned|full|{a}'))} |")
    h1 = res.get("H1", {})
    P(f"\n## H1 floor — verdict {h1.get('verdict')}, worst violation share {f3(h1.get('worst_violation_frac'))}, "
      f"floor_center calibration gap (error − floor) {f3(h1.get('floor_center_calibration_gap'))}")
    P("| gen | tau | bins used | violating | mean floor | error rate | excess | worst gap |")
    P("|---|---|---|---|---|---|---|---|")
    for g, per in h1.get("per_generator", {}).items():
        for tau, v in per.items():
            P(f"| {g} | {tau} | {v['bins_used']} | {v['violating_bins']} | {f3(v['mean_floor'])} | {f3(v['error_rate'])} | "
              f"{f3(v['excess'])} | {f3(v['worst_gap'])} |")
    for h in ("H2", "H2b", "H3", "H4", "H5", "H6", "H7"):
        P(f"\n## {h} — verdict {res.get(h, {}).get('verdict')}")
        P("```json\n" + js({k: v for k, v in res.get(h, {}).items() if k != "verdict"}, 12000) + "\n```")
    P("\n## Sensitivity (scene-level CTA, learned score): gen | tau | alpha | cta | risk | error rate")
    for k, v in res.get("sensitivity", {}).items():
        P(f"{k} | cta {f3(v['cta'])} | risk {f3(v['risk'])} | err {f3(v['error_rate'])}")
    P("\n## Guarantee tiers\n```json\n" + js(res.get("tiers", {})) + "\n```")

    # ---- 3. theory result files ----
    P("\n\n######## theory files ########")
    for nm in ("t3_unit_equivalence", "t4_law", "t5_spac", "t2_ceiling", "label_budget"):
        rel = f"{tag}/theory/{nm}.json"
        obj = load_json(rel) if hf_pull(rel) else None
        if obj is None:
            P(f"\n## {nm}: missing")
            continue
        if nm == "t3_unit_equivalence":
            rows = obj.get("rows", [])
            P(f"\n## T3: {len(rows)} configurations | H10 {js(obj.get('H10'), 1500)} | H11 {js(obj.get('H11'), 2500)}")
            for r in rows[:60]:
                P(js(r).replace("\n", " "))
        elif nm == "t4_law":
            P(f"\n## T4 / H12 flags: {js({k: v for k, v in obj.get('H12', {}).items() if k != 'info'})}")
            P(f"T4 info (j+, R(j+), sigma_b2, sigma_w2, icc): {js(obj.get('H12', {}).get('info'))}")
            P("| gen | m | n | mode | observed | predicted | abs error | identity failures |")
            P("|---|---|---|---|---|---|---|---|")
            for r in obj.get("rows", []):
                P(f"| {r['gen']} | {r['m']} | {r['n']} | {r['mode']} | {f3(r['observed'])} | {f3(r['predicted'])} | "
                  f"{f3(abs(r['observed'] - r['predicted']))} | {r['event_identity_failures']} |")
        elif nm == "t5_spac":
            P(f"\n## T5 / H13: {js(obj.get('H13'))}")
            for r in obj.get("rows", []):
                P(js(r).replace("\n", " "))
        elif nm == "t2_ceiling":
            P(f"\n## T2 / H8 {js(obj.get('H8'))}\nH9 {js(obj.get('H9'))}")
            for a, v in obj.get("T2", {}).items():
                P(f"alpha {a}: ceiling {f3(v['ceiling'])}; oracle areas " +
                  ", ".join(f"{k} {f3(x)}" for k, x in sorted(v["oracle_areas"].items(), key=lambda kv: -kv[1])))
        else:
            P(f"\n## label budget\n```json\n{js(obj)}\n```")

    # ---- 4. new computations on the stored histograms ----
    split = load_json(f"{tag}/split.json")
    cal = load_domain(cfg, "cal", [f"{a}_{b}" for a, b in split["cal"]])
    win = load_domain(cfg, "winter", [f"{a}_{b}" for a, b in split["winter"]])
    dfc_names = load_json(f"{tag}/hist/dfc_names.json") if hf_pull(f"{tag}/hist/dfc_names.json") else []
    dfc = load_domain(cfg, "dfc", dfc_names)
    edges = load_json(f"{tag}/models/score_edges.json")
    s2o, gens = cfg["s2o_gens"], all_gens(cfg)
    ti_main = cfg["taus"].index(cfg["tau_main"])
    ti2 = cfg["taus"].index(tau2)
    unit = cal["unit"]
    P(f"\n\n######## new diagnostics (computed now) ########\ncal patches {len(unit)}, scenes {len(np.unique(unit))}, "
      f"winter patches {len(win['unit'])}, DFC patches {len(dfc['unit'])}")

    # 4a. Is the pixel floor calibrated? empirical miss rate per floor decile vs the floor value (tau_main)
    P("\n## 4a. Floor calibration (pixel floor score, tau_main): decile | mean floor | miss rate of each generator")
    gl = [g for g in ("floor_center", "pix2pix", "pix2pix_mcd", "unet_l1", "flow_sample", "flow_mean") if f"cnt|{g}|floor" in cal]
    P("| decile | mean floor | " + " | ".join(gl) + " |")
    P("|---|---|" + "---|" * len(gl))
    e_all = np.r_[0.0, np.array(edges[gl[0]]["floor"], float), 1.0] if gl else np.zeros(2)
    mids = (e_all[:-1] + e_all[1:]) / 2
    nb = cfg["n_bins"]
    groups = np.array_split(np.arange(nb), 10)
    for gi, idx in enumerate(groups):
        c0 = cal[f"cnt|{gl[0]}|floor"].sum(0).astype(float)
        mf = float((c0[idx] * mids[idx]).sum() / max(c0[idx].sum(), 1))
        cells = []
        for g in gl:
            c = cal[f"cnt|{g}|floor"].sum(0).astype(float)
            b = cal[f"bad|{g}|floor|full|{ti_main}"].sum(0).astype(float)
            cells.append(f3(b[idx].sum() / max(c[idx].sum(), 1)))
        P(f"| {gi + 1} | {f3(mf)} | " + " | ".join(cells) + " |")
    if "h1|floor_center" in cal:
        H = cal["h1|floor_center"][ti_main]
        P("\nPer-band floor bins for floor_center (tau_main): bin | band-pixels | mean floor | empirical miss")
        for k in range(H.shape[0]):
            if H[k, 0] > 0:
                P(f"{k} | {int(H[k, 0])} | {f3(H[k, 2] / H[k, 0])} | {f3(H[k, 1] / H[k, 0])}")

    # 4b. Patch vs scene over a grid (headline table of the paper)
    P("\n## 4b. Patch-level vs scene-level CRC on real scenes (32 calibration / 32 test scenes per split)")
    P("| gen | score | alpha | tau | P_over patch | P_over scene | risk patch | risk scene | area patch | area scene | refused scene | ICC |")
    P("|---|---|---|---|---|---|---|---|---|---|---|---|")
    for g in ("pix2pix", "flow_sample", "unet_l1", "pix2pix_topoguard"):
        for s in ("learned", "floor"):
            for a in (0.05, 0.10):
                for tau in (cfg["tau_main"], tau2):
                    Kt, L = curves(cal, g, s, "full", cfg["taus"].index(tau))
                    if L is None:
                        continue
                    sr = split_resample(L, Kt, unit, a, R, seed=24)
                    jj = crc_pick(unit_mean(L, unit), a)
                    icc, _ = icc_oneway(L[:, jj] if jj >= 0 else L[:, -1], unit)
                    P(f"| {g} | {s} | {a} | {tau} | {f3((sr['patch']['risk'] > a).mean())} | {f3((sr['scene']['risk'] > a).mean())} | "
                      f"{f3(sr['patch']['risk'].mean())} | {f3(sr['scene']['risk'].mean())} | {f3(sr['patch']['trust'].mean())} | "
                      f"{f3(sr['scene']['trust'].mean())} | {f3(sr['scene']['refused'].mean())} | {f3(icc)} |")

    # 4c. Secondary operating point: the H3/H4/H5/H6/H7 tests repeated at (alpha2, tau2)
    P(f"\n## 4c. SECONDARY operating point alpha={alpha2}, tau={tau2} (fixed after the smoke run, before full results)")
    sec = {}
    P("| gen | PSNR | floor | topo | spread | learned | learned low-pass | risk (learned) | refused (learned) |")
    P("|---|---|---|---|---|---|---|---|---|")
    for g in gens:
        row = {"psnr": float(cal[f"psnr|{g}"].mean())}
        for s in cfg["scores"]:
            for e in ("full", "low"):
                Kt, L = curves(cal, g, s, e, ti2)
                if L is None:
                    continue
                sr = split_resample(L, Kt, unit, alpha2, min(R, 200), seed=25, modes=("scene",))["scene"]
                row[f"cta|{s}|{e}"], row[f"risk|{s}|{e}"], row[f"ref|{s}|{e}"] = float(sr["trust"].mean()), float(sr["risk"].mean()), float(sr["refused"].mean())
                if s == "spread" or s == "topo":
                    row[f"trust_draws|{s}|{e}"] = sr["trust"]
        sec[g] = row
        P(f"| {g} | {f3(row['psnr'])} | " + " | ".join(f3(row.get(f"cta|{s}|full")) for s in cfg["scores"]) +
          f" | {f3(row.get('cta|learned|low'))} | {f3(row.get('risk|learned|full'))} | {f3(row.get('ref|learned|full'))} |")
    ps = [sec[g]["psnr"] for g in s2o]
    for lab, key in (("learned score", lambda g: sec[g].get("cta|learned|full", np.nan)),
                     ("best score per generator", lambda g: max(sec[g].get(f"cta|{s}|full", 0.0) for s in cfg["scores"]))):
        ct = [key(g) for g in s2o]
        kt = float(sstats.kendalltau(ps, ct)[0]) if np.all(np.isfinite(ct)) and np.std(ct) > 0 else float("nan")
        P(f"H3 (secondary, {lab}): Kendall tau(PSNR, CTA) = {f3(kt)} over {len(s2o)} generators -> "
          f"{verdict(np.isfinite(kt) and kt < 0.6, insufficient=not np.isfinite(kt))}")
    for g in ("pix2pix_mcd", "pix2pix_topoguard", "flow_sample"):
        t1, t2_ = sec[g].get("trust_draws|topo|full"), sec[g].get("trust_draws|spread|full")
        if t1 is not None and t2_ is not None:
            d = t1 - t2_
            P(f"H4 (secondary) {g}: CTA topo {f3(t1.mean())} vs spread {f3(t2_.mean())}, P(topo better) {f3((d > 0).mean())} -> "
              f"{verdict(d.mean() > 0 and (d > 0).mean() >= 0.9, insufficient=(t1.max() == 0 and t2_.max() == 0))}"
              + ("  [primary H4 generator]" if g == "pix2pix_mcd" else ""))
    rat = {g: (sec[g].get("cta|learned|low", np.nan) / sec[g]["cta|learned|full"]) if sec[g].get("cta|learned|full", 0) > 0 else float("nan") for g in s2o}
    good = [r for r in rat.values() if np.isfinite(r)]
    P(f"H6 (secondary) low/full ratios: {js(rat).replace(chr(10), ' ')} -> {verdict(sum(r >= 2 for r in good) > len(good) / 2, insufficient=not good)}")
    if "segreal" in dfc:
        P(f"H7 (secondary) segmenter accuracy on real optical: {f3(dfc['segreal'][1] / max(dfc['segreal'][0], 1))}")
        diffs = []
        for g in s2o:
            Kc, Lc = curves(cal, g, "learned", "full", ti2)
            sg = dfc.get(f"seg|{g}")
            if Lc is None or sg is None:
                continue
            j = crc_pick(unit_mean(Lc, unit), alpha2)
            vin, cin = (sg[:j + 1, 0].sum(), sg[:j + 1, 1].sum()) if j >= 0 else (0.0, 0.0)
            vout, cout = sg[j + 1:, 0].sum(), sg[j + 1:, 1].sum()
            ai, ao = (cin / vin if vin > 0 else np.nan), (cout / vout if vout > 0 else np.nan)
            if np.isfinite(ai) and np.isfinite(ao):
                diffs.append(ai - ao)
            P(f"  {g}: acc inside {f3(ai)}, outside {f3(ao)}, labelled pixels trusted {f3(vin / max(vin + vout, 1))}")
        P(f"H7 (secondary) -> {verdict(sum(d >= 0.10 for d in diffs) > len(diffs) / 2, insufficient=not diffs)}")
    for g in ("pix2pix", "flow_sample", "unet_l1"):
        Kc, Lc = curves(cal, g, "learned", "full", ti2)
        if Lc is None:
            continue
        j = crc_pick(unit_mean(Lc, unit), alpha2)
        line = f"H5 (secondary) {g}: threshold bin {j}, calibration in-sample risk {f3(Lc[:, j].mean() if j >= 0 else 0.0)}"
        for nm, dom in (("winter", win), ("dfc", dfc)):
            if not len(dom["unit"]):
                continue
            Kd, Ld = curves(dom, g, "learned", "full", ti2)
            pu = unit_mean(Ld, dom["unit"])[:, j] if j >= 0 else np.zeros(len(np.unique(dom["unit"])))
            m_, lo, hi = bootstrap_mean_ci(pu)
            line += f" | {nm}: risk {f3(m_)} [{f3(lo)}, {f3(hi)}], trusted {f3(Kd[:, j].mean() if j >= 0 else 0.0)}"
        kneed = math.ceil(1 / alpha2) - 1
        if len(win["unit"]) and len(np.unique(win["unit"])) > kneed:
            Kw, Lw = curves(win, g, "learned", "full", ti2)
            wu = np.unique(win["unit"])
            rng = np.random.default_rng(26)
            rr, tt = [], []
            for _ in range(R):
                p = rng.permutation(wu)
                cm = np.isin(win["unit"], p[:kneed])
                jw = crc_pick(unit_mean(Lw[cm], win["unit"][cm]), alpha2)
                rr.append(Lw[~cm][:, jw].mean() if jw >= 0 else 0.0)
                tt.append(Kw[~cm][:, jw].mean() if jw >= 0 else 0.0)
            line += f" | winter recal with {kneed} labelled scenes: risk {f3(np.mean(rr))}, P_over {f3(np.mean(np.array(rr) > alpha2))}, trusted {f3(np.mean(tt))}"
        P(line)

    # 4d. Error rates by tau (how hard the task is) for every generator
    P("\n## 4d. Pixel error rate P(max-band error > tau) per generator")
    P("| gen | " + " | ".join(f"tau {t}" for t in cfg["taus"]) + " |")
    P("|---|" + "---|" * len(cfg["taus"]))
    for g in gens:
        cells = []
        for ti in range(len(cfg["taus"])):
            _, L = curves(cal, g, "learned", "full", ti)
            cells.append(f3(L[:, -1].mean()) if L is not None else "-")
        P(f"| {g} | " + " | ".join(cells) + " |")

    # ---- 5. log lines of the evaluation stages ----
    P("\n\n######## log (stage lines) ########")
    for ln in read_text(f"{tag}/log.txt").splitlines():
        if any(w in ln for w in ("split:", "train patches", "aux generator", "eval cal: 64", "eval winter", "DFC", "dfc", "SAR2Opt",
                                 "sar2opt", "complete", "segmenter", "error", "Error", "deadline")) and "epoch" not in ln:
            P(ln)
    P(f"\n(dump built in {(time.time() - t_start) / 60:.1f} min)")
    text = "\n".join(out)

    def _w(p):
        with open(p, "w") as fh:
            fh.write(text)
    atomic_write(f"{tag}/ALL_RESULTS_DUMP.md", _w)
    hf_push([f"{tag}/ALL_RESULTS_DUMP.md"])
    return text


CFG_cell24 = CFG_FULL if hf_pull(f"{CFG_FULL['tag']}/results.json") else CFG_SMOKE
DUMP_TEXT_cell24 = dump_all_cell24(CFG_cell24)
print(DUMP_TEXT_cell24)
print("\nsaved:", bucket_key(f"{CFG_cell24['tag']}/ALL_RESULTS_DUMP.md"), "| characters:", len(DUMP_TEXT_cell24))

## Part III — Capped trust masks and the certification tax (cells 26–37)

**Why this part exists.** Part II showed on real data that (i) patch-level calibration fails about half of deployed users, (ii) scene-level CRC keeps only the *average* promise, and (iii) the exact per-user (PAC) bound certifies almost nothing with 64 scenes, while the quick t-based bound is unsafe. Part III explains (iii) with one law and removes most of the problem with one construct.

**Capped trust mask.** Given a threshold λ and a cap q ∈ (0, 1]: in every patch trust the pixels with score ≤ λ, but never more than a fraction q of the patch (lowest scores first, ties broken at random). Then the false-trust loss of every patch, hence of every scene, lies in **[0, q]** instead of [0, 1]. The loss is still monotone in λ. q = 1 is the uncapped mask of Parts I–II.

**T7 (certification tax — two-sided).** Let scene losses lie in [0, q] and let R be the true risk of a threshold.
*Necessity.* Any procedure that is distribution-free (α, δ)-PAC over i.i.d. scenes accepts that threshold with probability at most δ·((q − R)/(q − α))^m. *Proof:* mix the scene-loss law with a point mass at q, weight ε > (α − R)/(q − R); the mixture has risk > α, and with probability (1 − ε)^m the sample looks exactly like the original law; so P(accept) ≤ δ/(1 − ε)^m. Hence accepting with probability ≥ β requires **R ≤ α − (q − α)·((β/δ)^{1/m} − 1)**.
*Sufficiency.* The product certificate below accepts whenever all observed scene losses are ≤ α − (q − α)(δ^{−1/m} − 1)/c.
*CRC analogue.* Scene-level CRC with losses in [0, q] needs an in-sample mean ≤ α − (q − α)/m.
**Meaning: every distribution-free certificate pays a tax proportional to (q − α)/m. The cap is the only term a user controls; scenes are the only currency.** With q = 1, α = 0.05, δ = 0.1 nothing can be trusted below 45 scenes (PAC) or 19 scenes (CRC).

**T8 (product certificate — closed form, exact).** Trust threshold j if Σ_i log(1 + c·(α − L_i(j))/(q − α)) ≥ log(1/δ), with scene losses L_i(j) ∈ [0, q], c ∈ (0, 1] (we fix c = 0.75), and take the largest such j. *Validity:* let j⁺ be the first threshold with R(j⁺) > α. Each factor is ≥ 1 − c ≥ 0 and has mean < 1 at j⁺, so by Markov P(product ≥ 1/δ) ≤ δ; the statistic is monotone in j, so {ĵ ≥ j⁺} is that single event. No multiplicity correction, no dependence on scene order, finite-sample, distribution-free. Needs i.i.d. scenes.

**Cap rule (fixed constant).** Spend a quarter of the budget on the tax: q*(m) = min(1, α·(1 + 0.25/u_m)), u_m = δ^{−1/m} − 1 for PAC and u_m = 1/m for CRC.

**Trivial baseline (honesty line).** Trusting any α-fraction of pixels is valid with zero labels. A certificate is informative only when its area is above α; every table reports the area next to α and the observed error rate among trusted pixels.

**T4′ (skew-corrected law).** P(over) ≈ Φ(x) − γ·(x² − 1)·φ(x)/(6√m), x = (t − R(j⁺))√m/τ_n, γ = skewness of scene losses. Real scene losses are right-skewed (γ ≈ 3), so the patch-level limit is ½ + γ/(6√(2π m)) > ½.

**Status of the evidence.** The cap, the rule constant 0.25 and c = 0.75 were developed on the 64 calibration scenes *after* the full run (exploratory: H15, H17, H18, H20). The **winter scenes were not used** for this development: H16 and H19 are confirmatory. Hypotheses and kill rules, written before these cells were run on the full data:

| ID | Prediction | Kill rule / consequence |
|---|---|---|
| H14 | T7 necessity holds for the certificates on real scenes: P(select ≥ j) ≤ δ((q−R_j)/(q−α))^m (+3 s.e.) for every threshold; the mixture bound is attained in simulation | Violation ⇒ error in T7 or in the code |
| H15 | Calibration scenes (exploratory): capped product certificate has P(over) ≤ δ + 0.03 in every configuration, and certifies ≥ 0.02 more area than the uncapped exact bound in ≥ 80% of configurations with m ≤ 32 | Failure ⇒ the cap does not help; drop it |
| H16 | **Winter scenes (confirmatory)**: same two criteria with m ∈ {8, 16, 24} | Failure ⇒ the cap rule was over-fitted to the calibration scenes; report as exploratory only |
| H17 | Real held-out split (32/32 scenes): share of splits with test risk > α is ≤ δ + 0.05 for the capped product certificate and ≥ 0.30 for patch-level CRC (pix2pix) | Failure ⇒ superpopulation results do not transfer to held-out scenes |
| H18 | SAR2Opt (second sensor, exploratory): tile-level CRC has city-weighted mean risk > α; city-level capped CRC has mean risk ≤ α with area > 0 | Failure ⇒ the unit result is specific to SEN12MS-CR |
| H19 | **Winter recalibration (confirmatory)**: with k = 8 labelled winter scenes, capped CRC keeps mean test risk ≤ α (+2 s.e.) with area ≥ 0.02; uncapped CRC certifies 0 | Failure ⇒ few-label recalibration is not practical even with the cap |
| H20 | T4′ has max error ≤ 0.05 and mean error below the CLT version (2,000 draws) | Failure ⇒ keep T4 as qualitative only |

In [ ]:
# ===== Cell 26: Part III setup — capped curves, certificates, cap rule, domains =====
import functools

CP_CFG = TH_CFG
CP_TAG = CP_CFG["tag"]
CP_DELTA = 0.10
CP_C = 0.75
CP_FRAC = 0.25
CP_DRAWS = 500 if CP_TAG == "full" else 100
CP_DOM = {"cal": TH_CAL, "winter": load_domain(CP_CFG, "winter", [f"{a}_{b}" for a, b in TH_SPLIT["winter"]])}
CP_TI_MAIN = CP_CFG["taus"].index(CP_CFG["tau_main"])
CP_TI_SEC = CP_CFG["taus"].index(0.20)
CP_POINTS = [(0.05, CP_TI_MAIN), (0.10, CP_TI_MAIN), (0.10, CP_TI_SEC)]      # (alpha, tau index)
CP_GENS = [g for g in CP_CFG["s2o_gens"] if g != "climatology"] + ["cr_unet"]


def cp_capped_flat(cnt, bad, q, pix):
    """Per-patch capped curves from the stored histograms. Trust bins 0..j but at most q*pix pixels per patch,
    lowest scores first; inside the bin where the cap falls, pixels are taken at random (linear interpolation =
    exact expected loss). Returns L (trusted-and-wrong fraction) and K (trusted fraction), shape (patches, bins)."""
    C = np.cumsum(cnt.astype(np.float64), 1)
    Bc = np.cumsum(bad.astype(np.float64), 1)
    Q = q * pix
    v = np.array([np.interp(Q, np.r_[0.0, C[i]], np.r_[0.0, Bc[i]]) for i in range(len(C))])
    return np.minimum(Bc, v[:, None]) / pix, np.minimum(C, Q) / pix


@functools.lru_cache(maxsize=256)
def cp_scene(domain, g, s, q, ti):
    """Scene-level capped curves for a domain: SL, SK (scenes, bins) and the scene-weighted population curves."""
    dom = CP_DOM[domain]
    cnt, bad = dom.get(f"cnt|{g}|{s}"), dom.get(f"bad|{g}|{s}|full|{ti}")
    if cnt is None or bad is None or len(dom["unit"]) == 0:
        return None
    L, K = cp_capped_flat(cnt, bad, q, P_PIX)
    SL, SK = unit_mean(L, dom["unit"]), unit_mean(K, dom["unit"])
    return {"SL": SL, "SK": SK, "R": SL.mean(0), "K": SK.mean(0), "M": SL.shape[0], "q": q}


def cp_q_rule(alpha, m, delta=None, frac=CP_FRAC):
    """Cap rule: tax = frac * alpha. delta=None -> CRC tax (q-alpha)/m; else PAC tax (q-alpha)(delta^(-1/m)-1)."""
    u = (1.0 / m) if delta is None else (delta ** (-1.0 / m) - 1.0)
    return float(round(min(1.0, alpha * (1.0 + frac / u)), 4))


def cp_prefix_last(ok):
    """ok (D, bins) boolean; index of the last True of the leading run of True (-1 if the first is False)."""
    fail = ~ok
    return np.where(fail.any(1), np.argmax(fail, 1), ok.shape[1]) - 1


def cp_pick_product(X, alpha, delta, q, c=CP_C):
    """T8 product certificate on scene losses X (D draws, m scenes, bins) in [0, q]."""
    if q <= alpha:                                   # loss <= q <= alpha: valid without data (trivial regime)
        return np.full(X.shape[0], X.shape[2] - 1)
    stat = np.log1p(np.maximum(c * (alpha - X) / (q - alpha), -0.999999)).sum(1)
    return cp_prefix_last(stat >= np.log(1.0 / delta))


def cp_wsr_reject(Z, a, delta):
    """Waudby-Smith & Ramdas betting test of H0: mean >= a for Z (D, m) in [0, 1] (same bets as cell 9)."""
    n_d, m = Z.shape
    t = np.arange(1, m + 1)
    mu = (0.5 + np.cumsum(Z, 1)) / (t + 1)
    s2 = (0.25 + np.cumsum((Z - mu) ** 2, 1)) / (t + 1)
    s2p = np.c_[np.full(n_d, 0.25), s2[:, :-1]]
    lam = np.minimum(np.sqrt(2 * np.log(1 / delta) / (m * s2p)), 0.75 / max(1 - a, 1e-9))
    return np.cumprod(1 + lam * (a - Z), 1).max(1) >= 1 / delta


def cp_pick_wsr(X, alpha, delta, q):
    """Exact WSR-PAC with losses rescaled by the cap; fixed sequence from the strictest threshold upward."""
    n_d = X.shape[0]
    if q <= alpha:
        return np.full(n_d, X.shape[2] - 1)
    J, alive = np.full(n_d, -1), np.ones(n_d, bool)
    for j in range(X.shape[2]):
        if not alive.any():
            break
        rej = np.zeros(n_d, bool)
        rej[alive] = cp_wsr_reject(X[alive, :, j] / q, alpha / q, delta)
        J[alive & rej] = j
        alive &= rej
    return J


def cp_pick_crc(X, alpha, B):
    """Scene-level conformal risk control with loss bound B (= cap)."""
    return cp_prefix_last((X.sum(1) + B) / (X.shape[1] + 1) <= alpha)


def cp_eval(J, R, K, alpha):
    """P(over) = share of draws whose selected threshold has true risk > alpha; mean area; mean true risk;
    observed error rate among trusted pixels (risk / area) averaged over draws that trust something."""
    Jc = np.maximum(J, 0)
    sel = J >= 0
    risk, area = np.where(sel, R[Jc], 0.0), np.where(sel, K[Jc], 0.0)
    err = float((risk[sel] / np.maximum(area[sel], 1e-12)).mean()) if sel.any() else float("nan")
    return {"P_over": float((sel & (R[Jc] > alpha)).mean()), "area": float(area.mean()), "risk": float(risk.mean()),
            "err_among_trusted": err, "refused": float((~sel).mean())}


def cp_oracle(W, alpha):
    ok = np.nonzero(W["R"] <= alpha)[0]
    return float(W["K"][ok.max()]) if ok.size else 0.0


print({"run": CP_TAG, "cal_scenes": int(len(np.unique(CP_DOM["cal"]["unit"]))),
       "winter_scenes": int(len(np.unique(CP_DOM["winter"]["unit"]))) if len(CP_DOM["winter"]["unit"]) else 0,
       "cap_rule_examples_alpha0.05": {m: (cp_q_rule(0.05, m, CP_DELTA), cp_q_rule(0.05, m)) for m in (8, 16, 32, 64)},
       "min_scenes_uncapped_PAC": math.ceil(np.log(1 / CP_DELTA) / np.log(1 / (1 - 0.05))), "min_scenes_uncapped_CRC": math.ceil(1 / 0.05) - 1})

In [ ]:
# ===== Cell 27: T7 — the certification tax: attained in simulation, never violated on real scenes (H14) =====
def cp_t7_simulation(alpha, delta, q_list, seed=27):
    """Zero-variance world (all scene losses = 0) against the mixture adversary (loss q with probability eps, just
    above alpha/q). The full-bet product certificate (c = 1) accepts the zero world exactly when
    m >= ln(1/delta)/ln(q/(q-alpha)); at that m the adversary is accepted with probability (1-eps)^m ~ delta."""
    rng = np.random.default_rng(seed)
    rows = []
    for q in q_list:
        m_min = math.ceil(np.log(1 / delta) / np.log(q / (q - alpha)))
        eps = alpha / q * 1.0001
        for m in (m_min - 1, m_min):
            if m < 1:
                continue
            zero = np.zeros((1, m, 1))
            adv = (rng.random((20000, m, 1)) < eps) * q
            rows.append({"q": q, "m": m, "m_min_formula": m_min,
                         "accepts_zero_world": bool(cp_pick_product(zero, alpha, delta, q, c=1.0)[0] >= 0),
                         "adversary_accept_rate": float((cp_pick_product(adv, alpha, delta, q, c=1.0) >= 0).mean()),
                         "delta": delta, "bound_(1-eps)^m": float((1 - eps) ** m)})
    return rows


def cp_t7_real(domain, g, s, alpha, ti, m_list, draws, seed=28):
    """Necessity bound on real scenes: for every threshold j with R_j < alpha,
    P(certificate selects >= j) <= delta * ((q - R_j)/(q - alpha))^m, up to 3 Monte Carlo standard errors."""
    rng = np.random.default_rng(seed)
    out = []
    for m in m_list:
        q = cp_q_rule(alpha, m, CP_DELTA)
        W = cp_scene(domain, g, s, q, ti)
        if W is None or q <= alpha:
            continue
        X = W["SL"][rng.integers(0, W["M"], (draws, m))]
        for name, J in (("product", cp_pick_product(X, alpha, CP_DELTA, q)), ("wsr", cp_pick_wsr(X, alpha, CP_DELTA, q))):
            viol, tight = 0, 0.0
            for j in np.nonzero(W["R"] < alpha)[0]:
                p_sel = float((J >= j).mean())
                bound = min(1.0, CP_DELTA * ((q - W["R"][j]) / (q - alpha)) ** m)
                viol += int(p_sel > bound + 3 * np.sqrt(0.25 / draws))
                tight = max(tight, p_sel / bound) if bound < 1 else tight
            out.append({"gen": g, "m": m, "q": q, "method": name, "violations": viol, "max_ratio_to_bound_when_binding": tight})
    return out


T7_SIM = cp_t7_simulation(0.05, CP_DELTA, [1.0, 0.5, 0.25, 0.15])
T7_REAL = []
for g_cell27 in ["pix2pix", "flow_sample", "unet_l1"]:
    T7_REAL += cp_t7_real("cal", g_cell27, "learned", 0.05, CP_TI_MAIN, [8, 16, 32, 64], CP_DRAWS)
sim_ok_cell27 = all((r["accepts_zero_world"] == (r["m"] >= r["m_min_formula"])) and r["adversary_accept_rate"] <= r["delta"] + 0.01
                    for r in T7_SIM)
H14_RESULT = {"simulation_rows": T7_SIM, "simulation_matches_formula": bool(sim_ok_cell27),
              "real_violations": int(sum(r["violations"] for r in T7_REAL)), "real_rows": T7_REAL}
H14_RESULT["verdict"] = verdict(sim_ok_cell27 and H14_RESULT["real_violations"] == 0, insufficient=not T7_REAL)
save_json(f"{CP_TAG}/cap/t7_tax.json", H14_RESULT)
print("H14:", H14_RESULT["verdict"], "| real violations:", H14_RESULT["real_violations"])
for r_cell27 in T7_SIM:
    print(f"  q={r_cell27['q']:.2f} m={r_cell27['m']:3d} (formula m_min {r_cell27['m_min_formula']}): zero world accepted "
          f"{r_cell27['accepts_zero_world']}, adversary accepted {r_cell27['adversary_accept_rate']:.3f} (bound {r_cell27['bound_(1-eps)^m']:.3f})")

In [ ]:
# ===== Cell 28: capped certificates vs uncapped on the calibration scenes (H15, exploratory) =====
def cp_compare(domain, g, s, alpha, ti, m, draws, seed):
    """One configuration in the scene superpopulation of a domain (scenes drawn with replacement)."""
    q_pac, q_crc = cp_q_rule(alpha, m, CP_DELTA), cp_q_rule(alpha, m)
    W1, Wp, Wc = cp_scene(domain, g, s, 1.0, ti), cp_scene(domain, g, s, q_pac, ti), cp_scene(domain, g, s, q_crc, ti)
    if W1 is None:
        return None
    idx = np.random.default_rng(seed).integers(0, W1["M"], (draws, m))
    row = {"domain": domain, "gen": g, "score": s, "alpha": alpha, "tau": CP_CFG["taus"][ti], "m": m, "q_pac": q_pac, "q_crc": q_crc,
           "oracle_uncapped": cp_oracle(W1, alpha), "oracle_capped_pac": cp_oracle(Wp, alpha), "trivial_area": alpha}
    row["pac_wsr_uncapped"] = cp_eval(cp_pick_wsr(W1["SL"][idx], alpha, CP_DELTA, 1.0), W1["R"], W1["K"], alpha)
    row["pac_product_capped"] = cp_eval(cp_pick_product(Wp["SL"][idx], alpha, CP_DELTA, q_pac), Wp["R"], Wp["K"], alpha)
    row["pac_wsr_capped"] = cp_eval(cp_pick_wsr(Wp["SL"][idx], alpha, CP_DELTA, q_pac), Wp["R"], Wp["K"], alpha)
    row["crc_uncapped"] = cp_eval(cp_pick_crc(W1["SL"][idx], alpha, 1.0), W1["R"], W1["K"], alpha)
    row["crc_capped"] = cp_eval(cp_pick_crc(Wc["SL"][idx], alpha, q_crc), Wc["R"], Wc["K"], alpha)
    return row


def cp_cap_verdict(rows, m_small):
    """H15/H16 criteria: validity everywhere; area gain >= 0.02 over the uncapped exact bound in >= 80% of small-m configurations."""
    if not rows:
        return {"verdict": "INSUFFICIENT"}
    worst = max(r["pac_product_capped"]["P_over"] for r in rows)
    small = [r for r in rows if r["m"] <= m_small]
    gain = [r["pac_product_capped"]["area"] - r["pac_wsr_uncapped"]["area"] for r in small]
    share = float(np.mean([x >= 0.02 for x in gain])) if gain else float("nan")
    return {"configs": len(rows), "worst_P_over_product_capped": worst, "worst_P_over_wsr_capped": max(r["pac_wsr_capped"]["P_over"] for r in rows),
            "share_small_m_with_gain_ge_0.02": share, "median_gain_small_m": float(np.median(gain)) if gain else float("nan"),
            "share_capped_above_trivial": float(np.mean([r["pac_product_capped"]["area"] > r["alpha"] for r in rows])),
            "share_uncapped_above_trivial": float(np.mean([r["pac_wsr_uncapped"]["area"] > r["alpha"] for r in rows])),
            "verdict": verdict(worst <= CP_DELTA + 0.03 and share >= 0.8, insufficient=not gain)}


def cp_print_rows(rows, gen):
    for r in [r for r in rows if r["gen"] == gen]:
        print(f"  {r['gen']} a={r['alpha']} tau={r['tau']} m={r['m']:2d} q={r['q_pac']:.3f} oracle {r['oracle_uncapped']:.3f} | uncapped exact: "
              f"{r['pac_wsr_uncapped']['area']:.3f} (P_over {r['pac_wsr_uncapped']['P_over']:.3f}) | capped product: "
              f"{r['pac_product_capped']['area']:.3f} (P_over {r['pac_product_capped']['P_over']:.3f}, err among trusted "
              f"{r['pac_product_capped']['err_among_trusted']:.3f}) | capped WSR: {r['pac_wsr_capped']['area']:.3f} "
              f"(P_over {r['pac_wsr_capped']['P_over']:.3f})")


CAP_ROWS_CAL = []
for g_cell28 in CP_GENS:
    for a_cell28, ti_cell28 in CP_POINTS:
        for m_cell28 in (8, 16, 32, 64):
            row_cell28 = cp_compare("cal", g_cell28, "learned", a_cell28, ti_cell28, m_cell28, CP_DRAWS, seed=28)
            if row_cell28 is not None:
                CAP_ROWS_CAL.append(row_cell28)
H15_RESULT = cp_cap_verdict(CAP_ROWS_CAL, 32)
save_json(f"{CP_TAG}/cap/cap_calibration_scenes.json", {"rows": CAP_ROWS_CAL, "H15": H15_RESULT})
print("H15:", H15_RESULT)
cp_print_rows(CAP_ROWS_CAL, "pix2pix")

In [ ]:
# ===== Cell 29: the same comparison on the WINTER scenes, never used to design the cap (H16, confirmatory) =====
CAP_ROWS_WIN = []
if len(CP_DOM["winter"]["unit"]):
    for g_cell29 in CP_GENS:
        for a_cell29, ti_cell29 in CP_POINTS:
            for m_cell29 in (8, 16, 24):
                row_cell29 = cp_compare("winter", g_cell29, "learned", a_cell29, ti_cell29, m_cell29, CP_DRAWS, seed=29)
                if row_cell29 is not None:
                    CAP_ROWS_WIN.append(row_cell29)
H16_RESULT = cp_cap_verdict(CAP_ROWS_WIN, 24)
save_json(f"{CP_TAG}/cap/cap_winter_scenes.json", {"rows": CAP_ROWS_WIN, "H16": H16_RESULT})
print("H16:", H16_RESULT)
cp_print_rows(CAP_ROWS_WIN, "pix2pix")

In [ ]:
# ===== Cell 30: real held-out split — calibrate on half of the scenes, measure on the other half (H17) =====
def cp_holdout(g, s, alpha, ti, R, seed=30):
    dom = CP_DOM["cal"]
    units = np.unique(dom["unit"])
    m = len(units) // 2
    if m < 4:
        return None
    q_pac, q_crc = cp_q_rule(alpha, m, CP_DELTA), cp_q_rule(alpha, m)
    Ws = {q: cp_scene("cal", g, s, q, ti) for q in {1.0, q_pac, q_crc}}
    if Ws[1.0] is None:
        return None
    n_s = np.bincount(np.searchsorted(units, dom["unit"]), minlength=len(units)).astype(float)   # patches per scene
    rng = np.random.default_rng(seed)
    perms = np.stack([rng.permutation(len(units)) for _ in range(R)])
    cal_i, te_i = perms[:, :m], perms[:, m:2 * m]
    out = {"gen": g, "score": s, "alpha": alpha, "tau": CP_CFG["taus"][ti], "scenes_cal": m, "q_pac": q_pac, "q_crc": q_crc}

    def _score(J, W):
        Jc = np.maximum(J, 0)
        risk = np.where(J >= 0, np.take_along_axis(W["SL"][te_i].mean(1), Jc[:, None], 1)[:, 0], 0.0)
        area = np.where(J >= 0, np.take_along_axis(W["SK"][te_i].mean(1), Jc[:, None], 1)[:, 0], 0.0)
        return {"share_over": float((risk > alpha).mean()), "mean_test_risk": float(risk.mean()), "area": float(area.mean()),
                "refused": float((J < 0).mean())}

    W1 = Ws[1.0]
    w = n_s[cal_i][:, :, None]
    pooled_ok = ((W1["SL"][cal_i] * w).sum(1) + 1.0) / (w.sum(1) + 1.0) <= alpha         # patch-level CRC (patches pooled)
    out["patch_crc"] = _score(cp_prefix_last(pooled_ok), W1)
    out["scene_crc"] = _score(cp_pick_crc(W1["SL"][cal_i], alpha, 1.0), W1)
    out["scene_crc_capped"] = _score(cp_pick_crc(Ws[q_crc]["SL"][cal_i], alpha, q_crc), Ws[q_crc])
    out["pac_wsr_uncapped"] = _score(cp_pick_wsr(W1["SL"][cal_i], alpha, CP_DELTA, 1.0), W1)
    out["pac_product_capped"] = _score(cp_pick_product(Ws[q_pac]["SL"][cal_i], alpha, CP_DELTA, q_pac), Ws[q_pac])
    return out


HOLDOUT_ROWS = []
for g_cell30 in CP_GENS:
    for a_cell30, ti_cell30 in CP_POINTS:
        row_cell30 = cp_holdout(g_cell30, "learned", a_cell30, ti_cell30, 300 if CP_TAG == "full" else 60)
        if row_cell30 is not None:
            HOLDOUT_ROWS.append(row_cell30)
prim_cell30 = [r for r in HOLDOUT_ROWS if r["gen"] == "pix2pix" and r["alpha"] == CP_CFG["alpha_main"] and r["tau"] == CP_CFG["tau_main"]]
H17_RESULT = {"worst_share_over_product_capped": max((r["pac_product_capped"]["share_over"] for r in HOLDOUT_ROWS), default=float("nan")),
              "primary": prim_cell30[0] if prim_cell30 else None}
H17_RESULT["verdict"] = verdict(bool(prim_cell30) and H17_RESULT["worst_share_over_product_capped"] <= CP_DELTA + 0.05
                                and prim_cell30[0]["patch_crc"]["share_over"] >= 0.30,
                                insufficient=not prim_cell30 or prim_cell30[0]["scenes_cal"] < 16)
save_json(f"{CP_TAG}/cap/holdout_split.json", {"rows": HOLDOUT_ROWS, "H17": H17_RESULT})
print("H17:", H17_RESULT["verdict"], "| worst share over (capped product):", H17_RESULT["worst_share_over_product_capped"])
for r_cell30 in [r for r in HOLDOUT_ROWS if r["gen"] in ("pix2pix", "flow_sample", "cr_unet")]:
    print(f"  {r_cell30['gen']:12s} a={r_cell30['alpha']} tau={r_cell30['tau']}: " + " | ".join(
        f"{k} over {r_cell30[k]['share_over']:.2f} risk {r_cell30[k]['mean_test_risk']:.3f} area {r_cell30[k]['area']:.3f}"
        for k in ("patch_crc", "scene_crc", "scene_crc_capped", "pac_wsr_uncapped", "pac_product_capped")))

In [ ]:
# ===== Cell 31: second sensor (SAR2Opt): with the cap, CITY-level calibration becomes possible (H18, exploratory) =====
def cp_sar2opt(alpha, ti, splits=252, seed=31):
    rel = f"{CP_TAG}/hist/sar2opt/hist.npz"
    if not hf_pull(rel):
        return None
    s2 = load_npz(rel)
    city = s2["parts"][:, 0]
    cities = np.unique(city)
    half = len(cities) // 2
    if half < 3:
        return None
    pix = 512 * 512
    q_crc, q_pac = cp_q_rule(alpha, half), cp_q_rule(alpha, half, CP_DELTA)
    rng = np.random.default_rng(seed)
    res = {"n_cities": int(len(cities)), "cal_cities": int(half), "alpha": alpha, "tau": CP_CFG["taus"][ti], "q_crc": q_crc, "q_pac": q_pac}
    for sname in ("topo", "spread"):
        curves_q = {q: cp_capped_flat(s2[f"cnt|{sname}"], s2[f"bad|{sname}|{ti}"], q, pix) for q in {1.0, q_crc, q_pac}}
        acc = {k: [] for k in ("tile_crc", "city_crc_uncapped", "city_crc_capped", "city_pac_product_capped")}
        for _ in range(splits):
            cm = np.isin(city, rng.permutation(cities)[:half])
            for name, q in (("tile_crc", 1.0), ("city_crc_uncapped", 1.0), ("city_crc_capped", q_crc), ("city_pac_product_capped", q_pac)):
                L, K = curves_q[q]
                Lc = unit_mean(L[cm], city[cm])
                if name == "tile_crc":
                    j = crc_pick(L[cm], alpha)
                elif name == "city_pac_product_capped":
                    j = int(cp_pick_product(Lc[None], alpha, CP_DELTA, q)[0])
                else:
                    j = int(cp_pick_crc(Lc[None], alpha, q)[0])
                risk = float(unit_mean(L[~cm], city[~cm])[:, j].mean()) if j >= 0 else 0.0     # city-weighted test risk
                area = float(unit_mean(K[~cm], city[~cm])[:, j].mean()) if j >= 0 else 0.0
                acc[name].append((risk, area, j < 0))
        res[sname] = {k: {"share_over": float(np.mean([r > alpha for r, _, _ in v])), "mean_risk": float(np.mean([r for r, _, _ in v])),
                          "area": float(np.mean([a for _, a, _ in v])), "refused": float(np.mean([z for _, _, z in v]))} for k, v in acc.items()}
    return res


S2O_CAP = cp_sar2opt(0.10, CP_TI_MAIN)
if S2O_CAP is not None:
    t_cell31 = S2O_CAP["topo"]
    H18_RESULT = {"results": S2O_CAP,
                  "verdict": verdict(t_cell31["tile_crc"]["mean_risk"] > S2O_CAP["alpha"] and t_cell31["city_crc_capped"]["mean_risk"] <= S2O_CAP["alpha"]
                                     and t_cell31["city_crc_capped"]["area"] > 0, insufficient=S2O_CAP["cal_cities"] < 4)}
    for s_cell31 in ("topo", "spread"):
        print(f"  SAR2Opt {s_cell31}: " + " | ".join(f"{k}: over {v['share_over']:.2f} risk {v['mean_risk']:.3f} area {v['area']:.3f} refused {v['refused']:.2f}"
                                                   for k, v in S2O_CAP[s_cell31].items()))
else:
    H18_RESULT = {"verdict": "INSUFFICIENT"}
save_json(f"{CP_TAG}/cap/sar2opt_city.json", H18_RESULT)
print("H18:", H18_RESULT["verdict"], {k: S2O_CAP[k] for k in ("n_cities", "cal_cities", "q_crc", "q_pac")} if S2O_CAP else "")

In [ ]:
# ===== Cell 32: a new season with few labels — recalibrate on k labelled winter scenes (H19, confirmatory) =====
def cp_winter_recal(g, alpha, ti, k_list, R, seed=32):
    rows = []
    W1 = cp_scene("winter", g, "learned", 1.0, ti)
    if W1 is None:
        return rows
    rng = np.random.default_rng(seed)
    for k in k_list:
        if k + 4 > W1["M"]:
            continue
        q_crc, q_pac = cp_q_rule(alpha, k), cp_q_rule(alpha, k, CP_DELTA)
        Wc, Wp = cp_scene("winter", g, "learned", q_crc, ti), cp_scene("winter", g, "learned", q_pac, ti)
        perms = np.stack([rng.permutation(W1["M"]) for _ in range(R)])
        lab, te = perms[:, :k], perms[:, k:]
        row = {"gen": g, "alpha": alpha, "tau": CP_CFG["taus"][ti], "k_labelled": k, "test_scenes": int(te.shape[1]), "q_crc": q_crc, "q_pac": q_pac}
        for name, W, J in (("crc_uncapped", W1, cp_pick_crc(W1["SL"][lab], alpha, 1.0)),
                           ("crc_capped", Wc, cp_pick_crc(Wc["SL"][lab], alpha, q_crc)),
                           ("pac_product_capped", Wp, cp_pick_product(Wp["SL"][lab], alpha, CP_DELTA, q_pac))):
            Jc = np.maximum(J, 0)
            risk = np.where(J >= 0, np.take_along_axis(W["SL"][te].mean(1), Jc[:, None], 1)[:, 0], 0.0)
            area = np.where(J >= 0, np.take_along_axis(W["SK"][te].mean(1), Jc[:, None], 1)[:, 0], 0.0)
            row[name] = {"mean_test_risk": float(risk.mean()), "se": float(risk.std(ddof=1) / np.sqrt(R)), "share_over": float((risk > alpha).mean()),
                         "area": float(area.mean()), "refused": float((J < 0).mean())}
        rows.append(row)
    return rows


WINTER_RECAL = []
for g_cell32 in ["pix2pix", "unet_l1", "flow_sample", "cr_unet"]:
    for a_cell32, ti_cell32 in CP_POINTS:
        WINTER_RECAL += cp_winter_recal(g_cell32, a_cell32, ti_cell32, [4, 8, 12, 16], 300 if CP_TAG == "full" else 60)
prim_cell32 = [r for r in WINTER_RECAL if r["gen"] == "pix2pix" and r["alpha"] == CP_CFG["alpha_main"] and r["tau"] == CP_CFG["tau_main"] and r["k_labelled"] == 8]
if prim_cell32:
    p_cell32 = prim_cell32[0]
    H19_RESULT = {"primary": p_cell32,
                  "verdict": verdict(p_cell32["crc_capped"]["mean_test_risk"] <= p_cell32["alpha"] + 2 * p_cell32["crc_capped"]["se"]
                                     and p_cell32["crc_capped"]["area"] >= 0.02 and p_cell32["crc_uncapped"]["area"] == 0.0)}
else:
    H19_RESULT = {"verdict": "INSUFFICIENT"}
save_json(f"{CP_TAG}/cap/winter_recalibration.json", {"rows": WINTER_RECAL, "H19": H19_RESULT})
print("H19:", H19_RESULT["verdict"])
for r_cell32 in [r for r in WINTER_RECAL if r["gen"] == "pix2pix"]:
    print(f"  pix2pix a={r_cell32['alpha']} tau={r_cell32['tau']} k={r_cell32['k_labelled']:2d}: " + " | ".join(
        f"{k} risk {r_cell32[k]['mean_test_risk']:.3f} over {r_cell32[k]['share_over']:.2f} area {r_cell32[k]['area']:.3f}"
        for k in ("crc_uncapped", "crc_capped", "pac_product_capped")))

In [ ]:
# ===== Cell 33: T4' — the skew-corrected training-conditional law (H20, exploratory) =====
def cp_t4_skew(g, s, alpha, m_list, n_list, draws, seed=33):
    W = th_world(g, s)
    if W is None:
        return []
    above = np.nonzero(W["R"] > alpha)[0]
    if above.size == 0:
        return []
    jp = int(above.min())
    v = W["WL"][:, :, jp]
    M, n0 = v.shape
    rng = np.random.default_rng(seed)
    rows = []
    for n in [n for n in n_list if n <= n0]:
        # population moments of the scene mean of n patches (Monte Carlo over patch subsets only)
        sm = np.concatenate([np.take_along_axis(v, np.argsort(rng.random((M, n0)), 1)[:, :n], 1).mean(1) for _ in range(200)])
        tau_n, gam = float(sm.std()), float(sstats.skew(sm))
        for m in m_list:
            si = rng.integers(0, M, (draws, m))
            pi = np.argsort(rng.random((draws, m, n0)), 2)[:, :, :n]
            A = np.take_along_axis(v[si], pi, 2).mean((1, 2))
            for mode in ("patch", "scene"):
                t = alpha - (1 - alpha) / (m * n) if mode == "patch" else alpha - (1 - alpha) / m
                x = (t - W["R"][jp]) * np.sqrt(m) / max(tau_n, 1e-12)
                clt = float(sstats.norm.cdf(x))
                edge = float(np.clip(clt - gam * (x * x - 1) * sstats.norm.pdf(x) / (6 * np.sqrt(m)), 0, 1))
                rows.append({"gen": g, "m": m, "n": n, "mode": mode, "observed": float((A <= t + 1e-12).mean()), "clt": clt,
                             "skew_corrected": edge, "skewness": gam})
    return rows


T4S_ROWS = []
for g_cell33 in ["pix2pix", "flow_sample", "unet_l1"]:
    T4S_ROWS += cp_t4_skew(g_cell33, "learned", TH_ALPHA, [16, 32, 64], [1, 2, 4, 8, 16, 32, 64], 2000 if CP_TAG == "full" else 300)
if T4S_ROWS:
    e_clt_cell33 = [abs(r["observed"] - r["clt"]) for r in T4S_ROWS]
    e_edge_cell33 = [abs(r["observed"] - r["skew_corrected"]) for r in T4S_ROWS]
    H20_RESULT = {"max_error_clt": max(e_clt_cell33), "max_error_skew_corrected": max(e_edge_cell33), "mean_error_clt": float(np.mean(e_clt_cell33)),
                  "mean_error_skew_corrected": float(np.mean(e_edge_cell33)), "skewness_range": [min(r["skewness"] for r in T4S_ROWS), max(r["skewness"] for r in T4S_ROWS)],
                  "max_observed_patch_P_over": max(r["observed"] for r in T4S_ROWS if r["mode"] == "patch"),
                  "informative": max(r["observed"] for r in T4S_ROWS) >= 0.02}
    H20_RESULT["verdict"] = verdict(H20_RESULT["max_error_skew_corrected"] <= 0.05 and H20_RESULT["mean_error_skew_corrected"] < H20_RESULT["mean_error_clt"],
                                    insufficient=not H20_RESULT["informative"])
else:
    H20_RESULT = {"verdict": "INSUFFICIENT"}
save_json(f"{CP_TAG}/cap/t4_skew.json", {"rows": T4S_ROWS, "H20": H20_RESULT})
print("H20:", H20_RESULT)

In [ ]:
# ===== Cell 34: where does trust live? within-patch vs between-patch ranking; and the precision tier (diagnostics) =====
def cp_trust_location(g, s, ti, area=0.20):
    """Error rate among trusted pixels at a fixed trusted area, for three rules that use the same score:
    global threshold (pixels compete across all patches), patch rank (best `area` share of EVERY patch: within-patch
    ranking only), whole patches (best patches by mean score: between-patch ranking only)."""
    dom = CP_DOM["cal"]
    cnt, bad = dom.get(f"cnt|{g}|{s}"), dom.get(f"bad|{g}|{s}|full|{ti}")
    if cnt is None or bad is None or s not in TH_EDGES.get(g, {}):
        return None
    c, b = cnt.astype(np.float64), bad.astype(np.float64)
    C, Bc = np.cumsum(c, 1), np.cumsum(b, 1)
    base = float(b.sum() / c.sum())
    Kg, Rg = C.mean(0) / P_PIX, Bc.mean(0) / P_PIX
    jg = int(min(np.searchsorted(Kg, area), len(Kg) - 1))
    within = float(np.mean([np.interp(area * P_PIX, np.r_[0.0, C[i]], np.r_[0.0, Bc[i]]) for i in range(len(C))]) / (area * P_PIX))
    e = np.r_[0.0, np.array(TH_EDGES[g][s], float), 1.0]
    mean_score = (c * ((e[:-1] + e[1:]) / 2)[None, :]).sum(1) / P_PIX
    order = np.argsort(mean_score)
    top = order[:max(1, int(round(area * len(order))))]
    return {"gen": g, "score": s, "tau": CP_CFG["taus"][ti], "area": area, "base_error": base, "global_threshold": float(Rg[jg] / max(Kg[jg], 1e-12)),
            "patch_rank_only": within, "whole_patches_only": float(b[top].sum() / c[top].sum())}


def cp_precision_tier(g, s, ti, eps, m_list, draws, seed=34):
    """Precision certificate: 'at most eps of trusted pixels are wrong' (ratio of expectations), product test on
    D = L - eps*K in [-eps, 1-eps], Bonferroni over a coarse fixed grid of thresholds (the ratio is not monotone)."""
    W = cp_scene("cal", g, s, 1.0, ti)
    if W is None:
        return None
    grid = np.array([9, 19, 29, 39, 59, 79, 119, 159, 199])
    grid = grid[grid < W["SL"].shape[1]]
    R, K = W["R"], W["K"]
    okg = np.nonzero(R[grid] <= eps * K[grid])[0]
    out = {"gen": g, "tau": CP_CFG["taus"][ti], "eps": eps, "oracle_area_on_grid": float(K[grid[okg.max()]]) if okg.size else 0.0}
    rng = np.random.default_rng(seed)
    for m in m_list:
        idx = rng.integers(0, W["M"], (draws, m))
        stat = np.log1p(CP_C * (eps * W["SK"][idx][:, :, grid] - W["SL"][idx][:, :, grid]) / (1 - eps)).sum(1)
        ok = stat >= np.log(len(grid) / CP_DELTA)
        J = np.where(ok.any(1), grid[len(grid) - 1 - np.argmax(ok[:, ::-1], 1)], -1)
        Jc = np.maximum(J, 0)
        out[f"m={m}"] = {"P_over": float(((J >= 0) & (R[Jc] > eps * K[Jc])).mean()), "area": float(np.where(J >= 0, K[Jc], 0.0).mean())}
    return out


TRUST_LOC = [r for r in (cp_trust_location(g, "learned", ti) for g in CP_GENS for ti in (CP_TI_MAIN, CP_TI_SEC)) if r is not None]
PREC_TIER = [r for r in (cp_precision_tier(g, "learned", ti, eps, [16, 32, 64], max(100, CP_DRAWS // 2))
                         for g in ("pix2pix", "unet_l1", "cr_unet") for ti in (CP_TI_MAIN, CP_TI_SEC) for eps in (0.2, 0.3)) if r is not None]
save_json(f"{CP_TAG}/cap/trust_location_and_precision.json", {"trust_location": TRUST_LOC, "precision_tier": PREC_TIER})
print("error rate among trusted pixels at 20% trusted area (lower is better):")
for r_cell34 in TRUST_LOC:
    print(f"  {r_cell34['gen']:18s} tau={r_cell34['tau']}: base {r_cell34['base_error']:.3f} | global threshold {r_cell34['global_threshold']:.3f} | "
          f"whole patches only {r_cell34['whole_patches_only']:.3f} | within-patch ranking only {r_cell34['patch_rank_only']:.3f}")
print("precision tier (P_over / area):")
for r_cell34 in PREC_TIER:
    print(f"  {r_cell34['gen']:8s} tau={r_cell34['tau']} eps={r_cell34['eps']} oracle {r_cell34['oracle_area_on_grid']:.2f}: " +
          " | ".join(f"{k} {v['P_over']:.2f}/{v['area']:.3f}" for k, v in r_cell34.items() if k.startswith("m=")))

In [ ]:
# ===== Cell 35: the label-budget table that follows from T7 (formula) next to what was observed =====
def cp_budget_table(alpha_list, delta, q_list):
    rows = []
    for alpha in alpha_list:
        for q in q_list:
            if q <= alpha:
                continue
            rows.append({"alpha": alpha, "cap_q": q, "min_scenes_CRC": max(1, math.ceil(q / alpha) - 1),
                         "min_scenes_PAC_any_method": math.ceil(np.log(1 / delta) / np.log(q / (q - alpha))),
                         "min_scenes_product_c0.75": math.ceil(np.log(1 / delta) / np.log(1 + CP_C * alpha / (q - alpha))),
                         "risk_target_at_m32_PAC": alpha - (q - alpha) * (delta ** (-1 / 32) - 1),
                         "risk_target_at_m32_CRC": alpha - (q - alpha) / 32})
    return rows


BUDGET_T7 = cp_budget_table([0.05, 0.10], CP_DELTA, [1.0, 0.5, 0.25, 0.15])
obs_cell35 = {}
for r_cell35 in CAP_ROWS_CAL:
    key_cell35 = f"{r_cell35['gen']}|a={r_cell35['alpha']}|tau={r_cell35['tau']}"
    ent_cell35 = obs_cell35.setdefault(key_cell35, {"first_m_uncapped_above_trivial": None, "first_m_capped_above_trivial": None})
    if ent_cell35["first_m_uncapped_above_trivial"] is None and r_cell35["pac_wsr_uncapped"]["area"] > r_cell35["alpha"]:
        ent_cell35["first_m_uncapped_above_trivial"] = r_cell35["m"]
    if ent_cell35["first_m_capped_above_trivial"] is None and r_cell35["pac_product_capped"]["area"] > r_cell35["alpha"]:
        ent_cell35["first_m_capped_above_trivial"] = r_cell35["m"]
save_json(f"{CP_TAG}/cap/label_budget_t7.json", {"formula": BUDGET_T7, "observed_first_informative_m": obs_cell35})
for r_cell35 in BUDGET_T7:
    print(f"  alpha={r_cell35['alpha']} cap={r_cell35['cap_q']:.2f}: min scenes CRC {r_cell35['min_scenes_CRC']:3d} | PAC (any method) "
          f"{r_cell35['min_scenes_PAC_any_method']:3d} | product c=0.75 {r_cell35['min_scenes_product_c0.75']:3d}")
print("first m (of 8, 16, 32, 64) at which the per-user certificate beats the trivial area alpha:")
for k_cell35, v_cell35 in obs_cell35.items():
    print(f"  {k_cell35}: uncapped {v_cell35['first_m_uncapped_above_trivial']} | capped {v_cell35['first_m_capped_above_trivial']}")

In [ ]:
# ===== Cell 36: figure for Part III =====
fig_cell36, axes_cell36 = plt.subplots(1, 3, figsize=(16.5, 4.6))
col_cell36 = {"pac_wsr_uncapped": "#0072B2", "pac_product_capped": "#D55E00", "pac_wsr_capped": "#009E73", "crc_capped": "#CC79A7"}
lab_cell36 = {"pac_wsr_uncapped": "exact PAC, uncapped", "pac_product_capped": "product certificate, capped", "pac_wsr_capped": "WSR, capped",
              "crc_capped": "scene CRC, capped"}
mk_cell36 = {"pac_wsr_uncapped": "o", "pac_product_capped": "s", "pac_wsr_capped": "^", "crc_capped": "D"}
sel_cell36 = sorted([r for r in CAP_ROWS_CAL if r["gen"] == "pix2pix" and r["alpha"] == CP_CFG["alpha_main"] and r["tau"] == CP_CFG["tau_main"]], key=lambda r: r["m"])
for ax_cell36, field_cell36, yl_cell36 in ((axes_cell36[0], "area", "certified trusted area"), (axes_cell36[1], "P_over", "P(user over budget)")):
    for k_cell36 in col_cell36:
        ax_cell36.plot([r["m"] for r in sel_cell36], [r[k_cell36][field_cell36] for r in sel_cell36], color=col_cell36[k_cell36],
                       marker=mk_cell36[k_cell36], lw=2, ms=7, label=lab_cell36[k_cell36])
    ax_cell36.set_xscale("log", base=2)
    ax_cell36.set_xticks([r["m"] for r in sel_cell36])
    ax_cell36.set_xticklabels([str(r["m"]) for r in sel_cell36])
    ax_cell36.set_xlabel("labelled calibration scenes m")
    ax_cell36.set_ylabel(yl_cell36)
    ax_cell36.grid(alpha=0.25, lw=0.6)
if sel_cell36:
    axes_cell36[0].axhline(sel_cell36[0]["oracle_uncapped"], color="#666666", ls="--", lw=1)
    axes_cell36[0].text(sel_cell36[0]["m"], sel_cell36[0]["oracle_uncapped"], " oracle (uncapped)", va="bottom", fontsize=9, color="#444444")
    axes_cell36[0].axhline(CP_CFG["alpha_main"], color="#666666", ls=":", lw=1)
    axes_cell36[0].text(sel_cell36[0]["m"], CP_CFG["alpha_main"], " trivial area = alpha", va="bottom", fontsize=9, color="#444444")
    axes_cell36[1].axhline(CP_DELTA, color="#666666", ls="--", lw=1)
    axes_cell36[1].text(sel_cell36[0]["m"], CP_DELTA, " delta", va="bottom", fontsize=9, color="#444444")
axes_cell36[0].set_title("Certified area vs labelled scenes (pix2pix, main point)")
axes_cell36[1].set_title("Share of users over budget vs labelled scenes")
axes_cell36[0].legend(frameon=False, fontsize=9)
pt_cell36 = [r for r in T4S_ROWS if r["mode"] == "patch"]
axes_cell36[2].plot([0, 0.7], [0, 0.7], color="#666666", ls="--", lw=1)
axes_cell36[2].scatter([r["observed"] for r in pt_cell36], [r["clt"] for r in pt_cell36], s=28, color="#0072B2", marker="o", label="normal approximation (T4)")
axes_cell36[2].scatter([r["observed"] for r in pt_cell36], [r["skew_corrected"] for r in pt_cell36], s=28, color="#D55E00", marker="s", label="skew-corrected (T4')")
axes_cell36[2].set_xlabel("observed P(over), patch-level CRC")
axes_cell36[2].set_ylabel("predicted P(over)")
axes_cell36[2].set_title("Training-conditional law: predicted vs observed")
axes_cell36[2].legend(frameon=False, fontsize=9)
axes_cell36[2].grid(alpha=0.25, lw=0.6)
fig_cell36.tight_layout()
atomic_write(f"{CP_TAG}/cap/part3_figure.png", lambda p: fig_cell36.savefig(p, dpi=160))
hf_push([f"{CP_TAG}/cap/part3_figure.png"])
plt.close(fig_cell36)
print("figure saved:", bucket_key(f"{CP_TAG}/cap/part3_figure.png"))

In [ ]:
# ===== Cell 37: Part III report (H14–H20), saved to the bucket and printed in full =====
CP_HYP_TEXT = {
    "H14": "T7 necessity: never violated on real scenes; attained by the mixture adversary in simulation",
    "H15": "Calibration scenes (exploratory): capped product certificate valid everywhere and >= 0.02 more area than the uncapped exact bound in >= 80% of configurations with m <= 32",
    "H16": "Winter scenes (CONFIRMATORY, not used to design the cap): same criteria, m in {8, 16, 24}",
    "H17": "Held-out split: capped product certificate over budget in <= delta + 0.05 of splits; patch-level CRC in >= 0.30",
    "H18": "SAR2Opt (exploratory): tile-level CRC breaks the average promise; city-level capped CRC keeps it with area > 0",
    "H19": "Winter recalibration (CONFIRMATORY): 8 labelled winter scenes suffice for capped CRC; uncapped certifies nothing",
    "H20": "T4' skew-corrected law: max error <= 0.05 and mean error below the normal approximation",
}
CP_RESULTS = {"H14": H14_RESULT, "H15": H15_RESULT, "H16": H16_RESULT, "H17": H17_RESULT, "H18": H18_RESULT, "H19": H19_RESULT, "H20": H20_RESULT}


def cp_method_table(rows, title):
    out = ["", f"## {title}", "",
           "| generator | alpha | tau | m | cap q | oracle (uncapped) | exact PAC uncapped: area (P_over) | product capped: area (P_over) | WSR capped: area (P_over) | error among trusted (product) | CRC uncapped: area (P_over) | CRC capped: area (P_over) |",
           "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for r in rows:
        out.append(f"| {r['gen']} | {r['alpha']} | {r['tau']} | {r['m']} | {r['q_pac']:.3f} | {r['oracle_uncapped']:.3f} | "
                   f"{r['pac_wsr_uncapped']['area']:.3f} ({r['pac_wsr_uncapped']['P_over']:.3f}) | "
                   f"{r['pac_product_capped']['area']:.3f} ({r['pac_product_capped']['P_over']:.3f}) | "
                   f"{r['pac_wsr_capped']['area']:.3f} ({r['pac_wsr_capped']['P_over']:.3f}) | {r['pac_product_capped']['err_among_trusted']:.3f} | "
                   f"{r['crc_uncapped']['area']:.3f} ({r['crc_uncapped']['P_over']:.3f}) | {r['crc_capped']['area']:.3f} ({r['crc_capped']['P_over']:.3f}) |")
    return out


lines_cell37 = [f"# CERTEO Part III report — capped trust masks and the certification tax — run '{CP_TAG}'", "",
                f"delta = {CP_DELTA}, product-certificate bet c = {CP_C}, cap rule: tax = {CP_FRAC} x alpha, draws per configuration: {CP_DRAWS}", ""]
if CP_TAG == "smoke":
    lines_cell37 += ["> **SMOKE RUN**: these verdicts only check that the cells work; do not quote them.", ""]
lines_cell37 += ["| Hypothesis | Prediction | Verdict |", "|---|---|---|"]
for h_cell37, txt_cell37 in CP_HYP_TEXT.items():
    lines_cell37.append(f"| {h_cell37} | {txt_cell37} | **{CP_RESULTS[h_cell37].get('verdict', 'INSUFFICIENT')}** |")
lines_cell37 += cp_method_table(CAP_ROWS_CAL, "Calibration scenes (scene superpopulation) — exploratory")
lines_cell37 += cp_method_table(CAP_ROWS_WIN, "Winter scenes (scene superpopulation) — confirmatory")
lines_cell37 += ["", "## Held-out split (half of the calibration scenes calibrate, the other half test): share of splits over budget / mean test risk / area", "",
                 "| generator | alpha | tau | patch CRC | scene CRC | scene CRC capped | exact PAC uncapped | product capped |", "|---|---|---|---|---|---|---|---|"]
for r_cell37 in HOLDOUT_ROWS:
    lines_cell37.append(f"| {r_cell37['gen']} | {r_cell37['alpha']} | {r_cell37['tau']} | " + " | ".join(
        f"{r_cell37[k]['share_over']:.2f} / {r_cell37[k]['mean_test_risk']:.3f} / {r_cell37[k]['area']:.3f}"
        for k in ("patch_crc", "scene_crc", "scene_crc_capped", "pac_wsr_uncapped", "pac_product_capped")) + " |")
lines_cell37 += ["", "## Winter recalibration with k labelled scenes: mean test risk / share over / area", "",
                 "| generator | alpha | tau | k | CRC uncapped | CRC capped | product capped |", "|---|---|---|---|---|---|---|"]
for r_cell37 in WINTER_RECAL:
    lines_cell37.append(f"| {r_cell37['gen']} | {r_cell37['alpha']} | {r_cell37['tau']} | {r_cell37['k_labelled']} | " + " | ".join(
        f"{r_cell37[k]['mean_test_risk']:.3f} / {r_cell37[k]['share_over']:.2f} / {r_cell37[k]['area']:.3f}"
        for k in ("crc_uncapped", "crc_capped", "pac_product_capped")) + " |")
lines_cell37 += ["", "## Where trust lives: error rate among trusted pixels at 20% trusted area", "",
                 "| generator | tau | base error | global threshold | whole patches only | within-patch ranking only |", "|---|---|---|---|---|---|"]
for r_cell37 in TRUST_LOC:
    lines_cell37.append(f"| {r_cell37['gen']} | {r_cell37['tau']} | {r_cell37['base_error']:.3f} | {r_cell37['global_threshold']:.3f} | "
                        f"{r_cell37['whole_patches_only']:.3f} | {r_cell37['patch_rank_only']:.3f} |")
lines_cell37 += ["", "## T4' rows (patch-level, m = 16)", "", "| generator | n | observed | normal | skew-corrected | skewness |", "|---|---|---|---|---|---|"]
for r_cell37 in [r for r in T4S_ROWS if r["mode"] == "patch" and r["m"] == 16]:
    lines_cell37.append(f"| {r_cell37['gen']} | {r_cell37['n']} | {r_cell37['observed']:.3f} | {r_cell37['clt']:.3f} | {r_cell37['skew_corrected']:.3f} | {r_cell37['skewness']:.2f} |")
lines_cell37 += ["", "## Precision tier, T7 budget table, SAR2Opt", "", "```json",
                 json.dumps({"precision_tier": PREC_TIER, "budget_formula": BUDGET_T7, "sar2opt": S2O_CAP}, indent=1, default=float)[:9000], "```", "", "## Details", ""]
for h_cell37, res_cell37 in CP_RESULTS.items():
    lines_cell37 += [f"### {h_cell37} — {res_cell37.get('verdict')}", "```json",
                     json.dumps({k: v for k, v in res_cell37.items() if k not in ("verdict", "real_rows", "results")}, indent=1, default=float)[:3500], "```", ""]
PART3_REPORT = "\n".join(lines_cell37)


def cp_write_report(p):
    with open(p, "w") as fh:
        fh.write(PART3_REPORT)


atomic_write(f"{CP_TAG}/PART3_REPORT.md", cp_write_report)
save_json(f"{CP_TAG}/cap/part3_results.json", {k: {kk: vv for kk, vv in v.items() if kk not in ("real_rows",)} for k, v in CP_RESULTS.items()})
hf_push([f"{CP_TAG}/PART3_REPORT.md"])
print(PART3_REPORT)
print("\nsaved:", bucket_key(f"{CP_TAG}/PART3_REPORT.md"), "| characters:", len(PART3_REPORT))

## Part IV — Checks on ideas that did not work, the qualitative example, and the paper figures (cells 38–41)

**Why this part exists.** Parts I–III test the claims of the paper. Part IV records the ideas we tried that did *not* work, so a reader can rerun them, and it produces the two figures of the paper from the results above.

The five checks below are **exploratory**. They were run on the 64 calibration scenes only, and none of them is used by the method in Part III.

| ID | Idea | What would have made it useful |
|---|---|---|
| N1 | Use the verifier's own predicted false trust on *unlabelled* scenes as a control variate | Scene-level correlation with the true loss ≥ 0.9, so the variance of the corrected loss drops a lot |
| N2 | Rank-normalised masks: trust the best q share of *every* patch | The best 1% of pixels in a patch are wrong far less often than the base rate |
| N3 | A skew-corrected t bound for the per-user (PAC) tier | Share of users over budget ≤ δ |
| N4 | Choose the cap q from a grid with a Bonferroni correction instead of the fixed rule | More certified area than the rule, at P(over) ≤ δ |
| N5 | Other bets c ∈ {0.5, 1} in the product certificate | More area than c = 0.75, at P(over) ≤ δ |

Cell 39 runs Pix2Pix and its verifier on three calibration scenes, which are left out of the calibration for this example. Cell 40 draws the two paper figures. Cell 41 writes the Part IV report.

In [ ]:
# ===== Cell 38: checks on ideas that did not work (N1–N5) =====
PT_DOMAIN = CP_DOM["cal"]
PT_M_LIST = [m for m in (16, 32, 64) if m <= max(2, int(len(np.unique(PT_DOMAIN["unit"]))))] or [4]
PT_SEED = 40


def pt_predicted_curve(g, s):
    """Scene-level predicted false trust from the score itself (score = predicted probability of error):
    sum over trusted bins of count x bin mid-point. Uses no labels."""
    e = np.r_[0.0, np.array(TH_EDGES[g][s], float), 1.0]
    mid = (e[:-1] + e[1:]) / 2
    cnt = PT_DOMAIN.get(f"cnt|{g}|{s}")
    if cnt is None:
        return None
    return unit_mean(np.cumsum(cnt * mid[None, :], 1) / P_PIX, PT_DOMAIN["unit"])


def pt_control_variate(g, s, alpha, ti):
    W = cp_scene("cal", g, s, 1.0, ti)
    G = pt_predicted_curve(g, s)
    if W is None or G is None:
        return None
    ok = np.nonzero(W["R"] <= alpha)[0]
    if not ok.size:
        return None
    j = int(ok.max())
    Lj, Gj = W["SL"][:, j], G[:, j]
    b = float(np.cov(Lj, Gj)[0, 1] / max(Gj.var(ddof=1), 1e-12))
    return {"gen": g, "score": s, "alpha": alpha, "tau": CP_CFG["taus"][ti], "bin": j, "mean_loss": float(Lj.mean()),
            "mean_predicted": float(Gj.mean()), "corr": float(np.corrcoef(Lj, Gj)[0, 1]),
            "variance_cut": float(1 - (Lj - b * Gj).var() / max(Lj.var(), 1e-12)),
            "range_ratio": float(np.ptp(Lj - b * Gj) / max(np.ptp(Lj), 1e-12))}


PT_QGRID = np.r_[0.01, 0.02, 0.03, 0.05, 0.075, np.arange(0.10, 1.0001, 0.05)]


def pt_rank_masks(g, s, ti):
    """Trust the lowest-score share q of every patch; error rate among trusted pixels, averaged over scenes."""
    cnt, bad = PT_DOMAIN.get(f"cnt|{g}|{s}"), PT_DOMAIN.get(f"bad|{g}|{s}|full|{ti}")
    if cnt is None or bad is None:
        return None
    C, Bc = np.cumsum(cnt.astype(float), 1), np.cumsum(bad.astype(float), 1)
    Z = np.stack([np.interp(PT_QGRID * P_PIX, np.r_[0, C[i]], np.r_[0, Bc[i]]) for i in range(len(C))]) / (PT_QGRID * P_PIX)[None, :]
    Zs = unit_mean(Z, PT_DOMAIN["unit"]).mean(0)
    return {"gen": g, "score": s, "tau": CP_CFG["taus"][ti], "base_error": float(Zs[-1]),
            "error_best_1pct": float(Zs[0]), "error_best_10pct": float(Zs[5]), "error_best_20pct": float(Zs[7])}


def pt_skew_t(X, alpha, delta):
    """Skew-corrected one-sided t upper bound, fixed sequence over thresholds (uncapped losses)."""
    m = X.shape[1]
    mean, sd = X.mean(1), X.std(1, ddof=1)
    gam = np.where(sd > 0, ((X - mean[:, None]) ** 3).mean(1) / np.maximum(sd, 1e-12) ** 3, 0.0)
    z, tq = sstats.norm.ppf(1 - delta), sstats.t.ppf(1 - delta, m - 1)
    ub = mean + sd / np.sqrt(m) * (tq + np.maximum(gam, 0) * (2 * z * z + 1) / (6 * np.sqrt(m)))
    return cp_prefix_last(ub <= alpha)


def pt_draws(W, m, seed):
    return W["SL"][np.random.default_rng(seed).integers(0, W["M"], (CP_DRAWS, m))]


N1_ROWS = [r for r in (pt_control_variate(g, s, a, CP_TI_MAIN) for g in CP_GENS for s in ("learned", "floor") for a in (0.05, 0.10)) if r]
N2_ROWS = [r for r in (pt_rank_masks(g, "learned", ti) for g in CP_GENS for ti in (CP_TI_MAIN, CP_TI_SEC)) if r]

N3_ROWS = []
for g_cell38 in ("pix2pix", "flow_sample", "unet_l1"):
    for alpha_cell38, ti_cell38 in CP_POINTS:
        W_cell38 = cp_scene("cal", g_cell38, "learned", 1.0, ti_cell38)
        if W_cell38 is None:
            continue
        for m_cell38 in PT_M_LIST:
            J_cell38 = pt_skew_t(pt_draws(W_cell38, m_cell38, PT_SEED), alpha_cell38, CP_DELTA)
            N3_ROWS.append({"gen": g_cell38, "alpha": alpha_cell38, "tau": CP_CFG["taus"][ti_cell38], "m": m_cell38,
                            **cp_eval(J_cell38, W_cell38["R"], W_cell38["K"], alpha_cell38)})

N4_ROWS, N5_ROWS = [], []
for g_cell38 in ("pix2pix", "flow_sample", "unet_l1"):
    for alpha_cell38, ti_cell38 in CP_POINTS:
        for m_cell38 in PT_M_LIST:
            q_rule_cell38 = cp_q_rule(alpha_cell38, m_cell38, CP_DELTA)
            if cp_scene("cal", g_cell38, "learned", 1.0, ti_cell38) is None:
                continue
            # N4: fixed q grid (each separately), the rule, and a Bonferroni choice over the grid
            row_cell38 = {"gen": g_cell38, "alpha": alpha_cell38, "tau": CP_CFG["taus"][ti_cell38], "m": m_cell38, "q_rule": q_rule_cell38, "by_q": {}}
            for q_cell38 in sorted({0.15, 0.25, 0.35, 0.5, 1.0, q_rule_cell38}):
                W_cell38 = cp_scene("cal", g_cell38, "learned", q_cell38, ti_cell38)
                J_cell38 = cp_pick_wsr(pt_draws(W_cell38, m_cell38, PT_SEED), alpha_cell38, CP_DELTA, q_cell38)
                row_cell38["by_q"][str(q_cell38)] = cp_eval(J_cell38, W_cell38["R"], W_cell38["K"], alpha_cell38)
            grid_cell38 = [1.0, 0.5, 0.35, 0.25, 0.15]
            idx_cell38 = np.random.default_rng(PT_SEED).integers(0, cp_scene("cal", g_cell38, "learned", 1.0, ti_cell38)["M"], (CP_DRAWS, m_cell38))
            best_est, best_over, best_area = np.full(CP_DRAWS, -1.0), np.zeros(CP_DRAWS, bool), np.zeros(CP_DRAWS)
            for q_cell38 in grid_cell38:
                W_cell38 = cp_scene("cal", g_cell38, "learned", q_cell38, ti_cell38)
                J_cell38 = cp_pick_wsr(W_cell38["SL"][idx_cell38], alpha_cell38, CP_DELTA / len(grid_cell38), q_cell38)
                Jc_cell38 = np.maximum(J_cell38, 0)
                est_cell38 = np.where(J_cell38 >= 0, np.take_along_axis(W_cell38["SK"][idx_cell38].mean(1), Jc_cell38[:, None], 1)[:, 0], 0.0)
                upd_cell38 = est_cell38 > best_est     # chosen by the calibration-sample area (no test labels)
                best_est = np.where(upd_cell38, est_cell38, best_est)
                best_area = np.where(upd_cell38, np.where(J_cell38 >= 0, W_cell38["K"][Jc_cell38], 0.0), best_area)
                best_over = np.where(upd_cell38, (J_cell38 >= 0) & (W_cell38["R"][Jc_cell38] > alpha_cell38), best_over)
            row_cell38["bonferroni"] = {"P_over": float(best_over.mean()), "area": float(best_area.mean())}
            N4_ROWS.append(row_cell38)
            # N5: bet c of the product certificate at the rule cap, against capped WSR
            W_cell38 = cp_scene("cal", g_cell38, "learned", q_rule_cell38, ti_cell38)
            X_cell38 = pt_draws(W_cell38, m_cell38, PT_SEED + 1)
            row5_cell38 = {"gen": g_cell38, "alpha": alpha_cell38, "tau": CP_CFG["taus"][ti_cell38], "m": m_cell38, "q": q_rule_cell38,
                           "wsr": cp_eval(cp_pick_wsr(X_cell38, alpha_cell38, CP_DELTA, q_rule_cell38), W_cell38["R"], W_cell38["K"], alpha_cell38)}
            for c_cell38 in (0.5, 0.75, 1.0):
                row5_cell38[f"product_c{c_cell38}"] = cp_eval(cp_pick_product(X_cell38, alpha_cell38, CP_DELTA, q_rule_cell38, c=c_cell38),
                                                             W_cell38["R"], W_cell38["K"], alpha_cell38)
            N5_ROWS.append(row5_cell38)

N_SUMMARY = {
    "N1_max_corr": max((r["corr"] for r in N1_ROWS), default=float("nan")),
    "N1_max_variance_cut": max((r["variance_cut"] for r in N1_ROWS), default=float("nan")),
    "N2_pix2pix_best_1pct_vs_base": [(r["tau"], round(r["error_best_1pct"], 3), round(r["base_error"], 3)) for r in N2_ROWS if r["gen"] == "pix2pix"],
    "N3_P_over_range": [min((r["P_over"] for r in N3_ROWS), default=float("nan")), max((r["P_over"] for r in N3_ROWS), default=float("nan"))],
    "N4_bonferroni_minus_rule_area": [round(r["bonferroni"]["area"] - r["by_q"][str(r["q_rule"])]["area"], 4) for r in N4_ROWS],
    "N5_max_area_change_vs_c0.75": max((max(abs(r[f"product_c{c}"]["area"] - r["product_c0.75"]["area"]) for c in (0.5, 1.0)) for r in N5_ROWS), default=float("nan")),
}
save_json(f"{CP_TAG}/paper/negative_checks.json", {"N1": N1_ROWS, "N2": N2_ROWS, "N3": N3_ROWS, "N4": N4_ROWS, "N5": N5_ROWS, "summary": N_SUMMARY})
print(json.dumps(N_SUMMARY, indent=1, default=float))

In [ ]:
# ===== Cell 39: qualitative example — Pix2Pix + verifier on three left-out calibration scenes =====
QL_ALPHA, QL_TI = CP_CFG["alpha_main"], CP_TI_MAIN
QL_TAU = CP_CFG["taus"][QL_TI]
ql_names_cell39 = [f"{a}_{b}" for a, b in TH_SPLIT["cal"]]
QL_SHOW = [n for n in ("summer_90", "spring_40", "fall_112") if n in ql_names_cell39] or ql_names_cell39[:3]
ql_keep_cell39 = np.array([n not in QL_SHOW for n in ql_names_cell39])
QL_M = int(ql_keep_cell39.sum())
QL_Q = cp_q_rule(QL_ALPHA, QL_M, CP_DELTA)
ql_W_cell39 = cp_scene("cal", "pix2pix", "learned", QL_Q, QL_TI)
QL_J = int(cp_pick_product(ql_W_cell39["SL"][ql_keep_cell39][None], QL_ALPHA, CP_DELTA, QL_Q)[0])
QL_EDGES = np.array(TH_EDGES["pix2pix"]["learned"], np.float32)
QL_REL = f"{CP_TAG}/paper/qual.npz"
print({"left_out_scenes": QL_SHOW, "calibration_scenes": QL_M, "cap_q": QL_Q, "selected_bin": QL_J,
       "calibration_area": float(ql_W_cell39["SK"][ql_keep_cell39].mean(0)[max(QL_J, 0)]) if QL_J >= 0 else 0.0})

if hf_pull(QL_REL):
    QL = load_npz(QL_REL)
    print("qualitative example loaded from storage:", QL_REL)
else:
    ql_G = UNetG(2, 3).to(DEVICE)
    ql_G.load_state_dict(load_ckpt(f"{CP_TAG}/models/pix2pix_final.pt")["model_G"])
    ql_V = UNetG(5, 1, base=32, p_drop=0.0, out_act=None).to(DEVICE)
    ql_V.load_state_dict(load_ckpt(f"{CP_TAG}/models/verifier_pix2pix_final.pt")["model_V"])
    ql_G.eval(), ql_V.eval()
    set_mc_dropout(ql_G, False)
    ql_out = {}
    for nm_cell39 in QL_SHOW:
        season_cell39, scene_cell39 = nm_cell39.rsplit("_", 1)
        rg_cell39 = scene_rowgroup_choice(season_cell39, int(scene_cell39), 1)[0]
        sh_cell39 = sen12mscr_shard(season_cell39, int(scene_cell39), rg_cell39, CP_CFG)
        sar_cell39 = sh_cell39["sar"][:64].astype(np.float32)
        opt_cell39 = sh_cell39["opt"][:64].astype(np.float32)
        yh_l, sc_l = [], []
        with torch.inference_mode():
            for k_cell39 in range(0, len(sar_cell39), 16):
                xb = torch.from_numpy(sar_cell39[k_cell39:k_cell39 + 16]).to(DEVICE)
                yb = ql_G(xb).float()
                sc_l.append(torch.sigmoid(ql_V(torch.cat([xb, yb], 1)).float())[:, 0].cpu().numpy())
                yh_l.append(yb.cpu().numpy())
        yh_cell39, sco_cell39 = np.concatenate(yh_l), np.concatenate(sc_l)
        wrong_cell39 = np.abs(yh_cell39 - opt_cell39).max(1) > QL_TAU
        trust_cell39 = np.searchsorted(QL_EDGES, sco_cell39, side="left") <= QL_J
        for i_cell39 in range(len(sco_cell39)):      # cap: at most q of each patch, lowest scores first
            if trust_cell39[i_cell39].mean() > QL_Q:
                thr_cell39 = np.sort(sco_cell39[i_cell39].ravel())[int(QL_Q * sco_cell39[i_cell39].size) - 1]
                trust_cell39[i_cell39] &= sco_cell39[i_cell39] <= thr_cell39
        for k_cell39, v_cell39 in (("sar", sar_cell39.astype(np.float16)), ("yh", yh_cell39.astype(np.float16)), ("opt", opt_cell39.astype(np.float16)),
                                   ("wrong", wrong_cell39), ("trust", trust_cell39)):
            ql_out[f"{nm_cell39}|{k_cell39}"] = v_cell39
    del ql_G, ql_V
    gpu_free() if "gpu_free" in globals() else None
    save_npz(QL_REL, **ql_out, J=np.array(QL_J), q=np.array(QL_Q), m=np.array(QL_M))
    QL = load_npz(QL_REL)
QL_STATS = []
for nm_cell39 in QL_SHOW:
    tr_cell39, wr_cell39 = QL[f"{nm_cell39}|trust"], QL[f"{nm_cell39}|wrong"]
    QL_STATS.append({"scene": nm_cell39, "patches": int(len(tr_cell39)), "error_rate": float(wr_cell39.mean()), "trusted": float(tr_cell39.mean()),
                     "false_trust": float((tr_cell39 & wr_cell39).mean()),
                     "error_among_trusted": float((tr_cell39 & wr_cell39).sum() / max(tr_cell39.sum(), 1))})
for r_cell39 in QL_STATS:
    print(r_cell39)

In [ ]:
# ===== Cell 40: the two paper figures =====
plt.rcParams.update({"font.size": 8, "font.family": "serif", "axes.linewidth": 0.6, "pdf.fonttype": 42, "legend.fontsize": 7, "axes.titlesize": 8})
PF_C = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]      # Okabe-Ito
PF_ALPHA, PF_TAU = CP_CFG["alpha_main"], CP_CFG["tau_main"]

# Figure 1: (a) failure rate vs patches per scene; (b), (c) certified area vs labelled scenes
fig_cell40, ax_cell40 = plt.subplots(1, 3, figsize=(7.16, 2.3))
t4_pix_cell40 = [r for r in T4S_ROWS if r["gen"] == "pix2pix"]
m_a_cell40 = 32 if any(r["m"] == 32 for r in t4_pix_cell40) else max((r["m"] for r in t4_pix_cell40), default=0)
for mode_cell40, c_cell40, mk_cell40, lab_cell40 in (("patch", PF_C[1], "s", "patch CRC"), ("scene", PF_C[0], "o", "scene CRC")):
    rr_cell40 = sorted([r for r in t4_pix_cell40 if r["m"] == m_a_cell40 and r["mode"] == mode_cell40], key=lambda r: r["n"])
    ax_cell40[0].plot([r["n"] for r in rr_cell40], [r["observed"] for r in rr_cell40], color=c_cell40, marker=mk_cell40, ms=3.5, lw=1.2, label=lab_cell40 + ", observed")
    ax_cell40[0].plot([r["n"] for r in rr_cell40], [r["skew_corrected"] for r in rr_cell40], color=c_cell40, ls="--", lw=0.9, label=lab_cell40 + ", T4$'$")
ns_cell40 = sorted({r["n"] for r in t4_pix_cell40})
ax_cell40[0].set_xscale("log", base=2)
ax_cell40[0].set_xticks(ns_cell40)
ax_cell40[0].set_xticklabels([str(n) for n in ns_cell40])
ax_cell40[0].set_xlabel(f"patches per scene $n$ ($m={m_a_cell40}$ scenes)")
ax_cell40[0].set_ylabel("P(user over budget)")
ax_cell40[0].set_ylim(-0.02, 0.62)
ax_cell40[0].legend(frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.13), handlelength=1.6)
ax_cell40[0].set_title("(a) users over budget")
for a_cell40, rows_cell40, ttl_cell40 in ((ax_cell40[1], CAP_ROWS_CAL, "(b) area, calibration scenes"),
                                          (ax_cell40[2], CAP_ROWS_WIN, "(c) area, winter scenes")):
    rr_cell40 = sorted([r for r in rows_cell40 if r["gen"] == "pix2pix" and r["alpha"] == PF_ALPHA and r["tau"] == PF_TAU], key=lambda r: r["m"])
    if not rr_cell40:
        continue
    for k_cell40, c_cell40, mk_cell40, lab_cell40 in (("pac_wsr_uncapped", PF_C[0], "o", "exact PAC, uncapped"),
                                                      ("pac_product_capped", PF_C[1], "s", "product certificate, capped"),
                                                      ("crc_capped", PF_C[3], "D", "scene CRC, capped (average only)")):
        a_cell40.plot([r["m"] for r in rr_cell40], [r[k_cell40]["area"] for r in rr_cell40], color=c_cell40, marker=mk_cell40, ms=3.5, lw=1.2, label=lab_cell40)
    a_cell40.axhline(rr_cell40[0]["oracle_uncapped"], color="#666", ls="--", lw=0.7)
    a_cell40.text(rr_cell40[0]["m"], rr_cell40[0]["oracle_uncapped"] + 0.004, "oracle", fontsize=7, color="#444")
    a_cell40.axhline(PF_ALPHA, color="#666", ls=":", lw=0.7)
    a_cell40.text(rr_cell40[-1]["m"], PF_ALPHA + 0.002, r"trivial ($\alpha$)", fontsize=7, color="#444", ha="right")
    a_cell40.set_xscale("log", base=2)
    a_cell40.set_xticks([r["m"] for r in rr_cell40])
    a_cell40.set_xticklabels([str(r["m"]) for r in rr_cell40])
    a_cell40.set_xlabel("labelled scenes $m$")
    a_cell40.set_ylim(-0.008, max(0.235, rr_cell40[0]["oracle_uncapped"] + 0.03))
    a_cell40.set_title(ttl_cell40)
ax_cell40[1].set_ylabel("certified trusted area")
h_cell40, l_cell40 = ax_cell40[1].get_legend_handles_labels()
fig_cell40.legend(h_cell40, l_cell40, frameon=False, loc="lower right", ncol=3, bbox_to_anchor=(0.995, -0.01), handlelength=1.6)
for a_cell40 in ax_cell40:
    a_cell40.grid(alpha=0.25, lw=0.4)
    a_cell40.tick_params(width=0.5, length=2.5)
fig_cell40.tight_layout(pad=0.3, w_pad=0.8, rect=(0, 0.08, 1, 1))
atomic_write(f"{CP_TAG}/paper/fig_curves.pdf", lambda p: fig_cell40.savefig(p))
plt.close(fig_cell40)

# Figure 2: one patch per left-out scene (the patch at the 70th percentile of trusted share)
fig_cell40, ax_cell40 = plt.subplots(len(QL_SHOW), 5, figsize=(7.16, 1.43 * len(QL_SHOW)), squeeze=False)
heads_cell40 = ["SAR (VV)", "Pix2Pix output", "Sentinel-2 reference", r"error $>\tau$ (dark)", "certified mask"]
for i_cell40, nm_cell40 in enumerate(QL_SHOW):
    tr_cell40 = QL[f"{nm_cell40}|trust"]
    p_cell40 = int(np.argsort(tr_cell40.mean((1, 2)))[int(0.7 * len(tr_cell40))])
    wr_cell40, t_cell40 = QL[f"{nm_cell40}|wrong"][p_cell40], tr_cell40[p_cell40]
    disp_cell40 = lambda x: np.clip(x.astype(np.float32).transpose(1, 2, 0) * 2.2, 0, 1)
    ax_cell40[i_cell40, 0].imshow(QL[f"{nm_cell40}|sar"][p_cell40, 0].astype(np.float32), cmap="gray", vmin=0, vmax=1)
    ax_cell40[i_cell40, 1].imshow(disp_cell40(QL[f"{nm_cell40}|yh"][p_cell40]))
    ax_cell40[i_cell40, 2].imshow(disp_cell40(QL[f"{nm_cell40}|opt"][p_cell40]))
    ax_cell40[i_cell40, 3].imshow(~wr_cell40, cmap="gray", vmin=0, vmax=1)
    mask_cell40 = np.ones(wr_cell40.shape + (3,)) * 0.93
    mask_cell40[t_cell40 & ~wr_cell40] = np.array([0, 0x72, 0xB2]) / 255
    mask_cell40[t_cell40 & wr_cell40] = np.array([0xD5, 0x5E, 0]) / 255
    ax_cell40[i_cell40, 4].imshow(mask_cell40)
    ax_cell40[i_cell40, 0].set_ylabel(nm_cell40.replace("_", " "), fontsize=7.5)
    ax_cell40[i_cell40, 4].text(0.5, 0.03, f"trusted {t_cell40.mean():.2f}, false trust {(t_cell40 & wr_cell40).mean():.3f}",
                                transform=ax_cell40[i_cell40, 4].transAxes, ha="center", va="bottom", fontsize=6.5,
                                bbox=dict(fc="white", ec="none", pad=1.2, alpha=0.9))
    for j_cell40 in range(5):
        ax_cell40[i_cell40, j_cell40].set_xticks([])
        ax_cell40[i_cell40, j_cell40].set_yticks([])
        if i_cell40 == 0:
            ax_cell40[i_cell40, j_cell40].set_title(heads_cell40[j_cell40], fontsize=8)
fig_cell40.subplots_adjust(left=0.035, right=0.998, top=0.955, bottom=0.008, wspace=0.03, hspace=0.04)
atomic_write(f"{CP_TAG}/paper/fig_qual.pdf", lambda p: fig_cell40.savefig(p, dpi=200))
plt.close(fig_cell40)
hf_push([f"{CP_TAG}/paper/fig_curves.pdf", f"{CP_TAG}/paper/fig_qual.pdf"])
print("figures saved:", local_path(f"{CP_TAG}/paper/fig_curves.pdf"), local_path(f"{CP_TAG}/paper/fig_qual.pdf"))

In [ ]:
# ===== Cell 41: Part IV report and the list of every output =====
lines_cell41 = [f"# CERTEO Part IV report — ideas that did not work, qualitative example, figures — run '{CP_TAG}'", ""]
if CP_TAG == "smoke":
    lines_cell41 += ["> **SMOKE RUN**: these numbers only check that the cells work; do not quote them.", ""]
lines_cell41 += ["## N1 — unlabelled scenes as a control variate", "", "| generator | score | alpha | bin | corr | variance cut | range ratio |", "|---|---|---|---|---|---|---|"]
lines_cell41 += [f"| {r['gen']} | {r['score']} | {r['alpha']} | {r['bin']} | {r['corr']:.2f} | {r['variance_cut']:.2f} | {r['range_ratio']:.2f} |" for r in N1_ROWS]
lines_cell41 += ["", "## N2 — rank-normalised masks: error rate among trusted pixels", "", "| generator | tau | base | best 1% | best 10% | best 20% |", "|---|---|---|---|---|---|"]
lines_cell41 += [f"| {r['gen']} | {r['tau']} | {r['base_error']:.3f} | {r['error_best_1pct']:.3f} | {r['error_best_10pct']:.3f} | {r['error_best_20pct']:.3f} |" for r in N2_ROWS]
lines_cell41 += ["", "## N3 — skew-corrected t bound (uncapped)", "", "| generator | alpha | tau | m | P(over) | area |", "|---|---|---|---|---|---|"]
lines_cell41 += [f"| {r['gen']} | {r['alpha']} | {r['tau']} | {r['m']} | {r['P_over']:.3f} | {r['area']:.3f} |" for r in N3_ROWS]
lines_cell41 += ["", "## N4 — cap sensitivity (capped WSR) and Bonferroni choice: area (P_over)", "", "| generator | alpha | tau | m | rule q | by q | Bonferroni |", "|---|---|---|---|---|---|---|"]
for r_cell41 in N4_ROWS:
    byq_cell41 = "; ".join(f"q={k}: {v['area']:.3f} ({v['P_over']:.3f})" for k, v in r_cell41["by_q"].items())
    lines_cell41.append(f"| {r_cell41['gen']} | {r_cell41['alpha']} | {r_cell41['tau']} | {r_cell41['m']} | {r_cell41['q_rule']:.3f} | {byq_cell41} | "
                        f"{r_cell41['bonferroni']['area']:.3f} ({r_cell41['bonferroni']['P_over']:.3f}) |")
lines_cell41 += ["", "## N5 — bet c of the product certificate at the rule cap: area (P_over)", "", "| generator | alpha | tau | m | q | WSR | c = 0.5 | c = 0.75 | c = 1 |", "|---|---|---|---|---|---|---|---|---|"]
for r_cell41 in N5_ROWS:
    lines_cell41.append(f"| {r_cell41['gen']} | {r_cell41['alpha']} | {r_cell41['tau']} | {r_cell41['m']} | {r_cell41['q']:.3f} | " + " | ".join(
        f"{r_cell41[k]['area']:.3f} ({r_cell41[k]['P_over']:.3f})" for k in ("wsr", "product_c0.5", "product_c0.75", "product_c1.0")) + " |")
lines_cell41 += ["", f"## Qualitative example (alpha = {QL_ALPHA}, tau = {QL_TAU}, m = {QL_M} calibration scenes, cap q = {QL_Q}, bin {QL_J})", "",
                 "| scene | patches | error rate | trusted | false trust | error among trusted |", "|---|---|---|---|---|---|"]
lines_cell41 += [f"| {r['scene']} | {r['patches']} | {r['error_rate']:.3f} | {r['trusted']:.3f} | {r['false_trust']:.4f} | {r['error_among_trusted']:.3f} |" for r in QL_STATS]
lines_cell41 += ["", "## Summary", "", "```json", json.dumps(N_SUMMARY, indent=1, default=float), "```"]
PART4_REPORT = "\n".join(lines_cell41)
atomic_write(f"{CP_TAG}/PART4_REPORT.md", lambda p: open(p, "w").write(PART4_REPORT))
hf_push([f"{CP_TAG}/PART4_REPORT.md"])
print(PART4_REPORT)

ALL_OUTPUTS = {"storage": (f"bucket {HF_BUCKET}/{RUN_ROOT}" if SYNC_ENABLED else f"local folder {LOCAL_ROOT}"), "run": CP_TAG,
               "reports": [r for r in (f"{CP_TAG}/HYPOTHESIS_REPORT.md", f"{CP_TAG}/PART3_REPORT.md", f"{CP_TAG}/PART4_REPORT.md") if os.path.exists(local_path(r))],
               "paper_figures": [f"{CP_TAG}/paper/fig_curves.pdf", f"{CP_TAG}/paper/fig_qual.pdf"],
               "json_results": sorted(os.path.relpath(os.path.join(dp, f), LOCAL_ROOT) for dp, _, fs in os.walk(local_path(CP_TAG)) for f in fs if f.endswith(".json"))}
print(json.dumps(ALL_OUTPUTS, indent=1))